# BreastVision-XAI V4 — BUS-VERIFY (Kaggle)
**Counterfactual Evidence Verification & Failure-Risk Detection cho siêu âm vú.** Công cụ NGHIÊN CỨU, không phải thiết bị y khoa.

> **TRẠNG THÁI: NOT RUN.** Notebook này được sinh tự động và đã kiểm cú pháp, nhưng CHƯA được chạy trên GPU/dữ liệu thật trong môi trường phát triển (không có torch, BUS-BRA, BUSI). Mọi con số chỉ có giá trị sau khi bạn chạy `RUN_MODE="full_experiment"`. `smoke_test` dùng dữ liệu tổng hợp: số liệu KHÔNG có giá trị khoa học.

**Cách dùng:** Add Input: BUS-BRA và BUSI -> bật GPU -> Run All. Quy trình (cố định trước trong `configs/bus_verify.json`):
1. Kiểm toán dữ liệu + chia theo bệnh nhân (khớp manifest V3).
2. Huấn luyện X6: 5 fold OOF x seeds trên dữ liệu phát triển (train+val), cộng mô hình cuối x seeds (resume được).
3. Fit T-scaling và ngưỡng Youden CHỈ trên dự đoán OOF của dữ liệu phát triển.
4. Trích đặc trưng (độ tin cậy, bất định, ổn định, counterfactual trên mask DỰ ĐOÁN).
5. Giao thức benchmark: nested CV trên dev (xác nhận chính) -> fit trên dev -> BUS-BRA test (ĐÃ khảo sát ở V3, không gọi là chưa từng thấy) và BUSI.
6. Khóa risk model + chính sách cảnh báo (sha256), robustness lab, thí nghiệm ORACLE tách riêng, ghi artifact cho web app.

Hạn chế đã biết: cross-fit (risk học từ mô hình fold, áp dụng cho mô hình cuối); ensemble chỉ cho baseline B3; BUSI không có patient ID; số FN ở test rất nhỏ nên CI rộng.

In [ ]:
import os
for d in ['bsv4/src/v3', 'bsv4/configs', 'bsv4/artifacts']:
    os.makedirs(d, exist_ok=True)

## 1. Ghi mã nguồn gói BUS-VERIFY (nội dung = src/ của ứng dụng)

In [ ]:
%%writefile bsv4/src/__init__.py
"""BreastVision-XAI V4 — BUS-VERIFY. Các module V3 đã kiểm thử nằm ở src/v3 (giữ nguyên, có SHA256SUMS)."""
from ._paths import ensure_v3
ensure_v3()
__version__ = "4.0.0"

In [ ]:
%%writefile bsv4/src/_paths.py
import os, sys

V3_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'v3')


def ensure_v3():
    """Thêm src/v3 vào sys.path để import bus_core, bus_trust, models_xai... (module V3 giữ nguyên)."""
    if V3_DIR not in sys.path:
        sys.path.insert(0, V3_DIR)
    return V3_DIR

In [ ]:
%%writefile bsv4/src/benchmark.py
"""Benchmark phát hiện lỗi BUS-VERIFY: B0..B6, P1, ablation; đánh giá (a) NESTED CV trên dữ liệu phát triển (xác nhận chính),
(b) fit-trên-dev -> áp dụng cho tập giữ lại (test BUS-BRA cũ, BUSI). Thuần numpy/sklearn.

Quy ước: điểm CAO = nguy cơ mô hình chính dự đoán SAI cao. Nhãn lỗi `wrong` do mô hình chính tạo ra. KHÔNG dùng nhãn test để huấn luyện/chọn C/ngưỡng."""
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from . import failure_risk as FR
from . import selective_prediction as SP
from . import statistical_tests as ST
from . import evaluation as EV

BASELINE_NOTES = {
    'B0': 'Random referral (cảnh báo ngẫu nhiên, seed cố định)',
    'B1': 'Maximum softmax confidence (CHƯA hiệu chuẩn, so với 0.5)',
    'B1b': 'Threshold margin: |logit - logit(ngưỡng quyết định)| (bổ sung: ngưỡng Youden != 0.5 nên B1 không phải khoảng cách tới ranh giới thật)',
    'B2': 'Calibrated confidence (T-scaling đơn điệu => CÙNG thứ hạng với B1; chỉ khác xác suất, không khác khả năng phát hiện lỗi)',
    'B3': 'Ensemble uncertainty (JS/BALD giữa các thành viên; nếu không có ensemble thật thì dùng bất đồng giữa các góc nhìn TTA và được GHI RÕ)',
    'B4': 'Learned confidence-only error detector (LR trên nhóm confidence)',
    'B5': 'Counterfactual features only (LR)',
    'B6': 'Uncertainty + stability, KHÔNG có counterfactual (LR)',
    'P1': 'BUS-VERIFY đầy đủ (LR trên confidence + bất định + ổn định + counterfactual)',
}


def present(df, names):
    """Chỉ giữ cột có mặt và có ít nhất một giá trị hữu hạn."""
    return [n for n in names if n in df and df[n].notna().any()]


ENS_COLS = ('ens_js', 'ens_dz_sd', 'ens_flip')


def feature_sets(df, drop_rank_duplicates=True, deploy_compatible=True):
    """FEATURE_SETS + ABLATIONS đã lọc theo cột thực có. Trả (dict tên -> list đặc trưng, dict ghi chú các cột bị bỏ)."""
    notes = {}
    out = {}
    for name, cols in {**FR.FEATURE_SETS, **{k: [c for c in FR.FEATURE_SETS['P1_full'] if c not in set(v)] for k, v in FR.ABLATIONS.items()}}.items():
        cols = present(df, cols)
        if deploy_compatible:                    # ứng dụng chỉ có MỘT checkpoint: đặc trưng ensemble chỉ dùng cho baseline B3, KHÔNG vào mô hình risk triển khai
            cols = [c for c in cols if c not in ENS_COLS]
        if drop_rank_duplicates:
            cols, dropped = FR.dedupe_rank_equivalent(df, cols)
            if dropped:
                notes[name] = dropped
        out[name] = cols
    return out, notes


def unlearned_scores(df, seed=0, ens_prefer=True):
    """B0..B3 (không học). Trả dict tên -> điểm nguy cơ (cao = nguy cơ sai cao) và nguồn của B3."""
    n = len(df)
    out = {'B0': np.random.default_rng(seed).random(n)}
    p_raw = df['p_raw'].to_numpy(float) if 'p_raw' in df else df['p_mal'].to_numpy(float)
    out['B1'] = 1.0 - np.maximum(p_raw, 1 - p_raw)
    out['B1b'] = -df['thr_margin'].to_numpy(float)
    pc = df['p_mal'].to_numpy(float)
    out['B2'] = 1.0 - np.maximum(pc, 1 - pc)
    src = 'ensemble' if (ens_prefer and 'ens_js' in df and df['ens_js'].notna().any()) else 'tta_views'
    out['B3'] = df['ens_js'].to_numpy(float) if src == 'ensemble' else df['view_js'].to_numpy(float)
    return out, src


def nested_oof_scores(df_dev, sets, wrong, groups, k=5, seed=0, C_grid=(0.01, 0.03, 0.1, 0.3, 1.0)):
    """Nested CV theo cụm: vòng NGOÀI k-fold cho điểm held-out; vòng TRONG chỉ chọn C trên phần train của vòng ngoài. Trả (scores, chosen_C)."""
    y = np.asarray(wrong).astype(int); g = np.asarray(groups)
    outer = list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, g))
    scores, chosen = {}, {}
    for name, cols in sets.items():
        if not cols:
            continue
        X = df_dev[cols].to_numpy(float)
        s = np.full(len(y), np.nan); cs = []
        for tr, te in outer:
            C, _ = FR.select_C(X[tr], y[tr], g[tr], grid=C_grid, k=3, seed=seed)
            m = FR.make_pipeline(C).fit(X[tr], y[tr])
            s[te] = m.predict_proba(X[te])[:, 1]; cs.append(C)
        scores[name] = s; chosen[name] = cs
    return scores, chosen


def fit_and_apply(df_dev, wrong_dev, groups_dev, split_dev, df_eval, sets, seed=0, k=5):
    """Huấn luyện trên TOÀN BỘ dev (assert_no_test) rồi chấm điểm tập giữ lại. Trả (scores, models)."""
    FR.assert_no_test(split_dev)
    models, scores = {}, {}
    for name, cols in sets.items():
        if not cols:
            continue
        m = FR.FailureRiskModel(cols, seed=seed, k=k).fit(df_dev, wrong_dev, groups_dev, split_dev)
        models[name] = m
        scores[name] = m.predict_risk(df_eval)
    return scores, models


PRIMARY_PAIRS = (('P1', 'B1'), ('P1', 'B3'), ('P1', 'B4'), ('P1', 'B6'))


def paired_comparisons(scores, wrong, is_fn, groups, pairs=PRIMARY_PAIRS, rate=0.2, B=2000, seed=0, alpha=0.05):
    """So sánh ghép cặp theo CỤM bệnh nhân (cùng mẫu bootstrap) cho hai điểm cuối chính: Error AUPRC và FN Alert Recall tại `rate`.
    Hiệu chỉnh Holm trên TOÀN BỘ các so sánh (bảo thủ). Báo cả effect size (Δ/SE) và CI."""
    wrong = np.asarray(wrong).astype(int); is_fn = np.asarray(is_fn).astype(bool); g = np.asarray(groups)

    def auprc(s):
        return lambda i: EV.safe_auprc(wrong[i], s[i])

    def fnr(s):
        def f(i):
            nf = is_fn[i].sum()
            if not nf:
                return float('nan')
            a = SP.alert_set(s[i], rate, rng=np.random.default_rng(seed))
            return float((a & is_fn[i]).sum() / nf)
        return f
    rows = []
    for a, b in pairs:
        if a not in scores or b not in scores:
            continue
        for metric, mk in (('error_auprc', auprc), (f'fn_alert_recall@{int(rate*100)}', fnr)):
            r = ST.paired_cluster_bootstrap(mk(np.asarray(scores[a], float)), mk(np.asarray(scores[b], float)), g, B=B, seed=seed, alpha=alpha)
            rows.append(dict(A=a, B=b, metric=metric, delta=r['delta'], ci_lo=r['lo'], ci_hi=r['hi'], p=r['p'], effect_z=r['z'], n_valid=r['n_valid']))
    df = pd.DataFrame(rows)
    if len(df):
        adj, rej = ST.holm_bonferroni(df['p'].fillna(1.0).to_numpy(), alpha)
        df['p_holm'] = adj; df['significant_holm'] = rej
        df['verdict'] = np.where(~df['significant_holm'], 'không đủ bằng chứng sau hiệu chỉnh đa so sánh',
                                 np.where(df['delta'] > 0, 'A tốt hơn B (có ý nghĩa sau Holm)', 'B tốt hơn A (có ý nghĩa sau Holm)'))
    return df


def risk_calibration_report(risk, wrong, n_bins=10):
    """Hiệu chuẩn của risk score (xác suất mô hình chính SAI) so với tần suất sai quan sát. KHÁC với hiệu chuẩn P(malignant)."""
    r = np.clip(np.asarray(risk, float), 0, 1); y = np.asarray(wrong, float)
    edges = np.quantile(r, np.linspace(0, 1, n_bins + 1)); edges[-1] += 1e-9
    ece, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (r >= lo) & (r < hi)
        if m.sum():
            rows.append(dict(mean_risk=float(r[m].mean()), observed_error=float(y[m].mean()), n=int(m.sum())))
            ece += m.mean() * abs(r[m].mean() - y[m].mean())
    return dict(ece=float(ece), brier=float(((r - y) ** 2).mean()), mean_risk=float(r.mean()), error_rate=float(y.mean()), bins=rows)


def make_alert_policy(risk_dev_oof, review_rate=0.20, high_rate=0.10):
    """Ngưỡng cảnh báo chỉ từ risk OOF của DỮ LIỆU PHÁT TRIỂN (phân vị), t_review <= t_high."""
    t_review = FR.threshold_for_alert_rate(risk_dev_oof, review_rate)
    t_high = FR.threshold_for_alert_rate(risk_dev_oof, high_rate)
    return dict(t_review=float(min(t_review, t_high)), t_high=float(t_high), review_rate_dev=review_rate, high_rate_dev=high_rate,
                n_dev=int(len(risk_dev_oof)), source='out-of-fold risk trên dữ liệu phát triển (không dùng test)')


def status_counts(risk, policy, in_scope=None):
    risk = np.asarray(risk, float)
    ins = np.ones(len(risk), bool) if in_scope is None else np.asarray(in_scope, bool)
    st = [SP.decide_status(r, policy['t_review'], policy['t_high'], in_scope=bool(i)) for r, i in zip(risk, ins)]
    return pd.Series(st).value_counts().to_dict(), st


def run_protocol(df_dev, wrong_dev, fn_dev, groups_dev, split_dev, df_eval_map, wrong_eval_map, fn_eval_map, groups_eval_map,
                 seed=0, k=5, B=1000, rates=(0.1, 0.2, 0.3), include_ablations=True):
    """Toàn bộ giao thức: (1) nested CV trên dev -> bảng xác nhận chính; (2) fit trên dev -> từng tập giữ lại trong df_eval_map.
    Trả dict kết quả (bảng + mô hình P1 + chính sách đã khóa)."""
    sets, notes = feature_sets(df_dev)
    if not include_ablations:
        sets = {k_: v for k_, v in sets.items() if k_ in FR.FEATURE_SETS}
    un_dev, src = unlearned_scores(df_dev, seed)
    nested, chosen = nested_oof_scores(df_dev, sets, wrong_dev, groups_dev, k=k, seed=seed)
    dev_scores = {**un_dev, **{('P1' if n == 'P1_full' else n.split('_')[0] if n in FR.FEATURE_SETS else n): s for n, s in nested.items()}}
    out = dict(sets=sets, dropped_rank_duplicates=notes, b3_source=src, chosen_C=chosen, dev_scores=dev_scores)
    out['dev_table'] = EV.error_detection_table(dev_scores, wrong_dev, fn_dev, groups_dev, rates=rates, B=B, seed=seed)
    out['dev_pairs'] = paired_comparisons(dev_scores, wrong_dev, fn_dev, groups_dev, B=B, seed=seed)
    _, models = fit_and_apply(df_dev, wrong_dev, groups_dev, split_dev, df_dev.iloc[:1], {n: c for n, c in sets.items() if n in FR.FEATURE_SETS}, seed=seed, k=k)
    p1 = models['P1_full']
    out['model'] = p1
    out['policy'] = make_alert_policy(p1.calib(p1.oof_raw))
    out['held_out'] = {}
    for name, dfe in df_eval_map.items():
        sc, _ = fit_and_apply(df_dev, wrong_dev, groups_dev, split_dev, dfe, {n: c for n, c in sets.items() if n in FR.FEATURE_SETS}, seed=seed, k=k)
        un, _ = unlearned_scores(dfe, seed)
        allsc = {**un, **{('P1' if n == 'P1_full' else n.split('_')[0]): s for n, s in sc.items()}}
        out['held_out'][name] = dict(table=EV.error_detection_table(allsc, wrong_eval_map[name], fn_eval_map[name], groups_eval_map[name], rates=rates, B=B, seed=seed),
                                     pairs=paired_comparisons(allsc, wrong_eval_map[name], fn_eval_map[name], groups_eval_map[name], B=B, seed=seed),
                                     scores=allsc, risk_calibration=risk_calibration_report(allsc['P1'], wrong_eval_map[name]))
    return out

In [ ]:
%%writefile bsv4/src/counterfactual_verifier.py
"""Counterfactual Evidence Verification Engine (BUS-VERIFY).

Ý tưởng: với một ảnh và mặt nạ phân vùng DỰ ĐOÁN, ta can thiệp có kiểm soát lên ba vùng (lesion / perilesional / background)
và đo mô hình phản ứng ra sao. Mô hình "dựa vào bằng chứng đúng chỗ" khi logit nhạy với can thiệp lên tổn thương nhiều hơn
so với can thiệp lên vùng ĐỐI CHỨNG cùng diện tích, cùng loại phép biến đổi, cường độ trung bình gần nhau, độ sâu gần nhau.

Ràng buộc khoa học (được kiểm thử):
  * Không dùng ground-truth mask ở suy luận: hàm công khai chỉ nhận `pred_mask`; không có tham số GT.
  * Phép biến đổi gây dịch chuyển phân phối (OOD): mỗi can thiệp ghi bản QA (diện tích, thay đổi pixel, 'đường nối', kết cấu) để phân tích
    xem độ nhạy là do bằng chứng hay do artefact; đối chứng matched-area/random là thước đo nền của artefact.
  * Thuần numpy/cv2. Mô hình được truyền vào dưới dạng `predict_fn(images uint8 (N,H,W,3)) -> dict(logits=(N,2), [seg_prob], [evidence])`.
"""
from dataclasses import dataclass, field, asdict
import numpy as np
import cv2

from .segmentation import build_regions, Regions

OPS = ('blur', 'replace', 'permute')
EPS = 1e-8


@dataclass
class CFConfig:
    ring_px: int = 12                  # độ rộng vùng quanh tổn thương (ở ảnh 256 px)
    blur_sigma: float = 6.0
    feather_px: float = 0.0            # 0 = biên cứng (mặc định, để diện tích kiểm soát chính xác)
    n_matched: int = 3                 # số bản đối chứng matched-area cho mỗi (vùng, phép biến đổi)
    n_random: int = 3                  # số đối chứng ngẫu nhiên (vị trí và diện tích ngẫu nhiên quanh diện tích tổn thương)
    n_candidates: int = 40             # số ứng viên vị trí khi khớp cường độ
    control_depth_tol: float = 0.10    # |dy| tối đa (tỉ lệ chiều cao) cho đối chứng 'cùng độ sâu'; None = bất kỳ
    min_region_px: int = 64
    ops: tuple = OPS
    ev_q: float = 0.20                 # tỉ lệ diện tích 'evidence cao nhất' bị can thiệp
    z_floor: float = 0.05              # sàn độ lệch chuẩn khi tính z-score (đơn vị logit)
    seed: int = 0


# ------------------------------------------------------------------ biến đổi vùng
def _alpha(region, feather):
    a = region.astype(np.float32)
    if feather and feather > 0:
        a = np.maximum(cv2.GaussianBlur(a, (0, 0), float(feather)), a)
    return a[..., None]


def apply_op(img, region, op, rng, ref=None, cfg=None):
    """Áp dụng một phép biến đổi lên `region` (bool HxW). Trả ảnh uint8.
    blur   : làm mờ Gauss (loại tần số cao, giữ cường độ trung bình).
    replace: thay bằng cường độ lấy mẫu có hoàn lại từ vùng tham chiếu `ref` (phá cấu trúc, cường độ khớp vùng tham chiếu).
    permute: hoán vị pixel trong vùng (giữ NGUYÊN histogram của vùng, phá cấu trúc không gian)."""
    cfg = cfg or CFConfig()
    src = img.astype(np.float32)
    out = src.copy()
    idx = np.where(region)
    n = len(idx[0])
    if n == 0:
        return img.copy()
    if op == 'blur':
        pert = cv2.GaussianBlur(src, (0, 0), cfg.blur_sigma)
        vals = pert[idx]
    elif op == 'replace':
        if ref is None or not np.asarray(ref).any():
            ref = ~region
        if not np.asarray(ref).any():
            return img.copy()
        ridx = np.where(ref)
        pick = rng.integers(0, len(ridx[0]), size=n)
        g = src[ridx[0][pick], ridx[1][pick]]                       # (n,3) giữ nguyên bộ ba kênh của pixel tham chiếu
        vals = g
    elif op == 'permute':
        perm = rng.permutation(n)
        vals = src[idx][perm]
    else:
        raise ValueError(f'op không hợp lệ: {op}')
    if cfg.feather_px and cfg.feather_px > 0 and op == 'blur':
        pert_full = cv2.GaussianBlur(src, (0, 0), cfg.blur_sigma)
        a = _alpha(region, cfg.feather_px)
        out = a * pert_full + (1 - a) * src
    else:
        out[idx] = vals
    return np.clip(out, 0, 255).astype(np.uint8)


def perturbation_qa(img, pert, region):
    """Thống kê QA của một can thiệp: diện tích, thay đổi pixel, 'đường nối' (gradient dọc biên), kết cấu trong vùng."""
    a, b = img.astype(np.float32).mean(2), pert.astype(np.float32).mean(2)
    reg = region
    area = float(reg.mean())
    chg = float(np.abs(b - a)[reg].mean()) if reg.any() else 0.0
    ring = cv2.dilate(reg.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
    band = ring & ~(cv2.erode(reg.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0)

    def gmag(x):
        gx = cv2.Sobel(x, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(x, cv2.CV_32F, 0, 1, ksize=3)
        return np.sqrt(gx * gx + gy * gy)
    ga, gb = gmag(a), gmag(b)
    seam = float(gb[band].mean() / (ga[band].mean() + EPS)) if band.any() else 1.0
    tex = float(b[reg].std() / (a[reg].std() + EPS)) if reg.sum() > 1 else 1.0
    return dict(area_frac=area, mean_abs_change=chg, seam_ratio=seam, texture_ratio=tex)


# ------------------------------------------------------------------ vùng đối chứng
def _bbox(mask):
    ys, xs = np.nonzero(mask)
    return ys.min(), ys.max(), xs.min(), xs.max()


def _shift(mask, dy, dx):
    ys, xs = np.nonzero(mask)
    out = np.zeros_like(mask)
    out[ys + dy, xs + dx] = True
    return out


def _disk_mask(shape, cy, cx, r):
    yy, xx = np.ogrid[:shape[0], :shape[1]]
    return (yy - cy) ** 2 + (xx - cx) ** 2 <= r * r


def matched_area_controls(region, forbidden, valid, gray, rng, k=3, n_candidates=40, depth_tol=0.10):
    """Tìm k vùng đối chứng: CÙNG HÌNH DẠNG và CÙNG DIỆN TÍCH (dịch tịnh tiến nguyên), nằm hoàn toàn trong `valid`, không giao `forbidden`,
    ưu tiên cường độ trung bình gần vùng gốc nhất và (nếu depth_tol) cùng độ sâu. Trả (list mask, list thông tin)."""
    H, W = region.shape
    y0, y1, x0, x1 = _bbox(region)
    target = float(gray[region].mean())
    cands = []
    tries = 0
    while len(cands) < n_candidates and tries < n_candidates * 25:
        tries += 1
        dx = int(rng.integers(-x0, W - x1))
        if depth_tol is None:
            dy = int(rng.integers(-y0, H - y1))
        else:
            lo, hi = max(-y0, -int(depth_tol * H)), min(H - 1 - y1, int(depth_tol * H))
            if lo > hi:
                continue
            dy = int(rng.integers(lo, hi + 1))
        if abs(dx) + abs(dy) < 3:
            continue
        m = _shift(region, dy, dx)
        if (m & forbidden).any() or (m & ~valid).any():
            continue
        cands.append((abs(float(gray[m].mean()) - target), dy, dx, m))
    cands.sort(key=lambda t: t[0])
    chosen, info, used = [], [], []
    for gap, dy, dx, m in cands:
        if any((m & u).any() for u in used):
            continue
        chosen.append(m); used.append(m)
        info.append(dict(kind='matched', dy=dy, dx=dx, intensity_gap=gap, area_px=int(m.sum())))
        if len(chosen) >= k:
            break
    return chosen, info


def random_controls(area_px, forbidden, valid, gray, rng, k=3, area_jitter=(0.5, 1.5)):
    """Đối chứng ngẫu nhiên: đĩa ở vị trí ngẫu nhiên (bất kỳ độ sâu) với diện tích ~ U(0.5,1.5)×diện tích tổn thương."""
    H, W = valid.shape
    out, info = [], []
    tries = 0
    while len(out) < k and tries < 400:
        tries += 1
        a = area_px * float(rng.uniform(*area_jitter))
        r = max(3.0, float(np.sqrt(a / np.pi)))
        cy, cx = int(rng.integers(0, H)), int(rng.integers(0, W))
        m = _disk_mask((H, W), cy, cx, r)
        if m.sum() < 16 or (m & forbidden).any() or (m & ~valid).any():
            continue
        out.append(m)
        info.append(dict(kind='random', cy=cy, cx=cx, area_px=int(m.sum()), intensity_gap=abs(float(gray[m].mean()) - float(gray[valid].mean()))))
    return out, info


# ------------------------------------------------------------------ lõi
def _z(logits, T):
    l = np.asarray(logits, np.float64)
    return (l[:, 1] - l[:, 0]) / float(T)


def _sig(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


def evidence_top_mask(evidence, valid, q, out_hw):
    """Mặt nạ q% diện tích có bằng chứng cao nhất (evidence của CHÍNH mô hình, không dùng GT)."""
    E = cv2.resize(np.asarray(evidence, np.float32), (out_hw[1], out_hw[0]), interpolation=cv2.INTER_LINEAR)
    vals = E[valid]
    if vals.size == 0:
        return np.zeros(out_hw, bool)
    k = max(1, int(round(q * valid.sum())))
    ys, xs = np.nonzero(valid)
    order = np.argsort(-E[ys, xs], kind='stable')[:k]            # chọn ĐÚNG k pixel (tie xác định theo vị trí): diện tích kiểm soát chính xác
    out = np.zeros(out_hw, bool)
    out[ys[order], xs[order]] = True
    return out


def verify_image(img, pred_mask, predict_fn, T=1.0, thr=0.5, cfg=None, valid=None, evidence=None, rng=None, return_records=False):
    """Chạy toàn bộ kiểm chứng phản thực cho MỘT ảnh. Chỉ nhận mặt nạ dự đoán.
    Trả dict đặc trưng (float; NaN nếu không áp dụng được) và `status` ('ok'|'empty_mask'|...)."""
    cfg = cfg or CFConfig()
    rng = rng if rng is not None else np.random.default_rng(cfg.seed)
    H, W = img.shape[:2]
    reg = build_regions(pred_mask, valid, cfg.ring_px)
    nan = float('nan')
    keys = []
    for r in ('lesion', 'peri', 'bg'):
        keys += [f'{r}_sens', f'{r}_dir', f'{r}_dp', f'{r}_flip', f'{r}_consistency']
    keys += ['matched_les_sens', 'matched_peri_sens', 'random_sens', 'cf_specificity', 'cf_ratio', 'cf_z', 'lesion_bg_contrast',
             'lesion_peri_contrast', 'cf_flip_rate', 'cf_consistency', 'ev_drop_top', 'ev_drop_random', 'ev_faithfulness', 'n_controls_ok',
             'mean_control_intensity_gap']
    feats = {k: nan for k in keys}
    if not reg.ok:
        return dict(status=reg.reason, features=feats, records=[])
    gray = img.astype(np.float32).mean(2)
    valid_m = reg.valid
    forbidden = (cv2.dilate(reg.lesion.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * cfg.ring_px + 1,) * 2)) > 0)
    ref_les = reg.peri if reg.peri.sum() >= 16 else reg.background
    refs = {'lesion': ref_les, 'peri': reg.background if reg.background.any() else ref_les, 'bg': reg.peri if reg.peri.any() else reg.lesion}
    regions = {'lesion': reg.lesion, 'peri': reg.peri, 'bg': reg.background}

    items = []                                 # (group, region_name, op, kind, mask)
    batch = [img]
    qa = []
    for rn, m in regions.items():
        if m.sum() < cfg.min_region_px:
            continue
        for op in cfg.ops:
            p = apply_op(img, m, op, rng, ref=refs[rn], cfg=cfg)
            items.append(('region', rn, op, 'target', m)); batch.append(p)
            qa.append(dict(group='region', region=rn, op=op, kind='target', **perturbation_qa(img, p, m)))
    n_ctrl = 0
    ctrl_gaps = []
    for rn in ('lesion', 'peri'):
        m = regions[rn]
        if m.sum() < cfg.min_region_px:
            continue
        ctrls, info = matched_area_controls(m, forbidden, valid_m, gray, rng, cfg.n_matched, cfg.n_candidates, cfg.control_depth_tol)
        if len(ctrls) < cfg.n_matched:       # nới ràng buộc độ sâu (ghi nhận trong n_controls_ok)
            more, info2 = matched_area_controls(m, forbidden, valid_m, gray, rng, cfg.n_matched - len(ctrls), cfg.n_candidates, None)
            ctrls += more; info += info2
        for cm, ci in zip(ctrls, info):
            ctrl_gaps.append(ci['intensity_gap']); n_ctrl += 1
            for op in cfg.ops:
                p = apply_op(img, cm, op, rng, ref=refs[rn], cfg=cfg)
                items.append(('matched', rn, op, 'matched', cm)); batch.append(p)
                qa.append(dict(group='matched', region=rn, op=op, kind='matched', **perturbation_qa(img, p, cm), **{k: v for k, v in ci.items() if k in ('dy', 'dx', 'intensity_gap')}))
    rc, rinfo = random_controls(int(reg.lesion.sum()), forbidden, valid_m, gray, rng, cfg.n_random)
    for cm, ci in zip(rc, rinfo):
        for op in cfg.ops:
            p = apply_op(img, cm, op, rng, ref=refs['lesion'], cfg=cfg)
            items.append(('random', 'lesion', op, 'random', cm)); batch.append(p)
            qa.append(dict(group='random', region='lesion', op=op, kind='random', **perturbation_qa(img, p, cm)))
    # can thiệp theo evidence của chính mô hình
    ev_items = []
    if evidence is not None:
        top = evidence_top_mask(evidence, valid_m, cfg.ev_q, (H, W))
        if top.sum() >= cfg.min_region_px:
            p = apply_op(img, top, 'replace', rng, ref=~top & valid_m, cfg=cfg)
            ev_items.append(('ev_top', len(batch))); batch.append(p)
            rr, _ = random_controls(int(top.sum()), top, valid_m, gray, rng, cfg.n_random, area_jitter=(1.0, 1.0))
            for cm in rr:
                p = apply_op(img, cm, 'replace', rng, ref=~cm & valid_m, cfg=cfg)
                ev_items.append(('ev_rand', len(batch))); batch.append(p)

    out = predict_fn(np.stack(batch, 0))
    z_all = _z(out['logits'], T)
    z0 = float(z_all[0]); p0 = float(_sig(z0)); y0 = int(p0 >= thr); s0 = 1.0 if z0 >= 0 else -1.0
    zp = z_all[1:1 + len(items)]
    recs = []
    for it, z in zip(items, zp):
        dz = float(z - z0)
        recs.append(dict(group=it[0], region=it[1], op=it[2], dz=dz, absdz=abs(dz), dir=s0 * dz, dp=float(_sig(z) - p0),
                         flip=int((_sig(z) >= thr) != bool(y0)), area_px=int(it[4].sum())))
    for r, q in zip(recs, qa):
        r.update({k: v for k, v in q.items() if k in ('mean_abs_change', 'seam_ratio', 'texture_ratio', 'intensity_gap')})

    def agg(group, region, field_, fn=np.mean):
        v = [r[field_] for r in recs if r['group'] == group and r['region'] == region]
        return float(fn(v)) if v else nan

    for rn in ('lesion', 'peri', 'bg'):
        v = [r for r in recs if r['group'] == 'region' and r['region'] == rn]
        if not v:
            continue
        feats[f'{rn}_sens'] = float(np.mean([r['absdz'] for r in v]))
        feats[f'{rn}_dir'] = float(np.mean([r['dir'] for r in v]))
        feats[f'{rn}_dp'] = float(np.mean([abs(r['dp']) for r in v]))
        feats[f'{rn}_flip'] = float(np.mean([r['flip'] for r in v]))
        sg = np.sign([r['dz'] for r in v])
        feats[f'{rn}_consistency'] = float(abs(sg.mean()))
    feats['matched_les_sens'] = agg('matched', 'lesion', 'absdz')
    feats['matched_peri_sens'] = agg('matched', 'peri', 'absdz')
    feats['random_sens'] = agg('random', 'lesion', 'absdz')
    ctrl_vals = np.array([r['absdz'] for r in recs if r['group'] == 'matched' and r['region'] == 'lesion'], float)
    if np.isfinite(feats['lesion_sens']) and ctrl_vals.size:
        mu, sd = float(ctrl_vals.mean()), float(ctrl_vals.std())
        feats['cf_specificity'] = feats['lesion_sens'] - mu
        feats['cf_ratio'] = feats['lesion_sens'] / (mu + 1e-2)
        feats['cf_z'] = (feats['lesion_sens'] - mu) / max(sd, cfg.z_floor)
    if np.isfinite(feats['lesion_sens']) and np.isfinite(feats['bg_sens']):
        feats['lesion_bg_contrast'] = feats['lesion_sens'] - feats['bg_sens']
    if np.isfinite(feats['lesion_sens']) and np.isfinite(feats['peri_sens']):
        feats['lesion_peri_contrast'] = feats['lesion_sens'] - feats['peri_sens']
    tgt = [r for r in recs if r['group'] == 'region']
    if tgt:
        feats['cf_flip_rate'] = float(np.mean([r['flip'] for r in tgt]))
        feats['cf_consistency'] = float(abs(np.sign([r['dz'] for r in tgt]).mean()))
    if ev_items:
        zt = [z_all[i] for k, i in ev_items if k == 'ev_top']
        zr = [z_all[i] for k, i in ev_items if k == 'ev_rand']
        if zt:
            feats['ev_drop_top'] = float(s0 * (z0 - zt[0]))             # >0: bỏ evidence cao làm GIẢM độ tin vào lớp dự đoán
        if zr:
            feats['ev_drop_random'] = float(np.mean([s0 * (z0 - z) for z in zr]))
        if zt and zr:
            feats['ev_faithfulness'] = feats['ev_drop_top'] - feats['ev_drop_random']
    feats['n_controls_ok'] = float(n_ctrl)
    feats['mean_control_intensity_gap'] = float(np.mean(ctrl_gaps)) if ctrl_gaps else nan
    res = dict(status='ok', features=feats, z0=z0, p0=p0, pred=y0)
    if return_records:
        res['records'] = recs
    return res


# ------------------------------------------------------------------ phép thử shortcut (thăm dò, NGOÀI P1 chính)
def annotation_mask(img, valid=None, bright=235, max_comp_px=200):
    """Phát hiện chữ/dấu đo (caliper) kiểu BUS-BRA: thành phần sáng nhỏ trong vùng hợp lệ. Heuristic, KHÔNG phải nhãn chuẩn."""
    g = img.max(2)
    m = ((g >= bright) & (valid if valid is not None else True)).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(m)
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] <= max_comp_px:
            out[lab == i] = 1
    return cv2.dilate(out, np.ones((5, 5), np.uint8)) > 0


def shortcut_probe(img, valid, predict_fn, T=1.0, thr=0.5):
    """Độ nhạy của logit khi (a) lấp dải letterbox bằng phản chiếu nội dung, (b) xoá chữ/dấu đo bằng inpainting.
    Phản ánh nghi vấn Grad-CAM của V3 dồn vào dải đen/mép ảnh. Trả {pad_sens, annot_sens, annot_frac}."""
    batch, kinds = [img], []
    pad = ~valid
    if pad.any():
        ys, xs = np.nonzero(valid)
        crop = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        top, left = ys.min(), xs.min()
        bottom, right = img.shape[0] - ys.max() - 1, img.shape[1] - xs.max() - 1
        filled = cv2.copyMakeBorder(crop, top, bottom, left, right, cv2.BORDER_REFLECT_101)
        if filled.shape[:2] == img.shape[:2]:
            batch.append(filled); kinds.append('pad')
    am = annotation_mask(img, valid)
    if am.any():
        batch.append(cv2.inpaint(img, am.astype(np.uint8), 3, cv2.INPAINT_TELEA)); kinds.append('annot')
    res = dict(pad_sens=float('nan'), annot_sens=float('nan'), annot_frac=float(am.mean()), pad_frac=float(pad.mean()))
    if len(batch) == 1:
        return res
    z = _z(predict_fn(np.stack(batch, 0))['logits'], T)
    for k, zz in zip(kinds, z[1:]):
        res[f'{k}_sens'] = float(abs(zz - z[0]))
    return res

In [ ]:
%%writefile bsv4/src/data_audit.py
"""Kiểm toán dữ liệu: dùng lại data_audit V3 (đã chạy thật trên BUS-BRA/BUSI) + bổ sung K-fold theo nhóm trên dữ liệu phát triển (OOF)."""
import numpy as np
import pandas as pd
from ._paths import ensure_v3
ensure_v3()
from data_audit import *  # noqa: F401,F403  (V3, giữ nguyên)
import data_audit as _V3  # noqa: F401

LABEL_MAP = {'benign': 0, 'malignant': 1}


def check_label_mapping(df):
    # benign = 0, malignant = 1; báo lỗi rõ ràng nếu có dòng sai
    bad = []
    for p, l in zip(df['pathology'].astype(str).str.lower(), df['label']):
        key = 'malignant' if p.startswith('m') else ('benign' if p.startswith('b') else None)
        if key is None or LABEL_MAP[key] != int(l):
            bad.append((p, l))
    if bad:
        raise ValueError(f'Mapping nhãn sai/lạ ở {len(bad)} dòng, ví dụ {bad[:3]}')
    return True


def dev_folds(df_split, k=5, seed=42):
    # gán fold 0..k-1 cho dữ liệu phát triển (train+val), nhóm theo cột 'group' (bệnh nhân + ảnh trùng); test = -1
    from sklearn.model_selection import StratifiedGroupKFold
    df = df_split.reset_index(drop=True)
    dev = np.where(df['split'].values != 'test')[0]
    fold = np.full(len(df), -1)
    sgk = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    for f, (_, te) in enumerate(sgk.split(dev, df['label'].values[dev], df['group'].values[dev])):
        fold[dev[te]] = f
    return fold


def oof_leakage_report(df_split, fold):
    df = df_split.reset_index(drop=True)
    rep = {}
    for col in ('group', 'patient_id', 'sha256'):
        if col not in df:
            continue
        d = df[fold >= 0]
        rep[f'{col}_across_folds'] = int((d.assign(f=fold[fold >= 0]).groupby(col)['f'].nunique() > 1).sum())
    rep['test_in_dev_folds'] = int(((df['split'] == 'test') & (fold >= 0)).sum())
    rep['ok'] = all(v == 0 for k, v in rep.items() if k != 'ok')
    return rep


# ------------------------------------------------------------------ kiểm toán manifest (chạy được không cần ảnh/torch)
def verify_manifest_sha(manifest_path, record_path):
    import hashlib, json
    got = hashlib.sha256(open(manifest_path, 'rb').read()).hexdigest()
    rec = json.load(open(record_path, encoding='utf-8'))
    return dict(ok=got == rec['sha256'], sha256=got, recorded=rec['sha256'], n_recorded=int(rec.get('n', -1)))


def check_pairing(df):
    """Ghép ảnh–mask theo tên tệp (bus_XXXX-l.png <-> mask_XXXX-l.png). Trả danh sách cặp không khớp."""
    import os, re
    bi = df['image_path'].map(lambda p: re.sub(r'^bus_', '', os.path.splitext(os.path.basename(p))[0]))
    bm = df['mask_path'].map(lambda p: re.sub(r'^mask_', '', os.path.splitext(os.path.basename(p))[0]))
    bad = np.where(bi.values != bm.values)[0]
    return dict(ok=len(bad) == 0, n_mismatch=int(len(bad)), examples=df['image_path'].iloc[bad[:3]].tolist(),
                image_unique=bool(df['image_path'].is_unique), mask_unique=bool(df['mask_path'].is_unique))


def audit_manifest(df):
    per = [dict(split=k, n=int(len(g)), n_patients=int(g['patient_id'].nunique()), malignant_rate=float(g['label'].mean()), n_malignant=int(g['label'].sum()))
           for k, g in df.groupby('split')]
    return dict(n=int(len(df)), n_patients=int(df['patient_id'].nunique()), exact_duplicate_sha256=int(df['sha256'].duplicated().sum()),
                empty_or_bad_masks=int(((df['mask_ratio'] <= 0) | df['mask_ratio'].isna()).sum()), per_split=per,
                issues=int((df['issue'].fillna('') != '').sum()))


def fold_malignant_rates(df_split, fold):
    d = df_split.reset_index(drop=True)
    return {int(f): float(d['label'].values[fold == f].mean()) for f in sorted(set(fold[fold >= 0]))}

In [ ]:
%%writefile bsv4/src/evaluation.py
"""Chỉ số đánh giá (thuần numpy/sklearn). Đối chiếu với V3 (bus_trust) trong tests/."""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from .segmentation import dice, iou
from . import selective_prediction as SP
from . import statistical_tests as ST

EPS = 1e-12


def safe_auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else float('nan')


def safe_auprc(y, s):
    y = np.asarray(y)
    return float(average_precision_score(y, s)) if y.sum() > 0 else float('nan')


def classification_metrics(y, p_mal, thr=0.5):
    y = np.asarray(y).astype(int); p = np.asarray(p_mal, float)
    pred = (p >= thr).astype(int)
    tp, tn = int(((pred == 1) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum())
    fp, fn = int(((pred == 1) & (y == 0)).sum()), int(((pred == 0) & (y == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else float('nan')
    spec = tn / (tn + fp) if tn + fp else float('nan')
    return dict(n=len(y), accuracy=float((pred == y).mean()), auroc=safe_auroc(y, p), auprc=safe_auprc(y, p),
                f1=float(f1_score(y, pred, zero_division=0)), sensitivity=sens, specificity=spec,
                balanced_accuracy=float(np.nanmean([sens, spec])), tp=tp, tn=tn, fp=fp, fn=fn, threshold=float(thr))


def brier_binary(p_mal, y):
    return float(((np.asarray(p_mal, float) - np.asarray(y)) ** 2).mean())


def nll_binary(p_mal, y):
    p = np.clip(np.asarray(p_mal, float), EPS, 1 - EPS); y = np.asarray(y)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def ece_binary(p_mal, y, n_bins=10):
    """ECE theo confidence của lớp dự đoán — cùng định nghĩa bus_trust.ece (V3) để so sánh được; có test tương đương."""
    p = np.asarray(p_mal, float); y = np.asarray(y)
    P = np.stack([1 - p, p], 1)
    conf, pred = P.max(1), P.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(out)


def deletion_insertion_auc(img, predict_fn, rank_map, cls_sign=1.0, steps=10, baseline='blur', T=1.0):
    """Deletion/Insertion AUC trên xác suất lớp malignant (dùng margin logit/T -> sigmoid). rank_map (H,W): điểm cao = quan trọng.
    cls_sign=+1: theo dõi P(malignant); -1: theo dõi P(benign). Trả (deletion_auc, insertion_auc). Deletion thấp & insertion cao là tốt."""
    import cv2
    H, W = img.shape[:2]
    base = cv2.GaussianBlur(img, (0, 0), 8) if baseline == 'blur' else np.full_like(img, int(img.mean()))
    order = np.argsort(-np.asarray(rank_map, np.float64).ravel(), kind='stable')
    n = order.size
    fr = np.linspace(0, 1, steps + 1)
    dele, ins = [], []
    for f in fr:
        k = int(round(f * n))
        m = np.zeros(n, bool); m[order[:k]] = True; m = m.reshape(H, W)[..., None]
        dele.append(np.where(m, base, img)); ins.append(np.where(m, img, base))
    out = predict_fn(np.stack(dele + ins, 0))['logits']
    z = (out[:, 1] - out[:, 0]) / T * cls_sign
    p = 1 / (1 + np.exp(-np.clip(z, -50, 50)))
    return float(np.trapz(p[:len(fr)], fr)), float(np.trapz(p[len(fr):], fr))


def error_detection_table(scores, wrong, is_fn, groups, rates=(0.1, 0.2, 0.3), B=1000, seed=0, primary_rate=0.2, ci=True):
    """scores: dict tên -> mảng điểm (cao = nguy cơ sai). Trả DataFrame: Error AUROC/AUPRC (CI cụm), FN Alert Recall tại mỗi tỉ lệ cảnh báo, AURC, AUGRC.
    So sánh CÙNG tỉ lệ cảnh báo: mỗi phương pháp cảnh báo đúng ceil(rate*n) ca điểm cao nhất (tie ngẫu nhiên có seed)."""
    wrong = np.asarray(wrong).astype(int); is_fn = np.asarray(is_fn).astype(bool)
    rows = []
    for name, s in scores.items():
        s = np.asarray(s, float)
        row = dict(method=name, n=len(s), n_errors=int(wrong.sum()), n_fn=int(is_fn.sum()), error_prevalence=float(wrong.mean()),
                   error_auroc=safe_auroc(wrong, s), error_auprc=safe_auprc(wrong, s), aurc=SP.aurc(s, wrong), augrc=SP.augrc(s, wrong))
        for r in rates:
            a = SP.alert_set(s, r, rng=np.random.default_rng(seed))
            m = SP.alert_metrics(a, wrong, is_fn)
            row[f'fn_alert_recall@{int(r*100)}'] = m['fn_alert_recall']; row[f'error_recall@{int(r*100)}'] = m['error_recall']
            row[f'fn_caught@{int(r*100)}'] = m['fn_caught']
        if ci:
            e = ST.bootstrap_ci(lambda i: safe_auprc(wrong[i], s[i]), groups, B=B, seed=seed)
            row['error_auprc_lo'], row['error_auprc_hi'] = e['lo'], e['hi']
            def fnr(i, s=s):
                a = SP.alert_set(s[i], primary_rate, rng=np.random.default_rng(seed))
                nf = is_fn[i].sum()
                return float((a & is_fn[i]).sum() / nf) if nf else float('nan')
            f = ST.bootstrap_ci(fnr, groups, B=B, seed=seed)
            row[f'fn_alert_recall@{int(primary_rate*100)}_lo'], row[f'fn_alert_recall@{int(primary_rate*100)}_hi'] = f['lo'], f['hi']
        rows.append(row)
    return pd.DataFrame(rows)


def ood_confound_analysis(records):
    """Phân tích gây nhiễu OOD của can thiệp: nếu |Δlogit| của đối chứng tương quan mạnh với 'thay đổi pixel'/'seam', thì độ nhạy của
    can thiệp một phần là artefact. records: DataFrame từ verify_image(return_records=True) gộp nhiều ảnh (cột group,absdz,mean_abs_change,seam_ratio).
    Trả Spearman(|Δz|, mean_abs_change) và Spearman(|Δz|, seam_ratio) theo nhóm, và so sánh target vs matched sau khi chia tầng theo thay đổi pixel."""
    from scipy.stats import spearmanr
    out = {}
    for g, d in records.groupby('group'):
        if len(d) > 10:
            out[f'{g}_spearman_absdz_vs_pixelchange'] = float(spearmanr(d['absdz'], d['mean_abs_change'])[0])
            out[f'{g}_spearman_absdz_vs_seam'] = float(spearmanr(d['absdz'], d['seam_ratio'])[0])
    t, m = records[(records.group == 'region') & (records.region == 'lesion')], records[(records.group == 'matched') & (records.region == 'lesion')]
    if len(t) > 20 and len(m) > 20:
        q = np.quantile(pd.concat([t['mean_abs_change'], m['mean_abs_change']]), [0, .25, .5, .75, 1.0])
        diffs = []
        for lo, hi in zip(q[:-1], q[1:]):
            a = t[(t.mean_abs_change >= lo) & (t.mean_abs_change <= hi)]['absdz']
            b = m[(m.mean_abs_change >= lo) & (m.mean_abs_change <= hi)]['absdz']
            if len(a) > 5 and len(b) > 5:
                diffs.append(float(a.mean() - b.mean()))
        out['lesion_minus_matched_absdz_within_pixelchange_strata'] = diffs
    return out

In [ ]:
%%writefile bsv4/src/evidence_maps.py
"""Tiện ích bản đồ bằng chứng (numpy). Evidence do mô hình V3 tạo (lưới h×w, tổng = 1); Grad-CAM nằm ở bus_core_v2 (cần torch)."""
import numpy as np
import cv2


def upsample_evidence(E, size_hw):
    E = np.asarray(E, np.float32)
    out = cv2.resize(E, (size_hw[1], size_hw[0]), interpolation=cv2.INTER_LINEAR)
    s = out.sum()
    return out / s if s > 0 else out


def evidence_mass_in(E, region, valid=None):
    """Tỉ lệ khối lượng evidence nằm trong `region` (mặt nạ DỰ ĐOÁN khi suy luận)."""
    E = np.asarray(E, np.float64)
    if valid is not None:
        E = E * valid
    s = E.sum()
    return float((E * region).sum() / s) if s > 0 else float('nan')


def evidence_entropy_norm(E):
    p = np.asarray(E, np.float64).ravel()
    p = p / max(p.sum(), 1e-12)
    h = -(p * np.log(np.clip(p, 1e-12, 1))).sum()
    return float(h / np.log(p.size))


def cosine(a, b, eps=1e-8):
    a, b = np.asarray(a, np.float64).ravel(), np.asarray(b, np.float64).ravel()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + eps))


def oracle_localisation(E, gt_mask, valid=None):
    """ORACLE (đánh giá offline, báo cáo tách riêng): khối lượng evidence trong GT lesion và 'area-adjusted' (0 = ngẫu nhiên)."""
    gt = np.asarray(gt_mask) > 0
    mass = evidence_mass_in(E, gt, valid)
    area = float(gt.mean())
    return dict(mass_in_gt=mass, area_adjusted=(mass - area) / max(1 - area, 1e-8))

In [ ]:
%%writefile bsv4/src/failure_risk.py
"""Failure-Risk Detection Engine: mô hình phụ dự đoán P(mô hình chính dự đoán SAI | đặc trưng).
Nguyên tắc: (1) nhãn mục tiêu = mô hình chính đúng/sai trên dự đoán OUT-OF-FOLD; (2) KHÔNG test labels trong huấn luyện/chọn đặc trưng/chọn C/
ngưỡng (có chốt chặn `assert_no_test`); (3) logistic regression L2 là mặc định; GBM chỉ so sánh khi đủ số ca lỗi; (4) risk != P(malignant)."""
import os, json, hashlib, time
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss

SCHEMA_VERSION = 'bus-verify-risk-1'

G_CONF = ['p_mal', 'margin_abs', 'entropy', 'conf', 'thr_margin']
G_UNC = ['view_js', 'view_dz_sd', 'ens_js', 'ens_dz_sd', 'ens_flip']      # ens_* chỉ có khi chạy ensemble thật; cột thiếu bị bỏ khi dựng
G_STAB = ['prediction_stability', 'view_dz_absmax', 'mask_consistency', 'evidence_stability']
G_CF = ['lesion_sens', 'peri_sens', 'bg_sens', 'lesion_dir', 'lesion_bg_contrast', 'lesion_peri_contrast', 'cf_specificity', 'cf_z',
        'cf_flip_rate', 'cf_consistency', 'lesion_flip', 'bg_flip', 'ev_faithfulness', 'cf_missing']
G_SHORTCUT = ['pad_sens', 'annot_sens']          # thăm dò, ngoài P1 chính
FEATURE_SETS = {
    'B4_conf_only': G_CONF,
    'B5_cf_only': G_CF,
    'B6_unc_stab_no_cf': G_CONF + G_UNC + G_STAB,
    'P1_full': G_CONF + G_UNC + G_STAB + G_CF,
}
ABLATIONS = {                                    # P1 bỏ từng nhóm đặc trưng (nhóm con theo yêu cầu)
    'P1-lesion_sens': ['lesion_sens', 'lesion_dir', 'lesion_bg_contrast', 'lesion_peri_contrast', 'cf_specificity', 'cf_z', 'lesion_flip'],
    'P1-background_sens': ['bg_sens', 'bg_flip', 'lesion_bg_contrast'],
    'P1-uncertainty': G_UNC,
    'P1-stability': G_STAB,
    'P1-counterfactual': G_CF,
}


class RiskModelMissing(RuntimeError):
    pass


class PolicyTamperedError(RuntimeError):
    pass


class TestLeakageError(RuntimeError):
    __test__ = False


def assert_no_test(split_labels):
    s = set(map(str, np.unique(np.asarray(split_labels))))
    if 'test' in s or 'external' in s:
        raise TestLeakageError(f'Dữ liệu test/external không được dùng để huấn luyện/chọn tham số/ngưỡng (phát hiện nhãn tập: {sorted(s)}).')


def add_derived_features(df):
    """Tính cột phái sinh từ đặc trưng thô: margin_abs, conf, entropy (nhị phân), cf_missing."""
    d = df.copy()
    if 'p_mal' in d:
        p = d['p_mal'].clip(1e-9, 1 - 1e-9)
        d['conf'] = np.maximum(p, 1 - p)
        d['entropy'] = -(p * np.log(p) + (1 - p) * np.log(1 - p))
        d['margin_abs'] = np.abs(np.log(p / (1 - p)))
    if 'lesion_sens' in d:
        d['cf_missing'] = d['lesion_sens'].isna().astype(float)
    return d


def dedupe_rank_equivalent(df, names, tol=1e-9):
    """Bỏ đặc trưng có |Spearman| = 1 với đặc trưng đã giữ (nhị phân: conf, entropy, margin_abs đơn điệu với nhau). Trả (giữ, bỏ{tên: tương đương với})."""
    keep, dropped = [], {}
    for n in names:
        if n not in df:
            continue
        v = df[n]
        dup = None
        for k in keep:
            ok = v.notna() & df[k].notna()
            if ok.sum() > 5 and abs(v[ok].rank().corr(df[k][ok].rank()) ) > 1 - tol:
                dup = k; break
        if dup is None:
            keep.append(n)
        else:
            dropped[n] = dup
    return keep, dropped


def _slog(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_pipeline(C):
    return Pipeline([('imp', SimpleImputer(strategy='median', keep_empty_features=True)), ('slog', FunctionTransformer(_slog)),
                     ('sc', StandardScaler()), ('lr', LogisticRegression(C=C, max_iter=2000))])


def _folds(y, groups, k, seed):
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, groups))


def oof_risk(X, y, groups, C, k=5, seed=0):
    X = np.asarray(X, float); y = np.asarray(y).astype(int)
    out = np.full(len(y), np.nan)
    for tr, te in _folds(y, groups, k, seed):
        m = make_pipeline(C).fit(X[tr], y[tr])
        out[te] = m.predict_proba(X[te])[:, 1]
    return out


def select_C(X, y, groups, grid=(0.01, 0.03, 0.1, 0.3, 1.0, 3.0), k=5, seed=0):
    """Chọn C bằng log-loss out-of-fold theo cụm TRÊN DỮ LIỆU PHÁT TRIỂN."""
    res = {}
    for C in grid:
        p = np.clip(oof_risk(X, y, groups, C, k, seed), 1e-6, 1 - 1e-6)
        res[C] = float(log_loss(y, p))
    best = min(res, key=res.get)
    return best, res


class RiskCalibrator:
    def __init__(self, method='auto', n_err=0):
        self.method = ('isotonic' if n_err >= 100 else 'sigmoid') if method == 'auto' else method
        self.iso = None; self.coef = None

    @staticmethod
    def _logit(p):
        p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    def fit(self, raw, y):
        if self.method == 'isotonic':
            self.iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(raw, y)
        else:
            lr = LogisticRegression(C=1e6, max_iter=1000).fit(self._logit(raw)[:, None], y)
            self.coef = (float(lr.coef_[0, 0]), float(lr.intercept_[0]))
        return self

    def __call__(self, raw):
        if self.method == 'isotonic':
            return self.iso.predict(np.asarray(raw, float))
        a, b = self.coef
        return 1 / (1 + np.exp(-(a * self._logit(raw) + b)))


class FailureRiskModel:
    def __init__(self, feature_names, seed=0, C=None, k=5):
        self.feature_names = list(feature_names); self.seed = seed; self.C = C; self.k = k
        self.pipe = None; self.calib = None; self.card = {}

    def fit(self, X_df, wrong, groups, split_labels):
        assert_no_test(split_labels)
        X = X_df[self.feature_names].to_numpy(float)
        y = np.asarray(wrong).astype(int)
        if y.sum() < 10 or (len(y) - y.sum()) < 10:
            raise ValueError(f'Quá ít ca lỗi/đúng để huấn luyện bộ phát hiện (lỗi={int(y.sum())}, đúng={int(len(y)-y.sum())}).')
        if self.C is None:
            self.C, grid = select_C(X, y, groups, k=self.k, seed=self.seed)
        else:
            grid = {}
        oof = oof_risk(X, y, groups, self.C, self.k, self.seed)
        self.calib = RiskCalibrator('auto', int(y.sum())).fit(oof, y)
        self.pipe = make_pipeline(self.C).fit(X, y)
        self.oof_raw = oof
        self.card = dict(schema=SCHEMA_VERSION, features=self.feature_names, C=float(self.C), C_grid_logloss=grid, n=int(len(y)), n_errors=int(y.sum()),
                         calibrator=self.calib.method, penalty='l2', trained_on_splits=sorted(map(str, np.unique(split_labels))),
                         created=time.strftime('%Y-%m-%d %H:%M:%S'))
        return self

    def _check(self):
        if self.pipe is None:
            raise RiskModelMissing('Mô hình risk chưa huấn luyện/chưa nạp.')

    def predict_raw(self, X_df):
        self._check()
        missing = [c for c in self.feature_names if c not in X_df]
        if missing:
            raise KeyError(f'Thiếu đặc trưng {missing}')
        return self.pipe.predict_proba(X_df[self.feature_names].to_numpy(float))[:, 1]

    def predict_risk(self, X_df):
        return self.calib(self.predict_raw(X_df))

    def contributions(self, x_row):
        """Đóng góp tuyến tính chính xác vào logit (sau impute/log/scale): coef_j * z_j. Tổng + intercept = logit thô."""
        self._check()
        x = np.asarray([x_row[c] for c in self.feature_names], float)[None]
        z = self.pipe[:-1].transform(x)[0]
        coef = self.pipe.named_steps['lr'].coef_[0]
        return pd.Series(coef * z, index=self.feature_names).sort_values(key=np.abs, ascending=False)

    def intercept(self):
        return float(self.pipe.named_steps['lr'].intercept_[0])

    def save(self, directory):
        import joblib
        os.makedirs(directory, exist_ok=True)
        p = os.path.join(directory, 'risk_model.joblib')
        joblib.dump(dict(pipe=self.pipe, calib=self.calib, feature_names=self.feature_names, card=self.card, schema=SCHEMA_VERSION), p)
        card = dict(self.card, sha256=sha256_file(p), coefficients=dict(zip(self.feature_names, map(float, self.pipe.named_steps['lr'].coef_[0]))),
                    intercept=self.intercept())
        json.dump(card, open(os.path.join(directory, 'risk_model_card.json'), 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
        return p

    @classmethod
    def load(cls, directory):
        import joblib
        p = os.path.join(directory, 'risk_model.joblib')
        if not os.path.exists(p):
            raise RiskModelMissing(f'Không có artifact risk model tại {p}. Chức năng risk score bị tắt (không giả lập).')
        d = joblib.load(p)
        if d.get('schema') != SCHEMA_VERSION:
            raise RiskModelMissing(f"schema không khớp: {d.get('schema')} != {SCHEMA_VERSION}")
        m = cls(d['feature_names']); m.pipe, m.calib, m.card = d['pipe'], d['calib'], d['card']
        cardp = os.path.join(directory, 'risk_model_card.json')
        if os.path.exists(cardp) and json.load(open(cardp, encoding='utf-8')).get('sha256') != sha256_file(p):
            raise RiskModelMissing('SHA256 của risk_model.joblib không khớp model card (artifact bị sửa).')
        return m


def fit_gbm_comparator(X_df, wrong, groups, feature_names, k=5, seed=0, min_errors=150):
    """So sánh gradient boosting CHỈ khi đủ ca lỗi; nếu không trả None kèm lý do (không ép chạy trên dữ liệu quá nhỏ)."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    y = np.asarray(wrong).astype(int)
    if y.sum() < min_errors:
        return None, f'bỏ qua GBM: chỉ {int(y.sum())} ca lỗi < {min_errors}'
    X = X_df[feature_names].to_numpy(float)
    oof = np.full(len(y), np.nan)
    for tr, te in _folds(y, groups, k, seed):
        m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, l2_regularization=1.0, random_state=seed).fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return oof, 'ok'


# ------------------------------------------------------------------ khóa ngưỡng cảnh báo
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def _canon(d):
    return json.dumps(d, sort_keys=True, ensure_ascii=False, default=str)


def threshold_for_alert_rate(risk_dev, rate):
    """Ngưỡng sao cho ~rate của dữ liệu PHÁT TRIỂN (OOF) bị cảnh báo: phân vị 1-rate."""
    return float(np.quantile(np.asarray(risk_dev, float), 1 - rate))


def lock_policy(path, content, force=False):
    """Ghi chính sách cảnh báo kèm sha256 nội dung. Không ghi đè khi đã tồn tại (trừ force) -> ngưỡng không bị đổi lặng lẽ."""
    if os.path.exists(path) and not force:
        raise PolicyTamperedError(f'{path} đã tồn tại: ngưỡng đã khóa, không được ghi đè.')
    payload = dict(content=content, content_sha256=hashlib.sha256(_canon(content).encode()).hexdigest(), locked_at=time.strftime('%Y-%m-%d %H:%M:%S'))
    json.dump(payload, open(path, 'w', encoding='utf-8'), indent=1, ensure_ascii=False, default=str)
    return payload['content_sha256']


def load_policy(path):
    if not os.path.exists(path):
        raise RiskModelMissing(f'Chưa có chính sách cảnh báo đã khóa: {path}')
    d = json.load(open(path, encoding='utf-8'))
    if hashlib.sha256(_canon(d['content']).encode()).hexdigest() != d['content_sha256']:
        raise PolicyTamperedError('Nội dung ngưỡng bị chỉnh sửa sau khi khóa (sha256 không khớp).')
    return d['content']


# ------------------------------------------------------------------ cổng phạm vi đầu vào (heuristic, cần được kiểm chứng)
def image_stats(img_uint8, valid=None):
    import cv2
    a = np.asarray(img_uint8)
    g = a.mean(2).astype(np.float32)
    v = np.ones(g.shape, bool) if valid is None else valid
    gv = g[v] if v.any() else g.ravel()
    lap = cv2.Laplacian(g, cv2.CV_32F).var()
    chroma = float(np.abs(a[..., 0].astype(np.float32) - a[..., 1]).mean() + np.abs(a[..., 1].astype(np.float32) - a[..., 2]).mean())
    return np.array([gv.mean(), gv.std(), float(v.mean()), np.log1p(lap), float((gv >= 250).mean()), float((gv <= 5).mean()), chroma], float)


class ImageStatGate:
    """Mahalanobis (co rút) trên thống kê ảnh, ngưỡng = phân vị q trên ảnh phát triển. Chỉ phát hiện bất thường thô (không phải ảnh siêu âm,
    cường độ/độ nét cực đoan) — KHÔNG đảm bảo phát hiện mọi dịch chuyển miền."""
    def __init__(self, q=0.995, shrink=0.1):
        self.q, self.shrink = q, shrink

    def fit(self, stats_dev):
        S = np.asarray(stats_dev, float)
        self.mu = S.mean(0)
        cov = np.cov(S.T) + 1e-6 * np.eye(S.shape[1])
        cov = (1 - self.shrink) * cov + self.shrink * np.diag(np.diag(cov))
        self.prec = np.linalg.inv(cov)
        self.thr = float(np.quantile(self.score(S), self.q))
        return self

    def score(self, S):
        d = np.asarray(S, float) - self.mu
        return np.sqrt(np.einsum('ij,jk,ik->i', d, self.prec, d))

    def in_scope(self, s):
        return bool(self.score(np.asarray(s)[None])[0] <= self.thr)

    def to_dict(self):
        return dict(mu=self.mu.tolist(), prec=self.prec.tolist(), thr=self.thr, q=self.q, shrink=self.shrink)

    @classmethod
    def from_dict(cls, d):
        g = cls(d['q'], d['shrink']); g.mu = np.array(d['mu']); g.prec = np.array(d['prec']); g.thr = d['thr']
        return g

In [ ]:
%%writefile bsv4/src/inference.py
"""Bộ chuyển đổi PyTorch -> predict_fn của BUS-VERIFY. Cần torch (Kaggle GPU hoặc Windows CPU). Tái dùng nguyên module V3 đã kiểm thử.
`logits` trả về = log(prob) để T-scaling khớp ĐÚNG pipeline V3 (softmax(log p / T)); với TTA, prob là trung bình hai góc nhìn như predict_arrays V3."""
import numpy as np
from ._paths import ensure_v3
ensure_v3()


class TorchPredictor:
    def __init__(self, model, device, tta=False, batch_size=32, amp=False, meta=None):
        import torch
        self.torch = torch
        self.model, self.device, self.tta, self.bs, self.amp, self.meta = model.eval().to(device), device, bool(tta), batch_size, amp, meta or {}

    def __call__(self, images):
        import torch
        import torch.nn.functional as F
        import bus_core as core
        probs, segp, evid = [], [], []
        for i in range(0, len(images), self.bs):
            x = torch.stack([core.normalize_to_tensor(a) for a in images[i:i + self.bs]]).to(self.device)
            with torch.no_grad(), torch.autocast(device_type=self.device.type, enabled=self.amp):
                o = self.model(x)
                of = self.model(x.flip(-1)) if self.tta else None
            p = F.softmax(o['logits'].float(), 1)
            if self.tta:
                p = 0.5 * (p + F.softmax(of['logits'].float(), 1))
            probs.append(p.cpu().numpy())
            if o.get('seg') is not None:
                sp = torch.sigmoid(o['seg'].float())
                if self.tta:
                    sp = 0.5 * (sp + torch.sigmoid(of['seg'].float()).flip(-1))
                segp.append(sp.squeeze(1).cpu().numpy().astype(np.float32))
            if o.get('evidence') is not None:
                ev = o['evidence'].float()
                if self.tta:
                    ev = 0.5 * (ev + of['evidence'].float().flip(-1))
                    ev = ev / (ev.sum((2, 3), keepdim=True) + 1e-8)
                evid.append(ev.cpu().numpy()[:, 0])
        cat = lambda a: np.concatenate(a) if a else None
        P = cat(probs)
        return dict(logits=np.log(np.clip(P, 1e-12, 1.0)), seg_prob=cat(segp), evidence=cat(evid))

    def gradcam(self, img):
        import bus_core as core
        import bus_core_v2 as C2
        x = core.normalize_to_tensor(img).unsqueeze(0).to(self.device)
        cam, _ = C2.gradcam_dict(self.model, x)
        return cam


def load_predictor(ckpt_path, device, tta=None):
    import bus_core_v2 as C2
    import models_xai as MX
    model, meta = C2.load_model_from_checkpoint(ckpt_path, device)
    if tta is None:
        tta = bool(MX.EXPERIMENTS.get(meta.get('experiment'), {}).get('tta', False))
    return TorchPredictor(model, device, tta=tta, meta=meta)

In [ ]:
%%writefile bsv4/src/models.py
"""Re-export kiến trúc V3 (src/v3/models_xai.py, bus_core.py) — giữ nguyên, đã kiểm thử 25/25 trong V3. Cần torch."""
from ._paths import ensure_v3
ensure_v3()
from models_xai import EXPERIMENTS, MAIN_CONFIGS, build_experiment      # noqa: E402,F401
from bus_core import build_effnet_b3, gradcam                           # noqa: E402,F401

In [ ]:
%%writefile bsv4/src/pipeline.py
"""Pipeline BUS-VERIFY dùng CHUNG cho notebook Kaggle và ứng dụng Streamlit (đảm bảo parity). Không import torch: mô hình được truyền như callable."""
import os, json, time
import numpy as np
import pandas as pd

from . import preprocessing as PP
from . import segmentation as SG
from . import uncertainty as UN
from . import robustness as RB
from . import counterfactual_verifier as CF
from . import failure_risk as FR
from . import selective_prediction as SP


def extract_features(img, predict_fn, T=1.0, thr=0.5, seg_thr=0.5, min_area_frac=0.0015, cf_cfg=None, seed=0, with_shortcut=False, with_cf=True, members=None):
    """Đặc trưng cho MỘT ảnh đã tiền xử lý (uint8 HxWx3). KHÔNG nhận ground-truth. Trả (dict hàng, dict phụ cho giao diện)."""
    cf_cfg = cf_cfg or CF.CFConfig(seed=seed)
    valid = PP.valid_content_mask(img)
    stab = RB.stability_features(img, predict_fn, T=T, thr=thr, seg_thr=seg_thr, valid=valid)
    base = stab['base']
    P = UN.softmax(base['logits'][None], T)[0]
    z0 = float(UN.margin(base['logits'], T)); zthr = float(np.log(thr / (1 - thr)))
    # p_raw: softmax CHƯA hiệu chuẩn (dùng cho baseline B1); thr_margin: khoảng cách tới NGƯỠNG QUYẾT ĐỊNH (khác |logit| khi thr != 0.5)
    row = dict(p_mal=float(P[1]), p_raw=float(UN.softmax(base['logits'][None], 1.0)[0, 1]), pred=int(P[1] >= thr), z=z0, thr_margin=abs(z0 - zthr))
    if members:                                  # ensemble THẬT: các mô hình cùng fold khác seed (không bao gồm mô hình chính)
        ml = np.stack([np.asarray(f(img[None])['logits'][0], np.float64) for f in members] + [np.asarray(base['logits'], np.float64)], 0)
        row.update(UN.ensemble_features(ml, T, thr))
    row.update(stab['features'])
    mask, info = (None, dict(empty=True, area_frac=0.0, n_components=0))
    if base['seg_prob'] is not None:
        mask, info = SG.postprocess_mask(base['seg_prob'], seg_thr, min_area_frac)
    row.update(mask_area_frac=info['area_frac'], mask_components=info['n_components'], mask_empty=int(info['empty']))
    extra = dict(mask=mask, valid=valid, seg_prob=base['seg_prob'], evidence=base['evidence'], mask_info=info, view_probs=stab['probs'])
    if with_cf and mask is not None:
        res = CF.verify_image(img, mask, predict_fn, T=T, thr=thr, cfg=cf_cfg, valid=valid, evidence=base['evidence'],
                              rng=np.random.default_rng([seed, 17]), return_records=True)
        row.update(res['features']); row['cf_status'] = res['status']
        extra['cf_records'] = res.get('records', [])
    else:
        row['cf_status'] = 'skipped' if mask is not None else 'no_segmentation'
    if with_shortcut:
        row.update(CF.shortcut_probe(img, valid, predict_fn, T, thr))
    row = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in row.items()}
    return row, extra


def extract_dataframe(images, ids, predict_fn, base_seed=0, progress=None, **kw):  # kw: T, thr, members, with_cf, ...
    """images: iterable uint8 HxWx3 (đã letterbox+resize). Seed mỗi ảnh = [base_seed, idx] -> tái lập."""
    rows = []
    for i, (img, id_) in enumerate(zip(images, ids)):
        r, _ = extract_features(img, predict_fn, seed=int(base_seed) * 1000003 + i, **kw)
        r['id'] = id_
        rows.append(r)
        if progress and (i + 1) % 50 == 0:
            progress(i + 1)
    return add_derived(pd.DataFrame(rows))


def add_derived(df):
    return FR.add_derived_features(df)


def make_error_targets(y, p_mal, thr):
    pred = (np.asarray(p_mal) >= thr).astype(int)
    y = np.asarray(y).astype(int)
    return dict(pred=pred, wrong=(pred != y).astype(int), is_fn=((y == 1) & (pred == 0)).astype(int), is_fp=((y == 0) & (pred == 1)).astype(int))


class Bundle:
    """Gói triển khai đã khóa: cấu hình, mô hình risk, chính sách cảnh báo, cổng phạm vi. Thiếu artifact -> ném RiskModelMissing (KHÔNG giả lập)."""
    def __init__(self, directory):
        self.dir = directory
        cfgp = os.path.join(directory, 'deploy_config.json')
        if not os.path.exists(cfgp):
            raise FR.RiskModelMissing(f'Không có deploy_config.json trong {directory}')
        self.cfg = json.load(open(cfgp, encoding='utf-8'))
        self.risk = None; self.policy = None; self.gate = None; self.risk_error = None
        try:
            self.risk = FR.FailureRiskModel.load(os.path.join(directory, 'risk'))
            self.policy = FR.load_policy(os.path.join(directory, 'alert_policy.locked.json'))
            if self.policy.get('gate'):
                self.gate = FR.ImageStatGate.from_dict(self.policy['gate'])
        except (FR.RiskModelMissing, FR.PolicyTamperedError) as e:
            self.risk_error = str(e)                 # ghi lý do; giao diện hiển thị rõ, KHÔNG tạo điểm giả

    @property
    def risk_enabled(self):
        return self.risk is not None and self.policy is not None


def analyze_array(rgb, predict_fn, bundle, with_cf=True):
    """Phân tích một ảnh người dùng: validate -> tiền xử lý -> đặc trưng -> (nếu có artifact) risk + trạng thái. Lỗi ảnh -> InvalidImageError."""
    t0 = time.time()
    PP.validate_image(rgb)
    cfg = bundle.cfg
    img = PP.preprocess_image(rgb, int(cfg['img_size']))
    row, extra = extract_features(img, predict_fn, T=cfg['temperature'], thr=cfg['threshold'], seg_thr=cfg['seg_threshold'],
                                  min_area_frac=cfg['min_area_frac'], seed=int(cfg.get('seed', 0)), with_cf=with_cf or bundle.risk_enabled)   # risk cần đặc trưng CF: không bao giờ tính risk với CF bị bỏ trống
    df = add_derived(pd.DataFrame([row]))
    out = dict(image=img, row=df.iloc[0].to_dict(), extra=extra, p_malignant=float(row['p_mal']),
               label='malignant' if row['pred'] == 1 else 'benign', inference_seconds=time.time() - t0, risk=None, status=None, contributions=None)
    if bundle.risk_enabled:
        for c in bundle.risk.feature_names:
            if c not in df:
                df[c] = np.nan
        r = float(bundle.risk.predict_risk(df)[0])
        oos = SG.mask_plausibility(extra['mask_info']) if extra['mask'] is not None else ['no_segmentation']
        stats = FR.image_stats(img, extra['valid'])
        if bundle.gate is not None and not bundle.gate.in_scope(stats):
            oos.append('image_statistics_out_of_range')
        out.update(risk=r, oos_reasons=oos, status=SP.decide_status(r, bundle.policy['t_review'], bundle.policy['t_high'], in_scope=not oos),
                   contributions=bundle.risk.contributions(df.iloc[0]))
    return out

In [ ]:
%%writefile bsv4/src/preprocessing.py
"""Tiền xử lý thuần numpy/cv2 (KHÔNG cần torch). Bản sao hành vi của bus_core V3; tests/test_preprocessing_parity.py
đối chiếu từng hàm với mã V3 (trích bằng AST) để bảo đảm tương đương từng pixel."""
import numpy as np
import cv2

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class InvalidImageError(ValueError):
    """Ảnh đầu vào không hợp lệ (None, sai chiều, quá nhỏ, chứa NaN/Inf, hoặc đồng nhất)."""


def validate_image(img, min_side=32):
    """Kiểm tra ảnh. Ném InvalidImageError với thông báo rõ ràng; KHÔNG nuốt lỗi."""
    if img is None:
        raise InvalidImageError('ảnh là None')
    a = np.asarray(img)
    if a.ndim not in (2, 3):
        raise InvalidImageError(f'ảnh phải có 2 hoặc 3 chiều, nhận {a.ndim}')
    if a.ndim == 3 and a.shape[2] not in (1, 3, 4):
        raise InvalidImageError(f'số kênh không hỗ trợ: {a.shape[2]}')
    if min(a.shape[0], a.shape[1]) < min_side:
        raise InvalidImageError(f'ảnh quá nhỏ {a.shape[:2]} (< {min_side}px)')
    if a.dtype.kind == 'f' and not np.isfinite(a).all():
        raise InvalidImageError('ảnh chứa NaN/Inf')
    if a.size and float(a.max()) == float(a.min()):
        raise InvalidImageError('ảnh đồng nhất (một giá trị duy nhất)')
    return a


def ensure_rgb_uint8(img):
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[2] == 4:
        img = img[:, :, :3]
    if img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)
    if img.dtype != np.uint8:
        mx = float(img.max()) if img.size else 1.0
        img = (img.astype(np.float32) / (mx if mx > 0 else 1.0) * 255.0).clip(0, 255).astype(np.uint8)
    return img


def letterbox_square(arr, pad_value=0):
    h, w = arr.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    canvas = np.full((s, s) + arr.shape[2:], pad_value, dtype=arr.dtype)
    canvas[top:top + h, left:left + w] = arr
    return canvas


def preprocess_image(img_rgb, size):
    sq = letterbox_square(ensure_rgb_uint8(img_rgb))
    interp = cv2.INTER_AREA if sq.shape[0] > size else cv2.INTER_LINEAR
    return cv2.resize(sq, (size, size), interpolation=interp)


def preprocess_mask(mask_bin, size):
    sq = letterbox_square((np.asarray(mask_bin) > 0).astype(np.uint8) * 255)
    interp = cv2.INTER_AREA if sq.shape[0] > size else cv2.INTER_LINEAR
    r = cv2.resize(sq, (size, size), interpolation=interp)
    return (r > 127).astype(np.uint8)


def normalize_np(img_uint8):
    """Tương đương bus_core.normalize_to_tensor nhưng trả ndarray CHW float32."""
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return np.ascontiguousarray(x.transpose(2, 0, 1))


def valid_content_mask(img_uint8, pad_value=0):
    """Mặt nạ vùng có nội dung (loại dải đen letterbox ở biên: cột/hàng liên tiếp từ biên có toàn bộ pixel == pad_value).
    Ảnh siêu âm thật có thể có biên đen tự nhiên; khi đó vùng này cũng được coi là 'không phải mô' — là cố ý (xem tài liệu shortcut)."""
    a = np.asarray(img_uint8)
    g = a.max(axis=2) if a.ndim == 3 else a
    nz = g != pad_value
    H, W = g.shape
    if not nz.any():
        return np.zeros((H, W), bool)
    cols = np.where(nz.any(0))[0]
    rows = np.where(nz.any(1))[0]
    m = np.zeros((H, W), bool)
    m[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1] = True
    return m

In [ ]:
%%writefile bsv4/src/robustness.py
"""Robustness Lab: nhiễu/biến đổi ảnh có kiểm soát, tách LABEL-PRESERVING và COUNTERFACTUAL. Thuần numpy/cv2.

label_preserving = True : biến đổi không đổi thông tin bệnh lý (kỳ vọng nhãn giữ nguyên) — dùng đo độ ổn định.
label_preserving = False: can thiệp có thể xoá/đổi thông tin bệnh lý (ví dụ che lên tổn thương) — là phép THỬ phản thực, không đo "ổn định".
Che cục bộ ở NỀN (xa tổn thương) được xem là label-preserving; che đè lên mask dự đoán là counterfactual."""
import numpy as np
import cv2

CORRUPTIONS = {
    #  tên                : (mức 1..5 -> tham số)
    'speckle':   dict(label_preserving=True,  levels=[0.05, 0.10, 0.15, 0.20, 0.30], desc='nhiễu nhân (multiplicative) Gauss'),
    'gaussian':  dict(label_preserving=True,  levels=[3, 6, 10, 15, 25], desc='nhiễu cộng Gauss, sigma theo thang 0-255'),
    'brightness': dict(label_preserving=True, levels=[-40, -20, 20, 40, 60], desc='dịch độ sáng (cộng hằng số)'),
    'contrast':  dict(label_preserving=True,  levels=[0.5, 0.7, 0.85, 1.2, 1.5], desc='nhân tương phản quanh trung bình'),
    'blur':      dict(label_preserving=True,  levels=[0.7, 1.2, 1.8, 2.5, 3.5], desc='làm mờ Gauss, sigma px'),
    'geometric': dict(label_preserving=True,  levels=[2, 4, 6, 8, 12], desc='xoay (độ) kèm co giãn/dịch nhỏ'),
    'hflip':     dict(label_preserving=True,  levels=[1], desc='lật ngang'),
    'occlusion_background': dict(label_preserving=True, levels=[0.02, 0.04, 0.08, 0.12, 0.2], desc='che ô vuông ở NỀN (tỉ lệ diện tích)'),
    'occlusion_lesion': dict(label_preserving=False, levels=[0.25, 0.5, 0.75, 1.0], desc='COUNTERFACTUAL: che một phần mask dự đoán (tỉ lệ diện tích tổn thương)'),
}


def _affine_matrix(h, w, angle, scale, tx, ty):
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, scale)
    M[0, 2] += tx * w
    M[1, 2] += ty * h
    return M


def apply_corruption(img, name, level, rng=None, pred_mask=None, valid=None):
    """Trả (img_out uint8, M 2x3 hoặc None). M != None khi biến đổi hình học (để inverse-warp mặt nạ/evidence)."""
    rng = rng if rng is not None else np.random.default_rng(0)
    spec = CORRUPTIONS[name]
    p = spec['levels'][int(level) - 1] if name != 'hflip' else None
    x = img.astype(np.float32)
    H, W = img.shape[:2]
    M = None
    if name == 'speckle':
        x = x * (1.0 + p * rng.standard_normal(x.shape[:2])[..., None])
    elif name == 'gaussian':
        x = x + p * rng.standard_normal(x.shape)
    elif name == 'brightness':
        x = x + p
    elif name == 'contrast':
        m = x.mean()
        x = (x - m) * p + m
    elif name == 'blur':
        x = cv2.GaussianBlur(x, (0, 0), p)
    elif name == 'geometric':
        sgn = 1 if rng.random() < 0.5 else -1
        M = _affine_matrix(H, W, sgn * p, 1.0 + sgn * 0.01 * p, 0.005 * p * sgn, -0.004 * p * sgn)
        x = cv2.warpAffine(x, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    elif name == 'hflip':
        x = x[:, ::-1].copy()
        M = np.array([[-1, 0, W - 1], [0, 1, 0]], np.float64)
    elif name == 'occlusion_background':
        side = max(4, int(np.sqrt(p * H * W)))
        forb = np.zeros((H, W), bool) if pred_mask is None else (cv2.dilate((pred_mask > 0).astype(np.uint8), np.ones((15, 15), np.uint8)) > 0)
        ok = False
        for _ in range(200):
            y0, x0 = int(rng.integers(0, H - side + 1)), int(rng.integers(0, W - side + 1))
            box = np.zeros((H, W), bool); box[y0:y0 + side, x0:x0 + side] = True
            if not (box & forb).any() and (valid is None or not (box & ~valid).any()):
                ok = True; break
        if not ok:
            raise ValueError(f'occlusion_background mức {level}: không đặt được ô che {side}px tránh vùng tổn thương/ngoài nội dung (giảm mức hoặc bỏ phép này cho ảnh này)')
        x[box] = float(x[valid].mean()) if valid is not None and valid.any() else float(x.mean())
    elif name == 'occlusion_lesion':
        if pred_mask is None or not (pred_mask > 0).any():
            raise ValueError('occlusion_lesion cần mặt nạ dự đoán không rỗng')
        ys, xs = np.nonzero(pred_mask > 0)
        k = max(1, int(p * len(ys)))
        order = np.argsort(xs + 1e-3 * ys)          # che phần bên trái của tổn thương trước (xác định, tái lập)
        sel = order[:k]
        fill = float(x[valid].mean()) if valid is not None and valid.any() else float(x.mean())
        x[ys[sel], xs[sel]] = fill
    else:
        raise ValueError(name)
    return np.clip(x, 0, 255).astype(np.uint8), M


def inverse_warp_map(arr, M, out_hw):
    """Đưa bản đồ (mask/evidence) của ảnh đã biến đổi về hệ toạ độ ảnh gốc."""
    if M is None:
        return arr
    Mi = cv2.invertAffineTransform(np.asarray(M, np.float64))
    return cv2.warpAffine(np.asarray(arr, np.float32), Mi, (out_hw[1], out_hw[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def scale_affine_to_grid(M, s):
    """Chuyển affine (toạ độ pixel ảnh HxW) sang lưới thấp hơn có tỉ lệ s = h/H (quy ước tâm pixel của OpenCV):
    A giữ nguyên, t' = s*t + 0.5*(1-s)*(A@1 - 1). Kiểm chứng với lật ngang: ra đúng t' = w-1."""
    M = np.asarray(M, np.float64).copy()
    A = M[:, :2]
    one = np.ones(2)
    M[:, 2] = s * M[:, 2] + 0.5 * (1 - s) * (A @ one - one)
    return M


# 5 góc nhìn xác định (không ngẫu nhiên) dùng cho đặc trưng ổn định/bất đồng. Tất cả label-preserving.
STABILITY_VIEWS = (
    dict(name='hflip', kind='hflip'),
    dict(name='aff+', kind='affine', angle=4.0, scale=1.03, tx=0.02, ty=0.0),
    dict(name='aff-', kind='affine', angle=-4.0, scale=0.97, tx=-0.02, ty=0.01),
    dict(name='contrast', kind='contrast', gain=1.10),
    dict(name='bright', kind='brightness', shift=12.0),
)


def make_view(img, spec):
    H, W = img.shape[:2]
    k = spec['kind']
    if k == 'hflip':
        return img[:, ::-1].copy(), np.array([[-1, 0, W - 1], [0, 1, 0]], np.float64)
    if k == 'affine':
        M = _affine_matrix(H, W, spec['angle'], spec['scale'], spec['tx'], spec['ty'])
        return cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0), M
    x = img.astype(np.float32)
    if k == 'contrast':
        m = x.mean(); x = (x - m) * spec['gain'] + m
    elif k == 'brightness':
        x = x + spec['shift']
    return np.clip(x, 0, 255).astype(np.uint8), None


def stability_features(img, predict_fn, T=1.0, thr=0.5, pred_mask=None, seg_thr=0.5, valid=None):
    """Chạy ảnh gốc + STABILITY_VIEWS qua mô hình; trả đặc trưng bất định/ổn định cho MỘT ảnh và đầu ra gốc.
    Trả dict(features=..., base=dict(logits, seg_prob, evidence))."""
    from .uncertainty import softmax, js_disagreement, margin, view_stats
    from .segmentation import dice, postprocess_mask
    from .evidence_maps import cosine
    views = [make_view(img, s) for s in STABILITY_VIEWS]
    batch = np.stack([img] + [v[0] for v in views], 0)
    out = predict_fn(batch)
    L = np.asarray(out['logits'], np.float64)
    P = softmax(L, T)                                # (S+1, 2)
    z = margin(L, T)
    zthr = float(np.log(thr / (1 - thr)))
    vs = view_stats(z[:, None], zthr)
    feats = dict(view_js=float(js_disagreement(P[:, None, :])[0]), view_pred_flip=float(vs['view_pred_flip'][0]),
                 view_dz_sd=float(vs['view_dz_sd'][0]), view_dz_absmax=float(vs['view_dz_absmax'][0]))
    feats['prediction_stability'] = 1.0 - feats['view_pred_flip']
    nanf = float('nan')
    feats['mask_consistency'] = nanf
    feats['evidence_stability'] = nanf
    seg = out.get('seg_prob')
    if seg is not None:
        m0 = postprocess_mask(seg[0], seg_thr)[0]
        ds = []
        for i, (_, M) in enumerate(views, start=1):
            mi = postprocess_mask(inverse_warp_map(seg[i], M, seg[i].shape), seg_thr)[0]
            if M is not None:                       # chỉ so trong vùng giao hợp lệ của phép warp
                vm = inverse_warp_map(np.ones_like(seg[i], np.float32), M, seg[i].shape) > 0.99
                a, b = (m0 > 0) & vm, (mi > 0) & vm
                ds.append(dice(a, b))
            else:
                ds.append(dice(m0, mi))
        feats['mask_consistency'] = float(np.mean(ds))
    ev = out.get('evidence')
    if ev is not None:
        cs = []
        for i, (_, M) in enumerate(views, start=1):
            e0 = np.asarray(ev[0], np.float32)
            ei = ev[i]
            if M is not None:                       # evidence ở lưới thấp: warp tương ứng bằng M co theo tỉ lệ
                ei = inverse_warp_map(ev[i], scale_affine_to_grid(M, ev[i].shape[0] / img.shape[0]), ev[i].shape)
            cs.append(cosine(e0, ei))
        feats['evidence_stability'] = float(np.mean(cs))
    return dict(features=feats, base=dict(logits=L[0], seg_prob=None if seg is None else seg[0], evidence=None if ev is None else ev[0]), probs=P, z=z)


def run_robustness_lab(images, predict_fn, corruptions=None, levels=(1, 3, 5), seed=0, T=1.0, thr=0.5, seg_thr=0.5, min_area_frac=0.0015, risk_fn=None):
    """Chạy các biến đổi có kiểm soát trên danh sách ảnh đã tiền xử lý. Mỗi dòng: độ ổn định phân loại (|Δp|, flip), phân vùng (Dice với mask sạch,
    sau inverse-warp nếu hình học), evidence (cosine), risk (Δ nếu có risk_fn: img -> float). `label_preserving` tách rõ biến đổi bảo toàn nhãn
    khỏi can thiệp phản thực (occlusion_lesion). Ca không thực hiện được (không đặt được ô che, mask rỗng) được GHI `status` chứ không bị bỏ."""
    import pandas as pd
    from .uncertainty import softmax, margin
    from .segmentation import postprocess_mask, dice
    from .evidence_maps import cosine
    names = list(corruptions or CORRUPTIONS)
    rows = []
    for i, img in enumerate(images):
        valid = (img.max(2) != 0)
        o = predict_fn(img[None])
        p0 = float(softmax(o['logits'], T)[0, 1]); lab0 = p0 >= thr
        m0 = postprocess_mask(o['seg_prob'][0], seg_thr, min_area_frac)[0] if o.get('seg_prob') is not None else None
        e0 = None if o.get('evidence') is None else np.asarray(o['evidence'][0], np.float32)
        r0 = None if risk_fn is None else float(risk_fn(img))
        for name in names:
            for lv in levels:
                if lv > len(CORRUPTIONS[name]['levels']):
                    continue
                row = dict(image=i, corruption=name, level=int(lv), label_preserving=CORRUPTIONS[name]['label_preserving'], p0=p0, status='ok')
                rng = np.random.default_rng([seed, i, lv, __import__('zlib').crc32(name.encode()) % 10007])
                try:
                    out, M = apply_corruption(img, name, lv, rng, pred_mask=m0, valid=valid)
                except ValueError as e:
                    row['status'] = f'skipped: {e}'[:120]; rows.append(row); continue
                o1 = predict_fn(out[None])
                p1 = float(softmax(o1['logits'], T)[0, 1])
                row.update(p1=p1, abs_dp=abs(p1 - p0), flip=int((p1 >= thr) != lab0), mean_abs_pixel_change=float(np.abs(out.astype(np.float32) - img).mean()))
                if m0 is not None and o1.get('seg_prob') is not None:
                    sp = inverse_warp_map(o1['seg_prob'][0], M, o1['seg_prob'][0].shape)
                    m1 = postprocess_mask(sp, seg_thr, min_area_frac)[0]
                    if M is not None:
                        vm = inverse_warp_map(np.ones_like(sp, np.float32), M, sp.shape) > 0.99
                        row['seg_dice'] = dice((m0 > 0) & vm, (m1 > 0) & vm)
                    else:
                        row['seg_dice'] = dice(m0, m1)
                if e0 is not None and o1.get('evidence') is not None:
                    e1 = o1['evidence'][0]
                    if M is not None:
                        e1 = inverse_warp_map(e1, scale_affine_to_grid(M, e1.shape[0] / img.shape[0]), e1.shape)
                    row['evidence_cosine'] = cosine(e0, e1)
                if r0 is not None:
                    r1 = float(risk_fn(out)); row['risk0'], row['risk1'], row['abs_drisk'] = r0, r1, abs(r1 - r0)
                rows.append(row)
    return pd.DataFrame(rows)


def summarize_robustness(df):
    """Tóm tắt theo (corruption, level): flip rate, |Δp| trung bình, Dice, evidence cosine, |Δrisk|; tách label_preserving."""
    d = df[df['status'] == 'ok']
    agg = {c: 'mean' for c in ('abs_dp', 'flip', 'seg_dice', 'evidence_cosine', 'abs_drisk', 'mean_abs_pixel_change') if c in d}
    g = d.groupby(['label_preserving', 'corruption', 'level']).agg(n=('image', 'count'), **{k: (k, v) for k, v in agg.items()}).reset_index()
    sk = df[df['status'] != 'ok'].groupby(['corruption', 'level']).size().rename('n_skipped').reset_index()
    return g.merge(sk, on=['corruption', 'level'], how='left').fillna({'n_skipped': 0})

In [ ]:
%%writefile bsv4/src/segmentation.py
"""Hậu xử lý mặt nạ phân vùng dự đoán và dựng 3 vùng (lesion / perilesional / background). Thuần numpy/cv2.
QUY ƯỚC: mọi hàm ở đây nhận mặt nạ DỰ ĐOÁN. Mặt nạ ground-truth chỉ được dùng trong các hàm có tiền tố `oracle_`."""
from dataclasses import dataclass
import numpy as np
import cv2


def dice(a, b, eps=1e-8):
    a, b = np.asarray(a) > 0, np.asarray(b) > 0
    s = a.sum() + b.sum()
    if s == 0:
        return 1.0                       # cả hai rỗng: trùng khớp tuyệt đối (quy ước, được ghi trong báo cáo)
    return float(2.0 * (a & b).sum() / (s + eps))


def iou(a, b, eps=1e-8):
    a, b = np.asarray(a) > 0, np.asarray(b) > 0
    u = (a | b).sum()
    return 1.0 if u == 0 else float((a & b).sum() / (u + eps))


def postprocess_mask(seg_prob, thr=0.5, min_area_frac=0.0015, keep='largest'):
    """seg_prob (H,W) -> mask uint8 + info. keep='largest' giữ thành phần liên thông lớn nhất; 'all' giữ mọi thành phần >= min_area."""
    sp = np.asarray(seg_prob, dtype=np.float32)
    H, W = sp.shape
    m = (sp >= thr).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    min_area = max(1, int(round(min_area_frac * H * W)))
    comps = [(stats[i, cv2.CC_STAT_AREA], i) for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area]
    comps.sort(reverse=True)
    out = np.zeros_like(m)
    if comps:
        if keep == 'largest':
            out[lab == comps[0][1]] = 1
        else:
            for _, i in comps:
                out[lab == i] = 1
    info = dict(n_components=len(comps), area_frac=float(out.mean()), empty=not out.any(), min_area_px=min_area,
                n_raw_components=int(n - 1))
    return out, info


@dataclass
class Regions:
    lesion: np.ndarray
    peri: np.ndarray
    background: np.ndarray
    valid: np.ndarray
    ok: bool
    reason: str = ''


def _disk(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def build_regions(pred_mask, valid=None, ring_px=12, min_lesion_px=16):
    """Ba vùng rời nhau, hợp = vùng nội dung hợp lệ: lesion = mask dự đoán ∩ valid; peri = giãn(lesion, ring_px) \\ lesion ∩ valid;
    background = valid \\ (lesion ∪ peri). Mặt nạ rỗng/quá nhỏ -> ok=False (KHÔNG tự bịa vùng)."""
    pm = np.asarray(pred_mask) > 0
    H, W = pm.shape
    v = np.ones((H, W), bool) if valid is None else np.asarray(valid, bool)
    les = pm & v
    if les.sum() < min_lesion_px:
        z = np.zeros((H, W), bool)
        return Regions(z, z, v & ~z, v, False, 'empty_mask' if les.sum() == 0 else 'mask_too_small')
    dil = cv2.dilate(les.astype(np.uint8), _disk(int(ring_px))) > 0
    peri = dil & ~les & v
    bg = v & ~dil
    return Regions(les, peri, bg, v, True, '')


def regions_are_disjoint(r: Regions):
    a, b, c = r.lesion, r.peri, r.background
    return not ((a & b).any() or (a & c).any() or (b & c).any())


def mask_plausibility(info, min_frac=0.0015, max_frac=0.60, max_components=3):
    """Cờ 'ngoài phạm vi đã kiểm định' theo hình thái mặt nạ: rỗng, quá lớn, quá nhiều mảnh."""
    reasons = []
    if info['empty']:
        reasons.append('mask_empty')
    if info['area_frac'] > max_frac:
        reasons.append('mask_too_large')
    if info['n_components'] > max_components:
        reasons.append('mask_fragmented')
    return reasons


def oracle_mask_agreement(pred_mask, gt_mask):
    """ORACLE (chỉ dùng khi đánh giá offline, báo cáo riêng): Dice/IoU giữa mask dự đoán và GT."""
    return dict(dice=dice(pred_mask, gt_mask), iou=iou(pred_mask, gt_mask))

In [ ]:
%%writefile bsv4/src/selective_prediction.py
"""Selective prediction / cảnh báo nguy cơ sai. Thuần numpy.
Quy ước: `score` CAO = nguy cơ sai CAO (cảnh báo). `wrong` = 1 nếu mô hình chính dự đoán sai.
`is_fn` = ca ác tính bị dự đoán thành lành tính (false negative) — kết cục lâm sàng nặng nhất."""
import numpy as np

STATUS_NONE = 'Không phát cảnh báo'
STATUS_HIGH = 'Nguy cơ dự đoán sai cao'
STATUS_REVIEW = 'Cần xem xét lại'
STATUS_OOS = 'Đầu vào ngoài phạm vi đã kiểm định'


def _order_by_score(score, rng=None):
    s = np.asarray(score, float)
    if rng is None:
        return np.argsort(-s, kind='stable')
    return np.lexsort((rng.random(len(s)), -s))      # tie ngẫu nhiên có seed


def risk_coverage_curve(score, wrong, rng=None):
    """Nhận ca có risk THẤP nhất trước. Trả coverage (k/n), selective risk (lỗi/k), generalized risk (lỗi nhận/n)."""
    s = np.asarray(score, float); w = np.asarray(wrong, float)
    order = np.argsort(s, kind='stable') if rng is None else np.lexsort((rng.random(len(s)), s))
    cum = np.cumsum(w[order])
    k = np.arange(1, len(s) + 1)
    return k / len(s), cum / k, cum / len(s)


def aurc(score, wrong):
    """AURC = trung bình selective risk qua mọi coverage k/n (Geifman & El-Yaniv)."""
    _, sr, _ = risk_coverage_curve(score, wrong)
    return float(sr.mean())


def augrc(score, wrong):
    """AUGRC (Traub et al., 2024) = trung bình generalized risk = P(được nhận VÀ sai) qua mọi coverage."""
    _, _, gr = risk_coverage_curve(score, wrong)
    return float(gr.mean())


def oracle_aurc(wrong):
    """AURC lý tưởng (mọi ca đúng được nhận trước): cận dưới để diễn giải."""
    w = np.asarray(wrong, float)
    return aurc(w, w)


def alert_set(score, alert_rate, rng=None):
    """Chọn ceil(rate*n) ca có score cao nhất (so sánh CÙNG tỉ lệ cảnh báo giữa các phương pháp)."""
    n = len(score)
    k = int(np.ceil(alert_rate * n - 1e-9))
    a = np.zeros(n, bool)
    a[_order_by_score(score, rng)[:k]] = True
    return a


def alert_metrics(alert, wrong, is_fn):
    alert = np.asarray(alert, bool); wrong = np.asarray(wrong, bool); is_fn = np.asarray(is_fn, bool)
    n = len(alert)
    nw, nf = int(wrong.sum()), int(is_fn.sum())
    out = dict(n=n, alert_rate=float(alert.mean()), n_errors=nw, n_fn=nf,
               error_recall=float((alert & wrong).sum() / nw) if nw else float('nan'),
               error_precision=float((alert & wrong).sum() / alert.sum()) if alert.any() else float('nan'),
               fn_alert_recall=float((alert & is_fn).sum() / nf) if nf else float('nan'),
               fn_caught=int((alert & is_fn).sum()),
               residual_error_rate=float((wrong & ~alert).sum() / max((~alert).sum(), 1)),
               residual_fn=int((is_fn & ~alert).sum()))
    return out


def random_referral_expectation(wrong, is_fn, alert_rate, n_draws=500, seed=0):
    """B0: cảnh báo ngẫu nhiên. Kỳ vọng error_recall = fn_alert_recall = alert_rate; ước lượng Monte-Carlo để kiểm chứng + AUPRC kỳ vọng."""
    rng = np.random.default_rng(seed)
    r, f = [], []
    for _ in range(n_draws):
        a = alert_set(rng.random(len(wrong)), alert_rate)
        m = alert_metrics(a, wrong, is_fn)
        r.append(m['error_recall']); f.append(m['fn_alert_recall'])
    return dict(error_recall=float(np.mean(r)), fn_alert_recall=float(np.nanmean(f)))


def decide_status(risk, t_review, t_high, in_scope=True, oos_reasons=()):
    """Bốn trạng thái. Ngưỡng t_review <= t_high đã KHÓA từ dữ liệu phát triển (không từ test)."""
    if not in_scope:
        return STATUS_OOS
    if risk >= t_high:
        return STATUS_HIGH
    if risk >= t_review:
        return STATUS_REVIEW
    return STATUS_NONE

In [ ]:
%%writefile bsv4/src/statistical_tests.py
"""Thống kê: bootstrap theo cụm (bệnh nhân), bootstrap ghép cặp, kiểm định hoán vị theo cụm, hiệu chỉnh đa so sánh, effect size, CI Wilson.
Khi KHÔNG có mã bệnh nhân hợp lệ (BUSI), nhóm phải là cụm gần-trùng; nếu không, CI là CI mức ảnh và PHẢI được ghi rõ là lạc quan."""
import numpy as np


def _cluster_index(groups):
    g = np.asarray(groups)
    uniq, inv = np.unique(g, return_inverse=True)
    members = [np.where(inv == i)[0] for i in range(len(uniq))]
    return members


def cluster_bootstrap_indices(groups, rng):
    """Một mẫu bootstrap: lấy có hoàn lại các CỤM (bệnh nhân), nối mọi ảnh của cụm."""
    members = _cluster_index(groups)
    pick = rng.integers(0, len(members), size=len(members))
    return np.concatenate([members[i] for i in pick])


def bootstrap_ci(metric_fn, groups, B=1000, seed=0, alpha=0.05):
    """CI percentile cho metric_fn(idx) -> float (hoặc NaN nếu không xác định; NaN bị bỏ và đếm lại)."""
    rng = np.random.default_rng(seed)
    base = metric_fn(np.arange(len(groups)))
    vals = []
    for _ in range(B):
        v = metric_fn(cluster_bootstrap_indices(groups, rng))
        if np.isfinite(v):
            vals.append(v)
    vals = np.asarray(vals)
    if len(vals) < max(20, 0.5 * B):
        return dict(estimate=float(base), lo=float('nan'), hi=float('nan'), n_valid=int(len(vals)))
    return dict(estimate=float(base), lo=float(np.quantile(vals, alpha / 2)), hi=float(np.quantile(vals, 1 - alpha / 2)),
                se=float(vals.std(ddof=1)), n_valid=int(len(vals)))


def paired_cluster_bootstrap(metric_a, metric_b, groups, B=2000, seed=0, alpha=0.05):
    """Δ = metric_a - metric_b trên CÙNG các mẫu bootstrap cụm. p hai phía = 2*min(P(Δ*<=0), P(Δ*>=0)) (bootstrap, xấp xỉ).
    Trả cả effect size chuẩn hoá Δ/SE_boot."""
    rng = np.random.default_rng(seed)
    n = len(groups)
    d0 = metric_a(np.arange(n)) - metric_b(np.arange(n))
    ds = []
    for _ in range(B):
        idx = cluster_bootstrap_indices(groups, rng)
        a, b = metric_a(idx), metric_b(idx)
        if np.isfinite(a) and np.isfinite(b):
            ds.append(a - b)
    ds = np.asarray(ds)
    if len(ds) < 20:
        return dict(delta=float(d0), lo=float('nan'), hi=float('nan'), p=float('nan'), z=float('nan'), n_valid=int(len(ds)))
    p = 2 * min((ds <= 0).mean(), (ds >= 0).mean())
    se = ds.std(ddof=1)
    return dict(delta=float(d0), lo=float(np.quantile(ds, alpha / 2)), hi=float(np.quantile(ds, 1 - alpha / 2)), p=float(min(1.0, max(p, 1.0 / (len(ds) + 1)))),
                se=float(se), z=float(d0 / se) if se > 0 else float('nan'), n_valid=int(len(ds)))


def paired_cluster_sign_flip_test(score_a, score_b, metric_of_scores, groups, B=2000, seed=0):
    """Kiểm định hoán vị ghép cặp: với mỗi cụm, hoán đổi ngẫu nhiên (score_a, score_b); p = P(|Δ*| >= |Δ_obs|) có hiệu chỉnh +1.
    H0: hai điểm số có cùng phân phối ghép cặp trong cụm. metric_of_scores(score) -> float."""
    rng = np.random.default_rng(seed)
    a, b = np.asarray(score_a, float), np.asarray(score_b, float)
    members = _cluster_index(groups)
    obs = metric_of_scores(a) - metric_of_scores(b)
    cnt = 0
    for _ in range(B):
        flip = rng.random(len(members)) < 0.5
        aa, bb = a.copy(), b.copy()
        for f, m in zip(flip, members):
            if f:
                aa[m], bb[m] = b[m], a[m]
        d = metric_of_scores(aa) - metric_of_scores(bb)
        cnt += abs(d) >= abs(obs) - 1e-12
    return dict(delta=float(obs), p=float((cnt + 1) / (B + 1)))


def holm_bonferroni(pvals, alpha=0.05):
    """Holm step-down. Trả (p_adj, reject)."""
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj, adj < alpha


def benjamini_hochberg(pvals, alpha=0.05):
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    prev = 1.0
    for rank in range(m - 1, -1, -1):
        i = order[rank]
        prev = min(prev, p[i] * m / (rank + 1))
        adj[i] = prev
    return adj, adj < alpha


def wilson_ci(k, n, alpha=0.05):
    """CI Wilson cho tỉ lệ (dùng cho FN Alert Recall khi n_FN nhỏ)."""
    if n == 0:
        return float('nan'), float('nan')
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return float(max(0, c - h)), float(min(1, c + h))


def min_detectable_n(p_null, p_alt, alpha=0.05, power=0.8):
    """Cỡ mẫu FN (xấp xỉ chuẩn, một mẫu) để phân biệt recall p_alt với p_null — dùng để chỉ ra giới hạn công suất của tập test nhỏ."""
    from scipy.stats import norm
    za, zb = norm.ppf(1 - alpha / 2), norm.ppf(power)
    num = (za * np.sqrt(p_null * (1 - p_null)) + zb * np.sqrt(p_alt * (1 - p_alt))) ** 2
    return int(np.ceil(num / (p_alt - p_null) ** 2))

In [ ]:
%%writefile bsv4/src/uncertainty.py
"""Độ tin cậy, entropy, bất định và bất đồng giữa các góc nhìn. Thuần numpy.
LƯU Ý nhị phân: entropy H(p) là hàm đơn điệu của max(p) = confidence, nên hai điểm số này cho CÙNG thứ hạng (xem test)."""
import numpy as np

EPS = 1e-12


def softmax(z, T=1.0):
    z = np.asarray(z, np.float64) / float(T)
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


def calibrated_probs(logits, T):
    return softmax(logits, T)


def margin(logits, T=1.0):
    l = np.asarray(logits, np.float64)
    return (l[..., 1] - l[..., 0]) / float(T)


def entropy(p):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    return -(p * np.log(p)).sum(-1)


def confidence(p):
    return np.asarray(p).max(-1)


def js_disagreement(P):
    """Generalized Jensen-Shannon giữa S phân phối (S,N,K) -> (N,). = H(mean) - mean(H). Bằng thông tin tương hỗ (BALD) khi các mẫu là các 'mô hình'.
    Dùng cho bất đồng giữa các góc nhìn TTA (một mô hình) HOẶC giữa các mô hình (ensemble) — nguồn được ghi trong `ens_source`."""
    P = np.asarray(P, np.float64)
    return entropy(P.mean(0)) - entropy(P).mean(0)


def view_stats(z_views, thr_logit=0.0):
    """z_views (S,N): margin logit hiệu chuẩn qua S góc nhìn (hàng 0 = ảnh gốc). Trả độ ổn định dự đoán."""
    z = np.asarray(z_views, np.float64)
    flip = ((z[1:] >= thr_logit) != (z[0:1] >= thr_logit)).mean(0) if len(z) > 1 else np.zeros(z.shape[1])
    return dict(view_pred_flip=flip, view_dz_sd=z.std(0), view_dz_absmax=np.abs(z[1:] - z[0:1]).max(0) if len(z) > 1 else np.zeros(z.shape[1]))


def ensemble_features(member_logits, T=1.0, thr=0.5):
    """member_logits (M,2): logits(hoặc log-prob) của M mô hình THẬT (cùng fold, khác seed; đều chưa thấy ảnh này). Trả đặc trưng bất đồng.
    ens_js = Jensen-Shannon/BALD giữa các thành viên; ens_dz_sd = độ lệch chuẩn margin; ens_flip = tỉ lệ thành viên bất đồng nhãn với trung bình."""
    L = np.asarray(member_logits, np.float64)
    if L.ndim != 2 or L.shape[0] < 2:
        raise ValueError('ensemble cần >= 2 thành viên')
    P = softmax(L, T)                                   # (M,2)
    z = margin(L, T)
    zthr = float(np.log(thr / (1 - thr)))
    pm = P.mean(0)
    lab = (P[:, 1] >= thr)
    return dict(ens_js=float(js_disagreement(P[:, None, :])[0]), ens_dz_sd=float(z.std()), ens_flip=float((lab != (pm[1] >= thr)).mean()),
                ens_p_mal=float(pm[1]))

In [ ]:
%%writefile bsv4/src/v3/bus_core.py
# bus_core.py - module dùng chung cho huấn luyện, đánh giá và ứng dụng
import os, csv
from datetime import datetime
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ------------------------- Tiền xử lý -------------------------
def ensure_rgb_uint8(img):
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[2] == 4:
        img = img[:, :, :3]
    if img.dtype != np.uint8:
        mx = float(img.max()) if img.size else 1.0
        img = (img.astype(np.float32) / (mx if mx > 0 else 1.0) * 255.0).clip(0, 255).astype(np.uint8)
    return img


def letterbox_square(arr, pad_value=0):
    h, w = arr.shape[:2]
    s = max(h, w)
    top, left = (s - h) // 2, (s - w) // 2
    canvas = np.full((s, s) + arr.shape[2:], pad_value, dtype=arr.dtype)
    canvas[top:top + h, left:left + w] = arr
    return canvas


def preprocess_image(img_rgb, size):
    sq = letterbox_square(ensure_rgb_uint8(img_rgb))
    interp = cv2.INTER_AREA if sq.shape[0] > size else cv2.INTER_LINEAR
    return cv2.resize(sq, (size, size), interpolation=interp)


def preprocess_mask(mask_bin, size):
    sq = letterbox_square((np.asarray(mask_bin) > 0).astype(np.uint8) * 255)
    interp = cv2.INTER_AREA if sq.shape[0] > size else cv2.INTER_LINEAR
    r = cv2.resize(sq, (size, size), interpolation=interp)
    return (r > 127).astype(np.uint8)


def normalize_to_tensor(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))


# ------------------------- Mô hình -------------------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, drop=0.0):
        super().__init__()
        layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                  nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        if drop > 0:
            layers.append(nn.Dropout2d(drop))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    # g: tín hiệu gating (đã upsample, cùng kích thước với x); x: đặc trưng skip-connection
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1, bias=True), nn.BatchNorm2d(f_int))
        self.w_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1, bias=True), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
        self.last_alpha = None

    def forward(self, g, x):
        a = self.relu(self.w_g(g) + self.w_x(x))
        alpha = self.psi(a)
        self.last_alpha = alpha.detach()
        return x * alpha


class UNetFamily(nn.Module):
    # attention=False -> U-Net cơ bản; attention=True -> Attention U-Net
    # num_cls>0 -> thêm đầu phân loại trên bottleneck (mô hình đa nhiệm); forward trả (seg_logits, cls_logits)
    def __init__(self, in_ch=3, base=32, attention=True, num_cls=0, cls_drop=0.3):
        super().__init__()
        ch = [base, base * 2, base * 4, base * 8, base * 16]
        self.attention = attention
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ConvBlock(in_ch, ch[0])
        self.enc2 = ConvBlock(ch[0], ch[1])
        self.enc3 = ConvBlock(ch[1], ch[2])
        self.enc4 = ConvBlock(ch[2], ch[3])
        self.bott = ConvBlock(ch[3], ch[4], drop=0.2)
        self.ups = nn.ModuleList()
        self.atts = nn.ModuleList()
        self.decs = nn.ModuleList()
        for i in [4, 3, 2, 1]:
            self.ups.append(nn.ConvTranspose2d(ch[i], ch[i - 1], 2, stride=2))
            if attention:
                self.atts.append(AttentionGate(ch[i - 1], ch[i - 1], max(ch[i - 1] // 2, 8)))
            self.decs.append(ConvBlock(ch[i - 1] * 2, ch[i - 1]))
        self.out = nn.Conv2d(ch[0], 1, 1)
        self.cls_head = nn.Sequential(nn.Dropout(cls_drop), nn.Linear(ch[4], num_cls)) if num_cls > 0 else None

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bott(self.pool(e4))
        d = b
        for k, skip in enumerate([e4, e3, e2, e1]):
            d = self.ups[k](d)
            if self.attention:
                skip = self.atts[k](d, skip)
            d = self.decs[k](torch.cat([d, skip], dim=1))
        seg = self.out(d)
        if self.cls_head is not None:
            return seg, self.cls_head(b.mean(dim=(2, 3)))
        return seg

    def attention_maps(self):
        return [a.last_alpha for a in self.atts] if self.attention else []


def build_effnet_b3(num_classes, pretrained=True):
    loaded = False
    if pretrained:
        try:
            weights = torchvision.models.EfficientNet_B3_Weights.IMAGENET1K_V1
            m = torchvision.models.efficientnet_b3(weights=weights)
            loaded = True
        except Exception as e:
            print('[CẢNH BÁO] Không tải được trọng số pretrained (' + type(e).__name__ + ': ' + str(e)[:150] + ').')
            print('           EfficientNet-B3 sẽ được khởi tạo NGẪU NHIÊN -> không còn là transfer learning.')
            print('           Bật Internet trên Kaggle (Settings) rồi chạy lại nếu muốn dùng pretrained.')
            m = torchvision.models.efficientnet_b3(weights=None)
    else:
        m = torchvision.models.efficientnet_b3(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(in_f, num_classes))
    return m, loaded


class FusionClassifier(nn.Module):
    # E4: EfficientNet-B3 nhận 4 kênh = RGB + xác suất phân vùng do seg_net (Attention U-Net, đóng băng) dự đoán.
    # Không dùng mask ground-truth khi suy luận. Kênh thứ 4 khởi tạo trọng số 0 nên lúc bắt đầu mạng tương đương E0.
    def __init__(self, num_classes, seg_base=32, pretrained=False):
        super().__init__()
        self.seg_net = UNetFamily(3, seg_base, attention=True)
        for p in self.seg_net.parameters():
            p.requires_grad = False
        net, self.pretrained_loaded = build_effnet_b3(num_classes, pretrained)
        conv = net.features[0][0]
        new = nn.Conv2d(4, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=False)
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, :3] = conv.weight
        net.features[0][0] = new
        self.cls_net = net
        self.last_seg_prob = None

    @property
    def features(self):
        return self.cls_net.features

    def train(self, mode=True):
        super().train(mode)
        self.seg_net.eval()          # phân vùng luôn ở chế độ eval (BN, dropout cố định)
        return self

    def forward(self, x):
        with torch.no_grad():
            sp = torch.sigmoid(self.seg_net(x).float())
        self.last_seg_prob = sp.detach()
        return self.cls_net(torch.cat([x, sp.to(x.dtype)], dim=1))


def build_model(arch, num_classes, base=32, pretrained=False):
    # trả về (model, pretrained_loaded)
    if arch == 'effnet_b3':
        return build_effnet_b3(num_classes, pretrained)
    if arch == 'fusion':
        m = FusionClassifier(num_classes, seg_base=base, pretrained=pretrained)
        return m, m.pretrained_loaded
    if arch == 'unet':
        return UNetFamily(3, base, attention=False), False
    if arch == 'attunet':
        return UNetFamily(3, base, attention=True), False
    if arch == 'attunet_mt':
        return UNetFamily(3, base, attention=True, num_cls=num_classes), False
    raise ValueError('arch không hợp lệ: ' + str(arch))


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ------------------------- Checkpoint -------------------------
def torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)   # chỉ dùng cho file do chính notebook tạo
    except TypeError:
        return torch.load(path, map_location=device)


def save_checkpoint(path, model, meta):
    torch.save({'model': model.state_dict(), 'meta': meta}, path)


def load_checkpoint(path, device):
    ck = torch_load(path, device)
    meta = ck['meta']
    model, _ = build_model(meta['arch'], meta['num_classes'], base=meta.get('base', 32), pretrained=False)
    model.load_state_dict(ck['model'])
    return model.to(device).eval(), meta


# ------------------------- Grad-CAM -------------------------
def gradcam(model, x, class_idx=None, target_layer=None):
    # x: tensor (1,3,H,W). Mặc định dùng khối đặc trưng cuối của EfficientNet (model.features[-1]).
    layer = target_layer if target_layer is not None else model.features[-1]
    store = {}

    def fwd_hook(module, inp, out):
        store['a'] = out
        out.register_hook(lambda g: store.__setitem__('g', g))

    handle = layer.register_forward_hook(fwd_hook)
    try:
        model.eval()
        with torch.enable_grad():
            xin = x.detach().clone().requires_grad_(True)
            logits = model(xin)
            logits = logits[1] if isinstance(logits, tuple) else logits
            probs = torch.softmax(logits.float(), 1)[0].detach().cpu().numpy()
            idx = int(logits.argmax(1)) if class_idx is None else int(class_idx)
            model.zero_grad(set_to_none=True)
            logits[0, idx].backward()
        acts = store['a'].detach()
        grads = store['g'].detach()
        w = grads.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * acts).sum(1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)[0, 0]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), probs, idx
    finally:
        handle.remove()


# ------------------------- Detection từ mask -------------------------
def mask_to_boxes(mask, min_area=100):
    m = (np.asarray(mask) > 0).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    boxes = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area >= min_area:
            boxes.append((int(x), int(y), int(x + w), int(y + h)))   # (x1, y1, x2, y2), x2/y2 không bao gồm
    return boxes


def box_iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


# ------------------------- Suy luận ảnh mới -------------------------
def predict_array(img_rgb, cls_model, seg_model, size, device, thr=0.5, min_area=100, with_cam=True):
    img = preprocess_image(img_rgb, size)
    x = normalize_to_tensor(img).unsqueeze(0).to(device)
    res = {'image': img, 'probs': None, 'pred_idx': None, 'cam': None,
           'seg_prob': None, 'mask': None, 'boxes': []}
    if cls_model is not None:
        cls_model.eval()
        with torch.no_grad():
            out = cls_model(x)
            logits = out[1] if isinstance(out, tuple) else out
            probs = torch.softmax(logits.float(), 1)[0].cpu().numpy()
        res['probs'] = probs
        res['pred_idx'] = int(probs.argmax())
        if with_cam and hasattr(cls_model, 'features'):
            cam, _, _ = gradcam(cls_model, x, class_idx=res['pred_idx'])
            res['cam'] = cam
    if seg_model is not None:
        seg_model.eval()
        with torch.no_grad():
            out = seg_model(x)
            seg = out[0] if isinstance(out, tuple) else out
            sp = torch.sigmoid(seg.float())[0, 0].cpu().numpy()
        res['seg_prob'] = sp
        res['mask'] = (sp >= thr).astype(np.uint8)
        res['boxes'] = mask_to_boxes(res['mask'], min_area)
    return res


# ------------------------- Hiển thị -------------------------
def overlay_mask(img, mask, color=(255, 0, 0), alpha=0.4):
    out = img.copy()
    m = np.asarray(mask) > 0
    if m.any():
        out[m] = ((1 - alpha) * out[m] + alpha * np.array(color)).astype(np.uint8)
    return out


def draw_boxes(img, boxes, color=(0, 255, 0), thickness=2):
    out = np.ascontiguousarray(img.copy())
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2) - 1, int(y2) - 1), color, thickness)
    return out


def overlay_cam(img, cam, alpha=0.45):
    heat = cv2.applyColorMap((np.clip(cam, 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return ((1 - alpha) * img + alpha * heat).astype(np.uint8)


def visualize_prediction(res, class_names, title=None):
    panels = [('Ảnh đã tiền xử lý', res['image'])]
    if res['mask'] is not None:
        panels.append(('Mask dự đoán', overlay_mask(res['image'], res['mask'])))
        panels.append((f"Bounding box ({len(res['boxes'])})", draw_boxes(res['image'], res['boxes'])))
    if res['cam'] is not None:
        panels.append(('Grad-CAM', overlay_cam(res['image'], res['cam'])))
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.4))
    axes = np.atleast_1d(axes)
    for ax, (t, im) in zip(axes, panels):
        ax.imshow(im)
        ax.set_title(t, fontsize=10)
        ax.axis('off')
    if res['probs'] is not None:
        txt = ' | '.join(f'{c}: {p * 100:.1f}%' for c, p in zip(class_names, res['probs']))
        head = f"Dự đoán: {class_names[res['pred_idx']]}  ({txt})"
    else:
        head = ''
    fig.suptitle((title + '\n' if title else '') + head, fontsize=11)
    fig.tight_layout()
    return fig


def log_prediction(log_path, filename, class_names, res):
    header = ['timestamp', 'filename', 'predicted_class'] + ['prob_' + c for c in class_names] + \
             ['mask_area_ratio', 'n_boxes', 'boxes_xyxy']
    new = not os.path.exists(log_path)
    probs = res['probs'] if res['probs'] is not None else [float('nan')] * len(class_names)
    area = float(res['mask'].mean()) if res['mask'] is not None else float('nan')
    row = [datetime.now().isoformat(timespec='seconds'), filename,
           class_names[res['pred_idx']] if res['pred_idx'] is not None else ''] + \
          [f'{float(p):.4f}' for p in probs] + [f'{area:.5f}', len(res['boxes']), str(res['boxes'])]
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if new:
            w.writerow(header)
        w.writerow(row)

In [ ]:
%%writefile bsv4/src/v3/bus_core_v2.py
# bus_core_v2.py - dữ liệu, augmentation siêu âm (speckle + affine), EMA, TTA, engine huấn luyện/resume, Grad-CAM
import os, json, time, random, gc, copy
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import bus_core as core
import losses_xai as LX
from data_audit import read_rgb, read_union_mask, read_mask


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================ Nạp và tiền xử lý ============================
def load_arrays(df, img_size, mask_col='mask_path', desc=None):
    N = len(df)
    X = np.zeros((N, img_size, img_size, 3), np.uint8)
    M = np.zeros((N, img_size, img_size), np.uint8)
    for i, r in enumerate(df.itertuples(index=False)):
        img = read_rgb(r.image_path)
        assert img is not None, r.image_path
        mp = getattr(r, mask_col)
        paths = mp if isinstance(mp, (list, tuple)) else [mp]
        X[i] = core.preprocess_image(img, img_size)
        M[i] = core.preprocess_mask(read_union_mask(paths, img.shape[:2]), img_size)
    return X, M, df['label'].values.astype(np.int64)


def augment_pair(img, m, speckle=True):
    # augmentation đồng bộ ảnh–mask + mô phỏng nhiễu nhân speckle đặc trưng siêu âm (tăng chịu đựng domain shift)
    h, w = m.shape
    m0 = m
    if random.random() < 0.5:
        img, m = img[:, ::-1], m[:, ::-1]
    img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)
    if random.random() < 0.8:
        A = cv2.getRotationMatrix2D((w / 2, h / 2), random.uniform(-15, 15), random.uniform(0.85, 1.15))
        A[:, 2] += (random.uniform(-0.08, 0.08) * w, random.uniform(-0.08, 0.08) * h)
        it = cv2.warpAffine(img, A, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        mt = cv2.warpAffine(m, A, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        if not (m0.sum() > 0 and mt.sum() == 0):
            img, m = it, mt
    out = img.astype(np.float32)
    if random.random() < 0.8:
        out = out * random.uniform(0.8, 1.2) + random.uniform(-20, 20)
    if random.random() < 0.3:
        out = 255.0 * (np.clip(out, 0, 255) / 255.0) ** random.uniform(0.8, 1.25)
    if speckle and random.random() < 0.25:
        # nhiễu nhân speckle + nhiễu cộng nhẹ
        out = out * (1.0 + np.random.normal(0, random.uniform(0.02, 0.06), out.shape[:2] + (1,)))
    elif random.random() < 0.3:
        out = out + np.random.normal(0, random.uniform(3, 10), out.shape)
    return np.clip(out, 0, 255).astype(np.uint8), m


class BUSDataset(Dataset):
    def __init__(self, X, M, y, indices, augment=False):
        self.X, self.M, self.y, self.indices, self.augment = X, M, y, np.asarray(indices), augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        k = int(self.indices[i])
        img, m = self.X[k], self.M[k]
        if self.augment:
            img, m = augment_pair(img, m)
        return (core.normalize_to_tensor(img), torch.from_numpy(np.ascontiguousarray(m[None].astype(np.float32))),
                torch.tensor(int(self.y[k]), dtype=torch.long), k)


def _winit(w):
    s = (torch.initial_seed() + w) % (2 ** 32)
    random.seed(s); np.random.seed(s)


def make_loader(X, M, y, indices, batch_size, augment, shuffle, num_workers=0, seed=42):
    g = torch.Generator(); g.manual_seed(seed)
    ds = BUSDataset(X, M, y, indices, augment)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=(shuffle and len(ds) > batch_size), num_workers=num_workers,
                      worker_init_fn=_winit, generator=g, pin_memory=torch.cuda.is_available())


# ============================ Dự đoán (chuẩn & 2-view TTA) ============================
@torch.no_grad()
def predict_arrays(model, X, indices, device, batch_size=32, amp=False, tta=False):
    model.eval()
    probs, segp, evid = [], [], []
    for i in range(0, len(indices), batch_size):
        idx = indices[i:i + batch_size]
        x = torch.stack([core.normalize_to_tensor(X[k]) for k in idx]).to(device)
        with torch.autocast(device_type=device.type, enabled=amp):
            o = model(x)
            if tta:
                of = model(x.flip(-1))
        if o.get('logits') is not None:
            p = F.softmax(o['logits'].float(), 1)
            if tta:
                p = 0.5 * (p + F.softmax(of['logits'].float(), 1))
            probs.append(p.cpu().numpy())
        if o.get('seg') is not None:
            sp = torch.sigmoid(o['seg'].float())
            if tta:
                sp = 0.5 * (sp + torch.sigmoid(of['seg'].float()).flip(-1))
            segp.append(sp.squeeze(1).cpu().numpy().astype(np.float16))
        if o.get('evidence') is not None:
            ev = o['evidence'].float()
            if tta:
                ev = 0.5 * (ev + of['evidence'].float().flip(-1))
                ev = ev / (ev.sum((2, 3), keepdim=True) + 1e-8)
            evid.append(ev.cpu().numpy()[:, 0])
    cat = lambda a: np.concatenate(a) if a else None
    return dict(idx=np.asarray(indices), prob=cat(probs), seg_prob=cat(segp), evidence=cat(evid))


# ============================ EMA helper (V3) ============================
class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def state_dict(self):
        return self.shadow


# ============================ Engine huấn luyện ============================
class TrainCfg:
    def __init__(self, epochs=40, patience=10, batch_size=16, lr_enc=1e-4, lr_new=1e-3, weight_decay=1e-4, grad_clip=1.0,
                 amp=True, num_workers=2, lr_patience=4, nan_limit=5, class_weights=True, use_ema=False, ema_decay=0.995):
        self.__dict__.update(locals()); self.__dict__.pop('self')


def _param_groups(model, cfg):
    enc, new = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (enc if (n.startswith('enc.') or n.startswith('model.features') or '.features.' in n) else new).append(p)
    gs = [{'params': new, 'lr': cfg.lr_new}] if new else []
    if enc:
        gs.append({'params': enc, 'lr': cfg.lr_enc})
    return gs


def _auroc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float('nan')


def val_metrics(pred, M, y):
    out = {}
    idx = pred['idx']
    if pred['prob'] is not None:
        out['auroc'] = _auroc(y[idx], pred['prob'][:, 1]) if pred['prob'].shape[1] == 2 else float('nan')
    if pred['seg_prob'] is not None:
        pb, gb = pred['seg_prob'].astype(np.float32) >= 0.5, M[idx] > 0
        inter = (pb & gb).sum((1, 2)); den = pb.sum((1, 2)) + gb.sum((1, 2))
        has = gb.sum((1, 2)) > 0
        out['dice'] = float((2 * inter / (den + 1e-7))[has].mean()) if has.any() else float('nan')
    return out


def selection_score(m):
    v = [m[k] for k in ('auroc', 'dice') if k in m and np.isfinite(m[k])]
    return float(np.mean(v)) if v else -1.0


def make_loss_cfg(spec):
    return LX.LossCfg(**spec.get('loss', {})) if spec['kind'] == 'xai' else LX.LossCfg()


def train_model(name, model, loss_cfg, data, cfg, out_dir, seed, device, augment=True, resume=True, meta=None, log=print):
    os.makedirs(out_dir, exist_ok=True)
    tag = f'{name}_s{seed}'
    best_p, last_p, hist_p = (os.path.join(out_dir, f'{tag}_{k}') for k in ('best.pt', 'last.pt', 'history.json'))
    seed_everything(seed)
    X, M, y = data['X'], data['M'], data['y']
    use_cons = loss_cfg.lam_cons_pred > 0 or loss_cfg.lam_cons_ev > 0
    use_faith = getattr(loss_cfg, 'lam_faith', 0.0) > 0
    tl = make_loader(X, M, y, data['train'], cfg.batch_size, augment, True, cfg.num_workers, seed)
    model = model.to(device)
    ema = ModelEMA(model, cfg.ema_decay) if getattr(cfg, 'use_ema', False) else None
    cw = None
    if cfg.class_weights:
        cnt = np.bincount(y[data['train']], minlength=int(y.max()) + 1).astype(float)
        cw = torch.tensor(len(data['train']) / (len(cnt) * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(_param_groups(model, cfg), weight_decay=cfg.weight_decay)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=cfg.lr_patience)
    use_amp = cfg.amp and device.type == 'cuda'
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ep0, best, best_ep, bad, hist, done = 0, -1e9, -1, 0, [], False
    if resume and os.path.exists(last_p):
        ck = core.torch_load(last_p, device)
        if not ck.get('compact'):
            model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); scaler.load_state_dict(ck['scaler'])
            if ema is not None and 'ema' in ck:
                ema.shadow = ck['ema']
        ep0, best, best_ep, bad, hist, done = ck['epoch'] + 1, ck['best'], ck['best_ep'], ck['bad'], ck['hist'], ck['done']
        log(f'[{tag}] resume từ epoch {ep0}' + (' (đã hoàn tất)' if done else ''))
    for ep in range(ep0, cfg.epochs):
        if done:
            break
        model.train()
        tot, n, skipped, parts_sum = 0.0, 0, 0, {}
        t0 = time.time()
        for xb, mb, yb, _ in tl:
            xb, mb, yb = xb.to(device, non_blocking=True), mb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            out_aug = theta = logits_hi = logits_lo = None
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(xb)
                if use_cons:
                    theta = LX.sample_affine(xb.shape[0], device)
                    out_aug = model(LX.apply_label_preserving(xb, theta))
                if use_faith and out.get('evidence') is not None and hasattr(model, 'forward_heads_from_Z'):
                    E = out['evidence']
                    q = getattr(loss_cfg, 'faith_q', 0.2)
                    thr_hi = torch.quantile(E.flatten(1), 1.0 - q, dim=1, keepdim=True).view(-1, 1, 1, 1)
                    thr_lo = torch.quantile(E.flatten(1), q, dim=1, keepdim=True).view(-1, 1, 1, 1)
                    keep_hi = (E < thr_hi).float()
                    keep_lo = (E > thr_lo).float()
                    logits_hi, _, _ = model.forward_heads_from_Z(out['Z'].detach(), out['mask_feat'].detach(), out.get('ring'), out.get('post'), E, ev_mask=keep_hi)
                    logits_lo, _, _ = model.forward_heads_from_Z(out['Z'].detach(), out['mask_feat'].detach(), out.get('ring'), out.get('post'), E, ev_mask=keep_lo)
            loss, parts = LX.total_loss(out, mb, yb, loss_cfg, cw, out_aug, theta, logits_hi, logits_lo)
            if not torch.isfinite(loss):
                skipped += 1
                if skipped > cfg.nan_limit:
                    raise FloatingPointError('Quá nhiều batch NaN/Inf')
                continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update()
            if ema is not None:
                ema.update(model)
            tot += float(loss.detach()) * xb.size(0); n += xb.size(0)
            for k, v in parts.items():
                parts_sum[k] = parts_sum.get(k, 0.0) + v * xb.size(0)
        if n == 0:
            raise RuntimeError('Epoch không có batch hợp lệ')
        if ema is not None:
            cur_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            model.load_state_dict(ema.state_dict())
            vp = predict_arrays(model, X, data['val'], device, amp=use_amp)
            eval_state = {k: v.detach().clone() for k, v in ema.state_dict().items()}
            model.load_state_dict(cur_state)
        else:
            vp = predict_arrays(model, X, data['val'], device, amp=use_amp)
            eval_state = model.state_dict()
        vm = val_metrics(vp, M, y)
        score = selection_score(vm)
        sch.step(score)
        imp = score > best + 1e-4
        if imp:
            best, best_ep, bad = score, ep, 0
            torch.save(dict(model=eval_state, meta=dict(meta or {}, name=name, seed=seed, best_score=best, best_epoch=ep + 1)), best_p)
        else:
            bad += 1
        done = bad >= cfg.patience or ep == cfg.epochs - 1
        rec = dict(epoch=ep + 1, train_loss=tot / n, skipped=skipped, score=float(score), time_s=time.time() - t0,
                   **{f'loss_{k}': v / n for k, v in parts_sum.items()}, **{f'val_{k}': float(v) for k, v in vm.items()})
        hist.append(rec)
        if done:
            torch.save(dict(compact=True, epoch=ep, best=best, best_ep=best_ep, bad=bad, hist=hist, done=True), last_p)
        else:
            ck_last = dict(model=model.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(), scaler=scaler.state_dict(), epoch=ep, best=best,
                           best_ep=best_ep, bad=bad, hist=hist, done=False)
            if ema is not None:
                ck_last['ema'] = ema.state_dict()
            torch.save(ck_last, last_p)
        json.dump(hist, open(hist_p, 'w'))
        log(f'[{tag}] ep {ep + 1:3d}/{cfg.epochs} loss {rec["train_loss"]:.4f} ' + ' '.join(f'{k}={v:.3f}' for k, v in vm.items()) + f' score {score:.4f}' + (' *best*' if imp else ''))
        if bad >= cfg.patience:
            log(f'[{tag}] early stopping'); break
    del opt, tl
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return dict(name=name, seed=seed, best_path=best_p, last_path=last_p, history_path=hist_p, best_score=best, best_epoch=best_ep + 1)


def train_with_oom_fallback(*args, cfg=None, **kw):
    bs = cfg.batch_size
    while True:
        oom = False
        try:
            cfg.batch_size = bs
            return train_model(*args, cfg=cfg, **kw)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and bs > 2:
                oom = True
            else:
                raise
        if oom:
            gc.collect(); torch.cuda.empty_cache(); bs //= 2
            print('!! hết bộ nhớ GPU -> batch', bs)


# ============================ Checkpoint ============================
def save_model_checkpoint(path, model, meta):
    torch.save(dict(model=model.state_dict(), meta=meta), path)


def load_model_from_checkpoint(path, device, pretrained=False):
    from models_xai import build_experiment
    ck = core.torch_load(path, device)
    meta = ck['meta']
    model, _ = build_experiment(meta['experiment'], num_classes=meta.get('num_classes', 2), pretrained=pretrained,
                                dim=meta.get('dim', 128), base=meta.get('base', 32))
    model.load_state_dict(ck['model'])
    return model.to(device).eval(), meta


# ============================ Grad-CAM cho mô hình trả dict ============================
def gradcam_dict(model, x, class_idx=None, layer=None):
    layer = layer if layer is not None else (model.neck if hasattr(model, 'neck') else model.model.features[-1])
    store = {}

    def hook(m, i, o):
        store['a'] = o
        o.register_hook(lambda g: store.__setitem__('g', g))
    h = layer.register_forward_hook(hook)
    try:
        model.eval()
        with torch.enable_grad():
            xin = x.detach().clone().requires_grad_(True)
            lg = model(xin)['logits']
            idx = int(lg.argmax(1)) if class_idx is None else int(class_idx)
            model.zero_grad(set_to_none=True)
            lg[0, idx].backward()
        w = store['g'].detach().mean((2, 3), keepdim=True)
        cam = F.relu((w * store['a'].detach()).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), idx
    finally:
        h.remove()


# ============================ Phân tích một ảnh (dùng cho ứng dụng) ============================
def analyze_image_v2(rgb, model, dcfg, device):
    import bus_trust as T
    size = int(dcfg['img_size'])
    img = core.preprocess_image(rgb, size)
    x = core.normalize_to_tensor(img).unsqueeze(0).to(device)
    model.eval()
    use_tta = bool(dcfg.get('tta', False))
    with torch.no_grad():
        o = model(x)
        if use_tta:
            of = model(x.flip(-1))
            lg = 0.5 * (F.softmax(o['logits'].float(), 1) + F.softmax(of['logits'].float(), 1))
            raw = lg[0].cpu().numpy()
            sp_t = 0.5 * (torch.sigmoid(o['seg'].float()) + torch.sigmoid(of['seg'].float()).flip(-1))
        else:
            raw = F.softmax(o['logits'].float(), 1)[0].cpu().numpy()
            sp_t = torch.sigmoid(o['seg'].float())
    cal = T.softmax(np.log(np.clip(raw, 1e-12, 1)), float(dcfg['temperature']))
    p_mal = float(cal[1])
    conf = float(cal.max())
    res = dict(image=img, p_raw=raw, p_cal=cal, p_malignant=p_mal, label='malignant' if p_mal >= float(dcfg['threshold']) else 'benign',
               confidence=conf, abstain=bool(conf < float(dcfg['abstain_threshold'])), evidence=None, cam=None)
    sp = sp_t[0, 0].cpu().numpy()
    res['seg_prob'] = sp
    res['mask'] = (sp >= float(dcfg['seg_threshold'])).astype(np.uint8)
    res['boxes'] = core.mask_to_boxes(res['mask'], int(dcfg['min_area']))
    if o.get('evidence') is not None:
        E = o['evidence'].float()
        if use_tta:
            E = 0.5 * (E + of['evidence'].float().flip(-1))
        res['evidence'] = F.interpolate(E / (E.sum() + 1e-12), size=(size, size), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()
    res['cam'], _ = gradcam_dict(model, x, class_idx=int(np.argmax(cal)))
    return res

In [ ]:
%%writefile bsv4/src/v3/bus_trust.py
# bus_trust.py - BUS-Trust: hiệu chuẩn, độ bất định, chọn lọc dự đoán, nhất quán đa mô hình, báo cáo PDF
import numpy as np
from datetime import datetime
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

EPS = 1e-12


# ------------------------- Xác suất, hiệu chuẩn -------------------------
def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


def entropy(p):
    p = np.clip(p, EPS, 1.0)
    return -(p * np.log(p)).sum(-1)


def nll(p, y):
    return float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1.0)).mean())


def brier(p, y):
    return float(((p - np.eye(p.shape[1])[y]) ** 2).sum(1).mean())


def ece(p, y, n_bins=10):
    conf, pred = p.max(1), p.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(out)


def fit_temperature(logits, y):
    # tìm T > 0 cực tiểu NLL trên tập validation (tìm lưới thô rồi tinh)
    grid = np.exp(np.linspace(np.log(0.2), np.log(10.0), 200))
    losses = [nll(softmax(logits, t), y) for t in grid]
    i = int(np.argmin(losses))
    fine = np.linspace(grid[max(i - 1, 0)], grid[min(i + 1, len(grid) - 1)], 100)
    return float(fine[int(np.argmin([nll(softmax(logits, t), y) for t in fine]))])


def mutual_information(P):
    # P: (S, N, K) các mẫu xác suất (MC dropout) -> thông tin tương hỗ (độ bất định tri thức)
    return entropy(P.mean(0)) - entropy(P).mean(0)


# ------------------------- Chọn lọc dự đoán -------------------------
def risk_coverage(unc, wrong):
    order = np.argsort(unc, kind='stable')
    w = np.asarray(wrong, float)[order]
    n = len(w)
    cov = np.arange(1, n + 1) / n
    risk = np.cumsum(w) / np.arange(1, n + 1)
    return cov, risk, float(risk.mean())        # (coverage, risk, AURC)


def safe_auroc(y, s):
    y, s = np.asarray(y), np.asarray(s, float)
    if len(np.unique(y)) < 2 or not np.isfinite(s).all():
        return float('nan')
    return float(roc_auc_score(y, s))


def accept_by_val_quantile(val_unc, test_unc, coverage):
    thr = float(np.quantile(val_unc, coverage))   # ngưỡng chọn CHỈ trên validation
    return thr, np.asarray(test_unc) <= thr


# ------------------------- Nhất quán phân loại - phân vùng - Grad-CAM -------------------------
def seg_evidence(seg_prob, thr=0.5):
    sp = np.asarray(seg_prob, np.float32)
    if sp.ndim == 2:
        sp = sp[None]
    area = (sp >= thr).mean((1, 2))
    peak = sp.reshape(len(sp), -1).max(1)
    return np.stack([peak, np.log10(area + 1e-4)], 1)


def cam_alignment(cam, mask):
    # tỉ lệ năng lượng Grad-CAM nằm trong mask dự đoán; NaN nếu mask rỗng
    cam = np.asarray(cam, np.float32)
    m = np.asarray(mask) > 0
    if cam.ndim == 2:
        cam, m = cam[None], m[None]
    tot = cam.reshape(len(cam), -1).sum(1) + 1e-8
    inside = (cam * m).reshape(len(cam), -1).sum(1) / tot
    area = m.reshape(len(m), -1).mean(1)
    return np.where(area > 0, inside, np.nan), area


class Standardizer:
    def fit(self, F):
        F = np.asarray(F, float)
        self.mean = np.nanmean(F, 0)
        self.mean = np.where(np.isfinite(self.mean), self.mean, 0.0)
        Fi = np.where(np.isfinite(F), F, self.mean)
        self.std = Fi.std(0) + 1e-8
        return self

    def transform(self, F):
        F = np.asarray(F, float)
        return (np.where(np.isfinite(F), F, self.mean) - self.mean) / self.std


class TumorPresenceMapper:
    # ánh xạ bằng chứng phân vùng -> P(ảnh có khối u); học trên validation để cùng thang xác suất với P(tumor) của bộ phân loại
    def fit(self, ev, has_tumor):
        self.sc = Standardizer().fit(ev)
        self.lr = LogisticRegression(C=1.0).fit(self.sc.transform(ev), has_tumor.astype(int))
        return self

    def predict(self, ev):
        return self.lr.predict_proba(self.sc.transform(ev))[:, 1]

    def to_dict(self):
        return dict(mean=self.sc.mean.tolist(), std=self.sc.std.tolist(), coef=self.lr.coef_[0].tolist(), intercept=float(self.lr.intercept_[0]))

    @staticmethod
    def from_dict(d):
        m = TumorPresenceMapper()
        m.sc = Standardizer(); m.sc.mean, m.sc.std = np.array(d['mean']), np.array(d['std'])
        m.coef, m.intercept = np.array(d['coef']), d['intercept']
        m.predict = lambda ev: 1 / (1 + np.exp(-(((np.asarray(ev) - m.sc.mean) / m.sc.std) @ m.coef + m.intercept)))
        return m


class ErrorPredictor:
    # hồi quy logistic chuẩn hóa, ít đặc trưng, regularization mạnh; học trên validation để dự đoán 'dự đoán của E0 sai'
    def __init__(self, C=0.3):
        self.C = C

    def fit(self, F, wrong):
        self.sc = Standardizer().fit(F)
        self.lr = LogisticRegression(C=self.C, class_weight='balanced', max_iter=1000).fit(self.sc.transform(F), np.asarray(wrong).astype(int))
        return self

    def score(self, F):
        return self.lr.decision_function(self.sc.transform(F))

    def to_dict(self):
        return dict(mean=self.sc.mean.tolist(), std=self.sc.std.tolist(), coef=self.lr.coef_[0].tolist(), intercept=float(self.lr.intercept_[0]), C=self.C)

    @staticmethod
    def from_dict(d):
        p = ErrorPredictor(d.get('C', 0.3))
        mean, std, coef, b = np.array(d['mean']), np.array(d['std']), np.array(d['coef']), d['intercept']
        p.score = lambda F: ((np.where(np.isfinite(np.asarray(F, float)), np.asarray(F, float), mean) - mean) / std) @ coef + b
        return p


# ------------------------- Phân tích một ảnh (dùng trong ứng dụng) -------------------------
def analyze_trust(res, cfg):
    # res: kết quả core.predict_array (cần probs, mask, seg_prob, cam); cfg: trust_config.json đã nạp
    names = cfg['class_names']
    probs = np.clip(np.asarray(res['probs'], float), EPS, 1.0)
    p_cal = softmax(np.log(probs), cfg['temperature'])
    if p_cal.ndim == 2:
        p_cal = p_cal[0]
    out = dict(p_cal=p_cal.tolist(), pred_idx=int(np.argmax(p_cal)), entropy=float(entropy(p_cal)), reasons=[])
    ni = cfg.get('normal_idx')
    ev = seg_evidence(res['seg_prob'], cfg['seg_threshold'])
    if ni is not None and cfg.get('mapper'):
        mapper = TumorPresenceMapper.from_dict(cfg['mapper'])
        p_seg = float(mapper.predict(ev)[0])
        p_cls = float(1.0 - p_cal[ni])
        out.update(p_tumor_cls=p_cls, p_tumor_seg=p_seg, dcs=abs(p_cls - p_seg))
    else:
        out.update(p_tumor_cls=float('nan'), p_tumor_seg=float('nan'), dcs=0.0)
    inside = float('nan')
    if res.get('cam') is not None:
        inside = float(cam_alignment(res['cam'], res['mask'])[0][0])
    out['cam_inside'] = inside
    out['mask_area'] = float((np.asarray(res['mask']) > 0).mean())
    F = np.array([[out['entropy'], out['dcs'], inside]])
    pred = ErrorPredictor.from_dict(cfg['predictor'])
    out['risk_score'] = float(pred.score(F)[0])
    th = cfg['thresholds']
    if out['entropy'] > th['entropy']:
        out['reasons'].append('Độ bất định của dự đoán cao (entropy vượt ngưỡng chọn trên tập validation).')
    if out['dcs'] > th['dcs']:
        out['reasons'].append('Kết quả phân loại và phân vùng chưa nhất quán về việc ảnh có khối u hay không.')
    if np.isfinite(inside) and inside < th['cam_inside']:
        out['reasons'].append('Vùng Grad-CAM ít trùng với vùng được phân vùng.')
    out['flag'] = bool(out['risk_score'] >= th['risk_score'])
    out['pred_name'] = names[out['pred_idx']]
    return out


# ------------------------- Báo cáo PDF (không cần thư viện ngoài) -------------------------
def make_pdf_report(path, filename, res, class_names, trust, meta_text, fig=None):
    import matplotlib
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    with PdfPages(path) as pdf:
        f = plt.figure(figsize=(8.27, 11.69))
        f.text(0.07, 0.95, 'BÁO CÁO PHÂN TÍCH ẢNH SIÊU ÂM VÚ (hỗ trợ nghiên cứu)', fontsize=14, weight='bold')
        lines = [f'Tệp ảnh: {filename}', f'Thời gian: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', meta_text, '',
                 'Xác suất (đã hiệu chuẩn bằng temperature scaling):']
        for c, p in zip(class_names, trust['p_cal']):
            lines.append(f'   - {c}: {p * 100:.1f}%')
        lines += ['', f'Lớp dự đoán: {trust["pred_name"]}', f'Entropy dự đoán: {trust["entropy"]:.3f}',
                  f'Tỉ lệ diện tích vùng phân vùng: {trust["mask_area"] * 100:.2f}%',
                  f'Chênh lệch P(có khối u) giữa phân loại và phân vùng: {trust["dcs"]:.3f}',
                  f'Độ trùng Grad-CAM với vùng phân vùng: ' + ('không xác định (mask rỗng)' if not np.isfinite(trust['cam_inside']) else f'{trust["cam_inside"]:.3f}'),
                  '', 'TRẠNG THÁI: ' + ('CẦN CHUYÊN GIA XEM XÉT' if trust['flag'] else 'Không có cảnh báo từ hệ thống kiểm tra độ tin cậy')]
        for r in trust['reasons']:
            lines.append('   * ' + r)
        lines += ['', 'GIỚI HẠN SỬ DỤNG:',
                  '   - Đây là công cụ hỗ trợ nghiên cứu, KHÔNG phải kết luận chẩn đoán.',
                  '   - Mô hình huấn luyện trên một bộ dữ liệu công khai nhỏ, chưa kiểm định ngoài.',
                  '   - Không có cảnh báo không có nghĩa là dự đoán đúng; mọi kết quả cần được bác sĩ chuyên khoa đánh giá.']
        y = 0.91
        for ln in lines:
            f.text(0.07, y, ln, fontsize=9.5)
            y -= 0.022
        pdf.savefig(f); plt.close(f)
        if fig is not None:
            pdf.savefig(fig)

In [ ]:
%%writefile bsv4/src/v3/data_audit.py
# data_audit.py - khám phá dataset BUS-BRA / BUSI, kiểm toán dữ liệu, chia theo bệnh nhân, kiểm tra rò rỉ, manifest
import os, re, json, hashlib
import numpy as np
import pandas as pd
import cv2

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
MASK_RE = re.compile(r'^(.*)_mask(?:_\d+)?$', re.IGNORECASE)


# ============================ Tiện ích ============================
def read_rgb(path):
    b = cv2.imread(path, cv2.IMREAD_COLOR)
    return None if b is None else cv2.cvtColor(b, cv2.COLOR_BGR2RGB)


def read_mask(path, hw=None):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    if hw is not None and m.shape != tuple(hw):
        m = cv2.resize(m, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    mx = int(m.max())
    return (m > mx // 2).astype(np.uint8) if mx > 0 else np.zeros_like(m)     # chịu được mask 0/1, 0/255, hay nhãn 0/k


def read_union_mask(paths, hw):
    u = np.zeros(hw, np.uint8)
    for p in paths:
        m = read_mask(p, hw)
        if m is not None:
            u |= m
    return u


def dhash256(img_rgb):
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    s = cv2.resize(g, (17, 16), interpolation=cv2.INTER_AREA)
    return (s[:, 1:] > s[:, :-1]).flatten().astype(np.uint8)


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


class _UF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def _key(name):
    s = os.path.splitext(os.path.basename(str(name)))[0].lower()
    s = re.sub(r'^(bus|mask|img|image)[_\-]?', '', s)
    s = re.sub(r'[_\-]?mask$', '', s)
    return s


# ============================ Khám phá BUS-BRA ============================
COLS = dict(image=['ID', 'id', 'Image', 'image', 'Filename', 'filename', 'File', 'file', 'Image_ID'],
            patient=['Case', 'case', 'Patient', 'patient', 'PatientID', 'Patient_ID', 'patient_id', 'Case_ID'],
            label=['Pathology', 'pathology', 'Label', 'label', 'Class', 'class'],
            birads=['BIRADS', 'BI-RADS', 'birads', 'BI_RADS'],
            device=['Device', 'device', 'Scanner', 'scanner'],
            fold=['Fold', 'fold', 'Split', 'split', 'Partition'])


def _pick(df, cands, override=None):
    if override:
        if override not in df.columns:
            raise KeyError(f'Cột {override!r} không có trong CSV. Các cột hiện có: {list(df.columns)}')
        return override
    for c in cands:
        if c in df.columns:
            return c
    return None


def find_busbra_candidates(base):
    out = []
    for dp, dns, fs in os.walk(base):
        low = {d.lower(): d for d in dns}
        if 'images' in low and 'masks' in low:
            out.append(dp)
            dns[:] = []
    return out


def find_csvs(root):
    return [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if f.lower().endswith('.csv')]


def index_busbra(root, csv_path=None, col_overrides=None, allow_missing_patient=False):
    # Dựng bảng (ảnh, mask, bệnh nhân, nhãn). Cấu trúc CSV của BUS-BRA chưa được xác minh trực tiếp bởi tác giả code:
    # hàm dò cột theo danh sách tên phổ biến; nếu không khớp sẽ DỪNG kèm danh sách cột để bạn đặt col_overrides.
    col_overrides = col_overrides or {}
    sub = {d.lower(): d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))}
    if 'images' not in sub or 'masks' not in sub:
        raise FileNotFoundError(f'{root} phải có thư mục Images và Masks. Hiện có: {sorted(os.listdir(root))}')
    img_dir, msk_dir = os.path.join(root, sub['images']), os.path.join(root, sub['masks'])
    parent = os.path.dirname(os.path.abspath(root))
    pool = find_csvs(root) + [os.path.join(parent, f) for f in os.listdir(parent) if f.lower().endswith('.csv')]       # CSV có thể nằm ở thư mục cha
    csvs = [csv_path] if csv_path else [c for c in pool if 'fold' not in os.path.basename(c).lower() and 'split' not in os.path.basename(c).lower()]
    if not csvs:
        raise FileNotFoundError('Không tìm thấy file CSV thông tin trong ' + root)
    csv = None
    for c in csvs:
        d = pd.read_csv(c)
        if _pick(d, COLS['label'], col_overrides.get('label')):
            csv, meta = c, d
            break
    if csv is None:
        raise KeyError('Không CSV nào có cột nhãn (Pathology). Hãy đặt col_overrides={"label": ...}. CSV tìm thấy: ' + str(csvs))
    c_img, c_pat, c_lab = (_pick(meta, COLS[k], col_overrides.get(k)) for k in ('image', 'patient', 'label'))
    if c_img is None:
        raise KeyError(f'Không xác định được cột tên ảnh. Cột hiện có: {list(meta.columns)}; đặt col_overrides={{"image": ...}}')
    if c_pat is None and not allow_missing_patient:
        raise KeyError(f'Không có cột mã bệnh nhân (ví dụ Case). Cột hiện có: {list(meta.columns)}. Đặt col_overrides={{"patient": ...}} '
                       'hoặc allow_missing_patient=True (KHÔNG khuyến nghị: mất khả năng chia theo bệnh nhân).')
    imgs = {_key(f): os.path.join(img_dir, f) for f in os.listdir(img_dir) if os.path.splitext(f)[1].lower() in IMG_EXT}
    msks = {_key(f): os.path.join(msk_dir, f) for f in os.listdir(msk_dir) if os.path.splitext(f)[1].lower() in IMG_EXT}
    rows, missing = [], []
    for _, r in meta.iterrows():
        k = _key(r[c_img])
        if k not in imgs or k not in msks:
            missing.append(str(r[c_img])); continue
        lab = str(r[c_lab]).strip().lower()
        label = 1 if lab.startswith('m') else (0 if lab.startswith('b') else None)
        if label is None:
            missing.append(f'{r[c_img]} (nhãn lạ: {lab})'); continue
        rows.append(dict(image_path=imgs[k], mask_path=msks[k], patient_id=str(r[c_pat]) if c_pat else f'img_{k}', label=label, pathology=lab,
                         birads=r[_pick(meta, COLS['birads'])] if _pick(meta, COLS['birads']) else None,
                         device=r[_pick(meta, COLS['device'])] if _pick(meta, COLS['device']) else None,
                         official_fold=r[_pick(meta, COLS['fold'])] if _pick(meta, COLS['fold']) else None, source='BUS-BRA'))
    df = pd.DataFrame(rows)
    info = dict(csv=csv, columns=dict(image=c_img, patient=c_pat, label=c_lab), n_csv_rows=len(meta), n_indexed=len(df), n_unpaired=len(missing), unpaired_examples=missing[:10])
    if len(df) == 0:
        raise RuntimeError(f'Không ghép được ảnh–mask nào. Ví dụ tên trong CSV: {meta[c_img].head(3).tolist()}; ví dụ tên ảnh: {list(imgs)[:3]}; mask: {list(msks)[:3]}')
    return df, info


# ============================ Khám phá BUSI (chỉ benign/malignant) ============================
def find_busi_candidates(base):
    out = []
    for dp, dns, _ in os.walk(base):
        low = {d.lower() for d in dns}
        if 'benign' in low and 'malignant' in low:
            out.append(dp); dns[:] = []
    return out


def index_busi_binary(root):
    # KHÔNG chuyển normal thành benign: ảnh normal bị loại khỏi tập đánh giá ngoài.
    rows, n_normal = [], 0
    for d in sorted(os.listdir(root)):
        c = d.lower()
        full = os.path.join(root, d)
        if not os.path.isdir(full) or c not in ('benign', 'malignant', 'normal'):
            continue
        imgs, masks = {}, {}
        for f in sorted(os.listdir(full)):
            stem, ext = os.path.splitext(f)
            if ext.lower() not in IMG_EXT:
                continue
            m = MASK_RE.match(stem)
            (masks.setdefault(m.group(1), []).append(os.path.join(full, f)) if m else imgs.__setitem__(stem, os.path.join(full, f)))
        if c == 'normal':
            n_normal += len(imgs); continue
        for stem, p in imgs.items():
            rows.append(dict(image_path=p, mask_paths=sorted(masks.get(stem, [])), patient_id=f'busi_{c}_{stem}', label=int(c == 'malignant'),
                             pathology=c, source='BUSI'))
    return pd.DataFrame(rows), dict(n_normal_excluded=n_normal)


# ============================ Kiểm toán ============================
def audit_dataframe(df, mask_col='mask_path', dup_hamming=8, read_pixels=True):
    # Trả (df_đã_gắn_cột, báo_cáo). Phát hiện: ảnh/mask lỗi, mask rỗng, lệch kích thước, trùng tuyệt đối, gần trùng, nhãn mâu thuẫn.
    df = df.reset_index(drop=True).copy()
    n = len(df)
    if n == 0:
        raise ValueError('audit_dataframe nhận bảng RỖNG: bước khám phá dataset không tìm thấy ảnh nào. Kiểm tra đường dẫn và tên cột.')
    md5, dh, ok, hh, ww, area, msz, issue = [], [], [], [], [], [], [], []
    for r in df.itertuples(index=False):
        iss = []
        img = read_rgb(r.image_path)
        if img is None:
            md5.append(''); dh.append(np.zeros(256, np.uint8)); hh.append(0); ww.append(0); area.append(0); msz.append(False); issue.append('anh_loi'); continue
        mp = getattr(r, mask_col)
        paths = mp if isinstance(mp, (list, tuple)) else [mp]
        m = np.zeros(img.shape[:2], np.uint8)
        mism = False
        for p in paths:
            raw = cv2.imread(p, cv2.IMREAD_GRAYSCALE) if p else None
            if raw is None:
                iss.append('mask_loi_hoac_thieu'); continue
            if raw.shape != img.shape[:2]:
                mism = True
            m |= read_mask(p, img.shape[:2])
        if m.sum() == 0:
            iss.append('mask_rong')
        md5.append(sha256_file(r.image_path)); dh.append(dhash256(img)); hh.append(img.shape[0]); ww.append(img.shape[1])
        area.append(float(m.mean())); msz.append(mism); issue.append(';'.join(iss))
    df['sha256'], df['h'], df['w'], df['mask_ratio'], df['mask_size_mismatch'], df['issue'] = md5, hh, ww, area, msz, issue
    bits = np.stack(dh).astype(np.int32)
    ham = bits @ (1 - bits).T + (1 - bits) @ bits.T
    uf = _UF(n)
    ii, jj = np.where(np.triu(ham <= dup_hamming, k=1))
    for a, b in zip(ii, jj):
        if df.loc[a, 'sha256'] != '' and df.loc[b, 'sha256'] != '':
            uf.union(int(a), int(b))
    df['dup_group'] = [uf.find(i) for i in range(n)]
    gs = df.groupby('dup_group').size()
    exact = df.groupby('sha256').filter(lambda g: len(g) > 1 and g.name != '')
    conflict_groups = df.groupby('dup_group')['label'].nunique()
    rep = dict(n=n, n_bad_image=int((df['issue'].str.contains('anh_loi')).sum()), n_bad_or_missing_mask=int(df['issue'].str.contains('mask_loi').sum()),
               n_empty_mask=int(df['issue'].str.contains('mask_rong').sum()), n_mask_size_mismatch=int(df['mask_size_mismatch'].sum()),
               n_exact_duplicate_images=int(len(exact)), n_neardup_groups=int((gs > 1).sum()), n_images_in_neardup_groups=int(gs[gs > 1].sum()),
               n_label_conflict_groups=int((conflict_groups > 1).sum()), class_counts=df['label'].value_counts().sort_index().to_dict(),
               n_patients=int(df['patient_id'].nunique()) if 'patient_id' in df else None)
    df['_dhash'] = [d.tobytes() for d in dh]
    return df, rep


def drop_invalid(df):
    # loại ảnh/mask lỗi, mask rỗng, nhãn mâu thuẫn trong cùng nhóm trùng; giữ 1 bản cho trùng tuyệt đối cùng nhãn
    bad = df['issue'] != ''
    conflict = df.groupby('dup_group')['label'].transform('nunique') > 1
    keep = ~bad & ~conflict
    out = df[keep].copy()
    if len(out) == 0:
        raise RuntimeError('Sau khi lọc không còn ảnh nào hợp lệ. Số lượng theo lý do: ' + str(df['issue'].replace('', 'ok').value_counts().to_dict())
                           + ' | nhãn mâu thuẫn trong nhóm trùng: ' + str(int(conflict.sum())) + '. Xem hướng dẫn xử lý trong README.')
    out = out.drop_duplicates(subset='sha256', keep='first')
    return out.reset_index(drop=True), dict(dropped_invalid=int(bad.sum()), dropped_label_conflict=int((conflict & ~bad).sum()), dropped_exact_dups=int(keep.sum() - len(out)))


# ============================ Gom nhóm và chia theo bệnh nhân ============================
def build_groups(df):
    # nhóm = hợp của (cùng bệnh nhân) và (ảnh trùng/gần trùng)
    n = len(df)
    uf = _UF(n)
    for col in ('patient_id', 'dup_group'):
        first = {}
        for i, v in enumerate(df[col].values):
            if v in first:
                uf.union(first[v], i)
            else:
                first[v] = i
    return np.array([uf.find(i) for i in range(n)])


def grouped_split(df, seed=42, val_frac=0.15, test_frac=0.15, official_test_fold=None):
    # Chia theo nhóm (bệnh nhân + ảnh trùng), phân tầng theo nhãn. Nếu official_test_fold được cho và cột official_fold có giá trị,
    # test = fold đó nhưng validation vẫn tái tạo từ bệnh nhân KHÔNG thuộc test (tránh lệch bệnh nhân giữa train/val được cung cấp).
    from sklearn.model_selection import StratifiedGroupKFold
    df = df.reset_index(drop=True).copy()
    df['group'] = build_groups(df)
    idx = np.arange(len(df))
    y, g = df['label'].values, df['group'].values
    if official_test_fold is not None and df['official_fold'].notna().any():
        test = np.where(df['official_fold'].astype(str).values == str(official_test_fold))[0]
        tg = set(g[test])
        test = np.array([i for i in idx if g[i] in tg])          # kéo cả nhóm vào test (bệnh nhân không tách đôi)
        dev = np.array([i for i in idx if g[i] not in tg])
    else:
        k1 = max(2, int(round(1 / test_frac)))
        dev, test = next(StratifiedGroupKFold(n_splits=k1, shuffle=True, random_state=seed).split(idx, y, g))
    k2 = max(2, int(round(1 / (val_frac / (1 - test_frac)))))
    tr_rel, va_rel = next(StratifiedGroupKFold(n_splits=k2, shuffle=True, random_state=seed).split(dev, y[dev], g[dev]))
    df['split'] = ''
    df.loc[dev[tr_rel], 'split'] = 'train'; df.loc[dev[va_rel], 'split'] = 'val'; df.loc[test, 'split'] = 'test'
    return df


def leakage_report(df, dup_hamming=8):
    # kiểm tra: nhóm, bệnh nhân, sha256, và cặp gần trùng (dHash) nằm ở các tập khác nhau
    rep = {}
    for col in ('group', 'patient_id', 'sha256'):
        if col in df:
            s = df.groupby(col)['split'].nunique()
            rep[f'{col}_across_splits'] = int((s > 1).sum())
    if '_dhash' in df:
        bits = np.stack([np.frombuffer(b, np.uint8) for b in df['_dhash']]).astype(np.int32)
        ham = bits @ (1 - bits).T + (1 - bits) @ bits.T
        sp = df['split'].values
        i, j = np.where(np.triu(ham <= dup_hamming, k=1))
        rep['neardup_pairs_across_splits'] = int(sum(sp[a] != sp[b] for a, b in zip(i, j)))
    rep['ok'] = all(v == 0 for k, v in rep.items() if k != 'ok')
    return rep


def cross_dataset_duplicates(dfA, dfB, dup_hamming=8):
    # kiểm tra ảnh trùng/gần trùng giữa hai dataset (ví dụ BUS-BRA và BUSI)
    if len(dfA) == 0 or len(dfB) == 0:
        raise ValueError(f'So trùng giữa hai dataset cần cả hai bảng khác rỗng (A={len(dfA)}, B={len(dfB)}). Bảng rỗng nghĩa là dataset đó bị loại hết ở bước audit.')
    out = dict(sha256_shared=int(len(set(dfA['sha256']) & set(dfB['sha256']))))
    A = np.stack([np.frombuffer(b, np.uint8) for b in dfA['_dhash']]).astype(np.int32)
    B = np.stack([np.frombuffer(b, np.uint8) for b in dfB['_dhash']]).astype(np.int32)
    ham = A @ (1 - B).T + (1 - A) @ B.T
    out['neardup_pairs'] = int((ham <= dup_hamming).sum())
    out['ok'] = out['sha256_shared'] == 0 and out['neardup_pairs'] == 0
    return out


def cross_dataset_duplicate_rows(dfA, dfB, dup_hamming=8):
    # mặt nạ boolean trên các dòng của dfB là bản sao/gần trùng của ảnh nào đó trong dfA (dùng để loại khỏi tập đánh giá ngoài)
    if len(dfA) == 0 or len(dfB) == 0:
        raise ValueError(f'So trùng giữa hai dataset cần cả hai bảng khác rỗng (A={len(dfA)}, B={len(dfB)}). Bảng rỗng nghĩa là dataset đó bị loại hết ở bước audit.')
    A = np.stack([np.frombuffer(b, np.uint8) for b in dfA['_dhash']]).astype(np.int32)
    B = np.stack([np.frombuffer(b, np.uint8) for b in dfB['_dhash']]).astype(np.int32)
    ham = A @ (1 - B).T + (1 - A) @ B.T
    near = (ham <= dup_hamming).any(0)
    exact = dfB['sha256'].isin(set(dfA['sha256'])).values
    return near | exact


def write_manifest(df, path):
    cols = [c for c in df.columns if not c.startswith('_')]
    d = df[cols].copy()
    for c in d.columns:
        if d[c].map(lambda v: isinstance(v, (list, tuple))).any():
            d[c] = d[c].map(lambda v: ';'.join(v) if isinstance(v, (list, tuple)) else v)
    d.to_csv(path, index=False)
    h = sha256_file(path)
    json.dump(dict(manifest=os.path.basename(path), sha256=h, n=len(d)), open(path + '.sha256.json', 'w'))
    return h


# ============================ Dữ liệu TỔNG HỢP (chỉ để kiểm thử code; KHÔNG phải dữ liệu y tế) ============================
def _synth_image(rng, malignant, size=(190, 250), seed_bg=None):
    h, w = size
    bg_rng = np.random.RandomState(seed_bg) if seed_bg is not None else rng
    img = cv2.GaussianBlur((bg_rng.rand(h, w) * 90).astype(np.float32), (0, 0), 2.0) + 50
    mask = np.zeros((h, w), np.uint8)
    cx, cy = rng.randint(60, w - 60), rng.randint(50, h - 50)
    ax, ay = rng.randint(14, 36), rng.randint(10, 26)
    cv2.ellipse(mask, (cx, cy), (ax, ay), int(rng.randint(0, 90)), 0, 360, 255, -1)
    img = img + (mask > 0) * (55 if malignant else -30) + rng.rand(h, w) * 8
    return cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR), mask


def make_synthetic_busbra(root, n_patients=30, seed=0):
    # CẤU TRÚC GIẢ ĐỊNH giống BUS-BRA (Images/, Masks/, bus_data.csv với ID, Case, Pathology). Dùng để kiểm thử mã, không đại diện dữ liệu thật.
    rng = np.random.RandomState(seed)
    os.makedirs(os.path.join(root, 'Images'), exist_ok=True); os.makedirs(os.path.join(root, 'Masks'), exist_ok=True)
    rows, k = [], 0
    for p in range(n_patients):
        mal = p % 3 == 0
        for v in range(rng.randint(1, 4)):
            k += 1
            img, m = _synth_image(rng, mal, seed_bg=p)
            name = f'bus_{k:04d}-l'
            cv2.imwrite(os.path.join(root, 'Images', name + '.png'), img); cv2.imwrite(os.path.join(root, 'Masks', name.replace('bus_', 'mask_') + '.png'), m)
            rows.append(dict(ID=name, Case=f'C{p:04d}', Pathology='malignant' if mal else 'benign', BIRADS=4 + int(mal), Device='synthetic'))
    pd.DataFrame(rows).to_csv(os.path.join(root, 'bus_data.csv'), index=False)
    open(os.path.join(root, 'SYNTHETIC_README.txt'), 'w').write('Dữ liệu tổng hợp chỉ để kiểm thử mã. KHÔNG phải ảnh y tế.')
    return root


def make_synthetic_busi(root, n=24, seed=1, with_normal=True):
    rng = np.random.RandomState(seed)
    for c, cnt in (('benign', n), ('malignant', n // 2), ('normal', 6 if with_normal else 0)):
        d = os.path.join(root, c); os.makedirs(d, exist_ok=True)
        for i in range(1, cnt + 1):
            img, m = _synth_image(rng, c == 'malignant')
            if c == 'normal':
                m[:] = 0
            cv2.imwrite(f'{d}/{c} ({i}).png', img); cv2.imwrite(f'{d}/{c} ({i})_mask.png', m)
    open(os.path.join(root, 'SYNTHETIC_README.txt'), 'w').write('Dữ liệu tổng hợp chỉ để kiểm thử mã.')
    return root

In [ ]:
%%writefile bsv4/src/v3/evaluation_xai.py
# evaluation_xai.py - đánh giá phân loại/phân vùng, XAI (định vị, trung thực, nhất quán), hiệu chuẩn, chọn lọc dự đoán, thống kê, khóa pipeline
import os, json, time, hashlib
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from scipy import ndimage

import bus_trust as T
import losses_xai as LX
import bus_core as core
from data_audit import sha256_file


# ============================ Phân loại (lớp 1 = malignant) ============================
def classification_metrics(y, p_mal, threshold=0.5):
    from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
    y, p_mal = np.asarray(y), np.asarray(p_mal, float)
    pred = (p_mal >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else float('nan')
    spec = tn / (tn + fp) if tn + fp else float('nan')
    two = len(np.unique(y)) == 2
    return dict(auroc=float(roc_auc_score(y, p_mal)) if two else float('nan'), auprc=float(average_precision_score(y, p_mal)) if two else float('nan'),
                sensitivity_malignant=float(sens), specificity=float(spec), balanced_accuracy=float(np.nanmean([sens, spec])),
                f1=float(f1_score(y, pred, zero_division=0)), accuracy=float((pred == y).mean()), threshold=float(threshold))


def select_threshold_youden(y, p_mal):
    # CHỈ gọi trên validation của BUS-BRA
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y, p_mal)
    return float(thr[int(np.argmax(tpr - fpr))])


# ============================ Phân vùng ============================
def hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() or not gt.any():
        return float('nan')
    ps = pred ^ ndimage.binary_erosion(pred); gs = gt ^ ndimage.binary_erosion(gt)
    d_to_gt, d_to_pred = ndimage.distance_transform_edt(~gt), ndimage.distance_transform_edt(~pred)
    return float(np.percentile(np.concatenate([d_to_gt[ps], d_to_pred[gs]]), 95))


def seg_metrics(seg_prob, gt, thr=0.5, with_hd95=True):
    pb, gb = seg_prob.astype(np.float32) >= thr, gt > 0
    inter = (pb & gb).sum((1, 2)).astype(float); ps, gs = pb.sum((1, 2)).astype(float), gb.sum((1, 2)).astype(float)
    has = gs > 0
    out = dict(dice=2 * inter / (ps + gs + 1e-7), iou=inter / (ps + gs - inter + 1e-7), has_lesion=has)
    out['hd95'] = np.array([hd95(p, g) for p, g in zip(pb, gb)]) if with_hd95 else None
    hd = out['hd95'][has] if with_hd95 else np.array([])
    ok_hd = np.isfinite(hd)
    out['summary'] = dict(dice=float(out['dice'][has].mean()), iou=float(out['iou'][has].mean()),
                          hd95=float(hd[ok_hd].mean()) if ok_hd.any() else float('nan'),
                          n_hd95_undefined=int((~ok_hd).sum()) if with_hd95 else 0)          # HD95 không xác định khi mask dự đoán rỗng
    return out


# ============================ XAI: định vị ============================
def upsample_evidence(E, size):
    E = torch.as_tensor(E, dtype=torch.float32)
    if E.ndim == 3:
        E = E[:, None]
    U = F.interpolate(E, size=size, mode='bilinear', align_corners=False).clamp(min=0)
    return (U / (U.sum((1, 2, 3), keepdim=True) + 1e-12))[:, 0].numpy()


def localisation_report(E, gt, ring_px=8, min_area=1e-3, max_area=0.9):
    # Relevance Mass (trong tổn thương và trong tổn thương+vòng), Pointing Game, area-adjusted attribution.
    # Đây là đo ĐỊNH VỊ so với mask; KHÔNG chứng minh tính trung thực (faithfulness).
    U = upsample_evidence(E, gt.shape[-2:])
    rm, rmc, pg, aaa, aaac = [], [], [], [], []
    k = np.ones((2 * ring_px + 1, 2 * ring_px + 1), np.uint8)
    for u, g in zip(U, gt):
        g = (g > 0).astype(np.uint8)
        a = g.mean()
        if not (min_area < a < max_area):
            continue
        roi = cv2.dilate(g, k)
        mass_l, mass_r, a_r = float((u * g).sum()), float((u * roi).sum()), float(roi.mean())
        rm.append(mass_l); rmc.append(mass_r)
        aaa.append((mass_l - a) / (1 - a)); aaac.append((mass_r - a_r) / (1 - a_r) if a_r < 1 else float('nan'))
        pg.append(float(roi[np.unravel_index(np.argmax(u), u.shape)] > 0))
    f = lambda v: float(np.nanmean(v)) if len(v) else float('nan')
    return dict(n=len(rm), relevance_mass_lesion=f(rm), relevance_mass_lesion_context=f(rmc), pointing_game=f(pg),
                area_adjusted_lesion=f(aaa), area_adjusted_lesion_context=f(aaac))


# ============================ XAI: trung thực (faithfulness) ============================
def _auc(q, v):
    q, v = np.asarray(q, float), np.asarray(v, float)
    return float(np.sum((q[1:] - q[:-1]) * (v[1:] + v[:-1]) / 2))


@torch.no_grad()
def _prob_of(model, x, cls):
    return float(F.softmax(model(x)['logits'].float(), 1)[0, cls])


@torch.no_grad()
def deletion_insertion_curves(model, x, rank_map, cls, steps=10):
    # rank_map (h,w): điểm quan trọng; xoá/chèn dần các ô quan trọng nhất; baseline = 0 (ảnh trung bình sau chuẩn hóa)
    h, w = rank_map.shape
    order = torch.as_tensor(rank_map).flatten().argsort(descending=True)
    base = torch.zeros_like(x)
    dele, ins = [], []
    for s in range(steps + 1):
        k = int(round(h * w * s / steps))
        cm = torch.zeros(h * w); cm[order[:k]] = 1
        pm = F.interpolate(cm.view(1, 1, h, w).to(x.device), size=x.shape[-2:], mode='nearest')
        dele.append(_prob_of(model, x * (1 - pm) + base * pm, cls))
        ins.append(_prob_of(model, base * (1 - pm) + x * pm, cls))
    q = np.linspace(0, 1, steps + 1)
    return q, np.array(dele), np.array(ins)


def faithfulness_report(model, X, indices, device, steps=8, max_images=40, seed=0, use_cam=True):
    # So sánh xếp hạng theo evidence / Grad-CAM với xếp hạng NGẪU NHIÊN. Deletion AUC thấp hơn và Insertion AUC cao hơn ngẫu nhiên
    # là bằng chứng (chưa phải chứng minh) rằng vùng được xếp hạng ảnh hưởng thật tới dự đoán.
    from bus_core_v2 import gradcam_dict
    rng = np.random.RandomState(seed)
    model.eval()
    rows = []
    for k in indices[:max_images]:
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(device)
        with torch.no_grad():
            o = model(x)
        cls = int(o['logits'].argmax(1))
        h, w = (o['evidence'].shape[-2:] if o.get('evidence') is not None else (16, 16))
        maps = {'random': rng.rand(h, w)}
        if o.get('evidence') is not None:
            maps['evidence'] = o['evidence'][0, 0].float().cpu().numpy()
        if use_cam:
            cam, _ = gradcam_dict(model, x, class_idx=cls)
            maps['gradcam'] = cv2.resize(cam, (w, h), interpolation=cv2.INTER_AREA)
        row = {}
        for nm, mp in maps.items():
            q, d, i = deletion_insertion_curves(model, x, mp, cls, steps)
            row[f'deletion_auc_{nm}'], row[f'insertion_auc_{nm}'] = _auc(q, d), _auc(q, i)
        rows.append(row)
    keys = rows[0].keys() if rows else []
    return {k: float(np.mean([r[k] for r in rows])) for k in keys} | {'n': len(rows)}


@torch.no_grad()
def evidence_intervention(model, X, indices, device, q=0.2, max_images=40, seed=0):
    # Can thiệp trực tiếp vào evidence: loại bỏ q% ô có evidence lớn nhất (ev_mask) và đo giảm xác suất lớp dự đoán, so với loại ô ngẫu nhiên cùng số lượng.
    if getattr(getattr(model, 'cfg', None), 'evidence_mode', 'none') not in ('ewp', 'egr'):
        return dict(applicable=False)
    g = torch.Generator().manual_seed(seed)
    model.eval()
    d_top, d_rand = [], []
    for k in indices[:max_images]:
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(device)
        o = model(x)
        cls = int(o['logits'].argmax(1)); p0 = float(F.softmax(o['logits'].float(), 1)[0, cls])
        E = o['evidence']; n = E.numel(); kk = max(1, int(q * n))
        top = E.flatten().topk(kk).indices.cpu()
        rnd = torch.randperm(n, generator=g)[:kk]
        for idxs, store in ((top, d_top), (rnd, d_rand)):
            mk = torch.ones(n); mk[idxs] = 0
            p1 = float(F.softmax(model(x, ev_mask=mk.view_as(E).to(device))['logits'].float(), 1)[0, cls])
            store.append(p0 - p1)
    return dict(applicable=True, q=q, n=len(d_top), mean_drop_top_evidence=float(np.mean(d_top)), mean_drop_random=float(np.mean(d_rand)))


# ============================ Phát hiện evidence collapse và kiểm tra nhiễu nền ============================
def evidence_collapse_report(E, uniform_entropy=0.98, ess_uniform=0.9, ess_peaked_cells=2.0):
    E = torch.as_tensor(E, dtype=torch.float32)
    if E.ndim == 4:
        E = E[:, 0]
    N, h, w = E.shape
    P = E.flatten(1).clamp(min=0)
    P = P / (P.sum(1, keepdim=True) + 1e-12)
    H = -(P * torch.log(P + 1e-12)).sum(1) / np.log(h * w)           # entropy chuẩn hóa: 1 = đồng đều
    ess = 1.0 / ((P ** 2).sum(1) + 1e-12)                            # số ô hiệu dụng
    uniform = (H > uniform_entropy) & (ess / (h * w) > ess_uniform)
    peaked = ess < ess_peaked_cells
    return dict(n=N, mean_norm_entropy=float(H.mean()), mean_ess_fraction=float((ess / (h * w)).mean()), uniform_collapse_rate=float(uniform.float().mean()),
                peaked_collapse_rate=float(peaked.float().mean()), collapse_flag=bool(uniform.float().mean() > 0.5 or peaked.float().mean() > 0.5))


@torch.no_grad()
def background_perturbation(model, X, M, indices, device, ring_px=8, max_images=30, seed=0):
    # Xáo trộn pixel NGOÀI vùng (tổn thương + vòng) -> kỳ vọng dự đoán/evidence ít đổi nếu mô hình dựa vào tổn thương.
    # Xáo trộn pixel TRONG tổn thương (đối chứng) -> kỳ vọng thay đổi lớn hơn. Đây là kiểm tra mô tả, không phải chứng minh.
    rng = np.random.RandomState(seed)
    kern = np.ones((2 * ring_px + 1, 2 * ring_px + 1), np.uint8)
    model.eval()
    d_bg, d_les, mass_b, mass_a = [], [], [], []
    for k in indices[:max_images]:
        g = (M[k] > 0).astype(np.uint8)
        if not (0.001 < g.mean() < 0.9):
            continue
        roi = cv2.dilate(g, kern).astype(bool)
        x = core.normalize_to_tensor(X[k]).unsqueeze(0).to(device)
        o = model(x)
        cls = int(o['logits'].argmax(1)); p0 = float(F.softmax(o['logits'].float(), 1)[0, cls])

        def shuffled(region):
            xs = x.clone()
            ii = np.flatnonzero(region.ravel())
            perm = rng.permutation(ii)
            flat = x.view(1, 3, -1)
            xs.view(1, 3, -1)[:, :, torch.as_tensor(ii)] = flat[:, :, torch.as_tensor(perm)]
            return xs
        ob = model(shuffled(~roi)); ol = model(shuffled(g.astype(bool)))
        d_bg.append(abs(p0 - float(F.softmax(ob['logits'].float(), 1)[0, cls]))); d_les.append(abs(p0 - float(F.softmax(ol['logits'].float(), 1)[0, cls])))
        if o.get('evidence') is not None:
            mk = F.interpolate(torch.as_tensor(roi[None, None].astype(np.float32)), size=o['evidence'].shape[-2:], mode='nearest').to(device)
            En = LX.normalize_evidence(o['evidence'].float()); Eb = LX.normalize_evidence(ob['evidence'].float())
            mass_b.append(float((En * mk).sum())); mass_a.append(float((Eb * mk).sum()))
    f = lambda v: float(np.mean(v)) if len(v) else float('nan')
    return dict(n=len(d_bg), mean_abs_dprob_background_shuffle=f(d_bg), mean_abs_dprob_lesion_shuffle=f(d_les),
                evidence_mass_in_roi_before=f(mass_b), evidence_mass_in_roi_after_background_shuffle=f(mass_a))


# ============================ XAI: nhất quán ============================
@torch.no_grad()
def explanation_consistency_eval(model, X, indices, device, batch_size=16, seed=0, hflip_only=False):
    # prediction consistency và evidence consistency được TÁCH RIÊNG; evidence được inverse-warp và chỉ tính trên pixel hợp lệ
    g = torch.Generator(device=device).manual_seed(seed)
    model.eval()
    agree, dprob, cos_all = [], [], []
    for i in range(0, len(indices), batch_size):
        idx = indices[i:i + batch_size]
        x = torch.stack([core.normalize_to_tensor(X[k]) for k in idx]).to(device)
        theta = LX.hflip_theta(len(idx), device) if hflip_only else LX.sample_affine(len(idx), device, generator=g)
        o1, o2 = model(x), model(LX.apply_label_preserving(x, theta))
        p1, p2 = F.softmax(o1['logits'].float(), 1), F.softmax(o2['logits'].float(), 1)
        agree += (p1.argmax(1) == p2.argmax(1)).float().cpu().tolist()
        dprob += (p1[:, 1] - p2[:, 1]).abs().cpu().tolist()
        if o1.get('evidence') is not None:
            cos, ok, _, _ = LX.evidence_alignment(o1['evidence'], o2['evidence'], theta)
            cos_all += cos[ok > 0].cpu().tolist()
    return dict(prediction_agreement=float(np.mean(agree)), mean_abs_dprob_malignant=float(np.mean(dprob)),
                evidence_cosine_after_inverse_warp=float(np.mean(cos_all)) if cos_all else float('nan'), n=len(agree))


# ============================ Hiệu chuẩn, chọn lọc dự đoán ============================
def calibration_report(prob_val, y_val, prob_test, y_test):
    # T chọn trên validation, áp dụng cho test (hoặc cho tập ngoài)
    lg_v, lg_t = np.log(np.clip(prob_val, 1e-12, 1)), np.log(np.clip(prob_test, 1e-12, 1))
    Tm = T.fit_temperature(lg_v, y_val)
    rows = []
    for tag, tt in (('trước', 1.0), ('sau', Tm)):
        p = T.softmax(lg_t, tt)
        rows.append(dict(hiệu_chuẩn=tag, T=tt, NLL=T.nll(p, y_test), Brier=T.brier(p, y_test), ECE=T.ece(p, y_test)))
    return Tm, rows


def selective_report(prob_val, prob_test, y_test, coverages=(0.9, 0.8, 0.7), malignant=1):
    # độ tin cậy = max xác suất; ngưỡng chọn trên validation theo coverage mục tiêu
    cv, ct = prob_val.max(1), prob_test.max(1)
    pred = prob_test.argmax(1)
    wrong = pred != y_test
    cov, risk, aurc = T.risk_coverage(-ct, wrong)
    out = dict(aurc=float(aurc), rows=[])
    for c in coverages:
        thr = float(np.quantile(cv, 1 - c))
        acc = ct >= thr
        mal = y_test == malignant
        out['rows'].append(dict(coverage_target=c, coverage_actual=float(acc.mean()), selective_risk=float(wrong[acc].mean()) if acc.any() else float('nan'),
                                abstain_threshold=thr, sensitivity_malignant_accepted=float((pred[acc & mal] == malignant).mean()) if (acc & mal).any() else float('nan'),
                                malignant_abstained=float((~acc[mal]).mean()) if mal.any() else float('nan')))
    return out


# ============================ Thống kê ============================
def bootstrap_ci(fn, n, B=1000, seed=0):
    rng = np.random.RandomState(seed); v = []
    for _ in range(B):
        x = fn(rng.randint(0, n, n))
        if np.isfinite(x):
            v.append(x)
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) > 20 else (float('nan'), float('nan'))


def paired_bootstrap_diff(fa, fb, n, B=1000, seed=0):
    rng = np.random.RandomState(seed); d = []
    for _ in range(B):
        i = rng.randint(0, n, n)
        x = fa(i) - fb(i)
        if np.isfinite(x):
            d.append(x)
    return (float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))) if len(d) > 20 else (float('nan'),) * 3


def summarize_seeds(values):
    v = np.asarray([x for x in values if np.isfinite(x)], float)
    return dict(n_seeds=int(len(v)), mean=float(v.mean()) if len(v) else float('nan'), sd=float(v.std(ddof=1)) if len(v) > 1 else float('nan'), values=v.tolist())


# ============================ Khóa pipeline và đánh giá ngoài ============================
class LockError(RuntimeError):
    pass


class ExternalEvalLock:
    # Trình tự: (1) chọn mô hình/ngưỡng/temperature CHỈ trên BUS-BRA -> lock(); (2) evaluate_external() kiểm tra khóa rồi mới đọc BUSI;
    # (3) sau khi đánh giá, khóa bị đánh dấu 'đã dùng' và KHÔNG cho khóa lại hoặc đánh giá lại (một lần).
    def __init__(self, path):
        self.path = path

    def read(self):
        if not os.path.exists(self.path):
            raise LockError('Chưa có pipeline lock: không được truy cập dữ liệu ngoài (BUSI).')
        return json.load(open(self.path, encoding='utf-8'))

    def lock(self, ckpt_path, config, temperature, threshold, abstain_threshold, notes=''):
        if os.path.exists(self.path) and self.read().get('external_evaluated'):
            raise LockError('Pipeline đã được dùng để đánh giá ngoài; không được khóa lại.')
        cfg_sha = hashlib.sha256(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()
        json.dump(dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), ckpt=os.path.basename(ckpt_path), ckpt_sha256=sha256_file(ckpt_path), config=config,
                       config_sha256=cfg_sha, temperature=float(temperature), threshold=float(threshold), abstain_threshold=float(abstain_threshold),
                       notes=notes, external_evaluated=False), open(self.path, 'w', encoding='utf-8'), indent=1, default=str)

    def verify(self, ckpt_path):
        L = self.read()
        if sha256_file(ckpt_path) != L['ckpt_sha256']:
            raise LockError('Checkpoint đã thay đổi sau khi khóa (sha256 không khớp).')
        return L

    def guard_access(self, ckpt_path):
        L = self.verify(ckpt_path)
        if L.get('external_evaluated'):
            raise LockError('Đánh giá ngoài chỉ được thực hiện MỘT lần với pipeline đã khóa.')
        return L

    def mark_evaluated(self, summary):
        L = self.read()
        L['external_evaluated'] = True; L['external_summary'] = summary; L['evaluated_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        json.dump(L, open(self.path, 'w', encoding='utf-8'), indent=1, default=str)


def evaluate_external(lock, ckpt_path, load_external_fn, predict_fn):
    # load_external_fn(): đọc BUSI (chỉ được gọi SAU khi qua guard). predict_fn(model_data) -> dict(prob, ...)
    L = lock.guard_access(ckpt_path)
    data = load_external_fn()
    out = predict_fn(data, L)
    lock.mark_evaluated({k: v for k, v in out.items() if isinstance(v, (int, float, str, dict, list))})
    return out

In [ ]:
%%writefile bsv4/src/v3/losses_xai.py
# losses_xai.py - loss phân vùng/phân loại, boundary loss, ràng buộc giải thích không gian, counterfactual faithfulness, nhất quán và hình học
import math
import torch
import torch.nn.functional as F

EPS = 1e-8


# ============================ Phân vùng / phân loại ============================
def dice_loss_from_logits(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float()).flatten(1)
    t = target.float().flatten(1)
    inter = (p * t).sum(1)
    return 1 - ((2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


def seg_loss(seg_logits, masks):
    assert seg_logits.shape == masks.shape, f'seg {tuple(seg_logits.shape)} != mask {tuple(masks.shape)}'
    return F.binary_cross_entropy_with_logits(seg_logits.float(), masks.float()) + dice_loss_from_logits(seg_logits, masks)


def boundary_mask(masks):
    # Đường biên hình thái học (dilate - erode) trên mask GT, dùng để tăng trọng số loss tại ranh giới tổn thương
    m = masks.float()
    eroded = 1.0 - dilate(1.0 - m, 1)
    return (dilate(m, 1) - eroded).clamp(0, 1)


def boundary_loss(seg_logits, masks, edge_weight=3.0):
    assert seg_logits.shape == masks.shape
    m = masks.float()
    edge = boundary_mask(m)
    w = 1.0 + edge_weight * edge
    bce = F.binary_cross_entropy_with_logits(seg_logits.float(), m, weight=w)
    p = torch.sigmoid(seg_logits.float())
    inter = (p * m * edge).sum((1, 2, 3))
    den = (p * edge).sum((1, 2, 3)) + (m * edge).sum((1, 2, 3))
    edge_dice = 1.0 - ((2.0 * inter + 1.0) / (den + 1.0)).mean()
    return 0.5 * bce + 0.5 * edge_dice


def cls_loss(logits, labels, class_weights=None, label_smoothing=0.0):
    return F.cross_entropy(logits.float(), labels, weight=class_weights, label_smoothing=float(label_smoothing))


# ============================ ROI: tổn thương, vòng quanh và vùng hậu âm BI-RADS ============================
def downsample_mask(mask, size):
    # mask (B,1,H,W) -> (B,1,h,w) trung bình diện tích, giá trị trong [0,1]
    return F.adaptive_avg_pool2d(mask.float(), size)


def dilate(m, k):
    # giãn nở bằng max-pool (k = số ô đặc trưng); khả vi theo đoạn, dùng cho mask mềm
    return F.max_pool2d(m, kernel_size=2 * k + 1, stride=1, padding=k)


def lesion_ring(m, k):
    # vòng quanh tổn thương: dilate_k(m) - m, nằm trong [0,1]
    return (dilate(m, k) - m).clamp(0, 1)


def lesion_plus_context(m, k):
    return dilate(m, k).clamp(0, 1)


def posterior_zone(m, depth_cells=6):
    # Vùng hậu âm phía dưới tổn thương theo trục dọc chùm tia siêu âm (BI-RADS posterior acoustic shadowing/enhancement).
    # Chỉ phụ thuộc mask dự đoán m (B,1,h,w), hoàn toàn không dùng GT mask lúc suy luận; tách rời với m.
    if depth_cells <= 0:
        return torch.zeros_like(m)
    k_h = 2 * depth_cells + 1
    ker = m.new_zeros((1, 1, k_h, 3))
    ker[0, 0, :depth_cells, :] = 1.0
    raw = F.conv2d(m.float(), ker, padding=(depth_cells, 1))
    return (raw.clamp(0.0, 1.0) * (1.0 - m.float())).clamp(0.0, 1.0)


def normalize_evidence(E):
    # chuẩn hóa tổng = 1 trên không gian: loại bỏ phụ thuộc vào thang đo tuyệt đối
    return E / (E.sum((1, 2, 3), keepdim=True) + EPS)


# ============================ Spatial explanation constraint ============================
def explanation_loss(E, gt_mask, mode='lesion_context', ring_cells=2, delta=0.5, density_margin=1.0, min_roi=1e-3, max_roi=0.9):
    En = normalize_evidence(E.float())
    m = downsample_mask(gt_mask, E.shape[-2:])
    roi = lesion_plus_context(m, ring_cells)
    a_l, a_r = m.mean((1, 2, 3)), roi.mean((1, 2, 3))
    M_l, M_r = (En * m).sum((1, 2, 3)), (En * roi).sum((1, 2, 3))
    if mode == 'none':
        per = torch.zeros_like(M_l)
    elif mode == 'mask_only':
        per = 1 - M_l
    elif mode == 'lesion_context':
        per = F.relu((a_r + delta * (1 - a_r)) - M_r)
    elif mode == 'density':
        rho_in, rho_out = M_r / (a_r + EPS), (1 - M_r).clamp(min=0) / (1 - a_r + EPS)
        per = F.relu(torch.log(rho_out + EPS) - torch.log(rho_in + EPS) + math.log(1 + density_margin))
    else:
        raise ValueError('expl mode không hợp lệ: ' + str(mode))
    valid = ((a_r > min_roi) & (a_r < max_roi)).float()
    loss = (per * valid).sum() / valid.sum().clamp(min=1.0)
    return loss, dict(M_l=M_l.detach(), M_r=M_r.detach(), a_l=a_l.detach(), a_r=a_r.detach(), valid=valid.detach())


# ============================ Counterfactual faithfulness loss (V3) ============================
def counterfactual_faithfulness_loss(logits_full, logits_drop_hi, logits_drop_lo, margin=0.15, low_tol=0.05, conf_thr=0.6):
    # Tối ưu trực tiếp tính trung thực nhân quả: che vùng evidence cao nhất phải làm tụt xác suất lớp dự đoán >= margin,
    # trong khi che vùng evidence thấp nhất chỉ gây thay đổi nhỏ <= low_tol (trên các mẫu có độ tự tin >= conf_thr).
    p_full = F.softmax(logits_full.float(), 1)
    p_hi = F.softmax(logits_drop_hi.float(), 1)
    p_lo = F.softmax(logits_drop_lo.float(), 1)
    cls = p_full.argmax(1)
    idx = torch.arange(p_full.size(0), device=p_full.device)
    pc = p_full[idx, cls]
    d_hi = pc - p_hi[idx, cls]
    d_lo = pc - p_lo[idx, cls]
    per = F.relu(margin - d_hi) + F.relu(d_lo.abs() - low_tol)
    valid = (pc >= conf_thr).float()
    return (per * valid).sum() / valid.sum().clamp(min=1.0)


# ============================ Hình học: affine, inverse warp, mặt nạ pixel hợp lệ ============================
def sample_affine(B, device, max_rot_deg=10.0, scale=(0.9, 1.1), max_trans=0.05, flip_prob=0.5, generator=None):
    def u(lo, hi):
        return lo + (hi - lo) * torch.rand(B, device=device, generator=generator)
    ang = u(-max_rot_deg, max_rot_deg) * math.pi / 180
    s = u(scale[0], scale[1])
    fx = torch.where(torch.rand(B, device=device, generator=generator) < flip_prob, -torch.ones(B, device=device), torch.ones(B, device=device))
    tx, ty = u(-max_trans, max_trans) * 2, u(-max_trans, max_trans) * 2
    c, sn = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=device)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = c / s * fx, -sn / s, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sn / s * fx, c / s, ty
    return theta


def hflip_theta(B, device):
    theta = torch.zeros(B, 2, 3, device=device)
    theta[:, 0, 0], theta[:, 1, 1] = -1.0, 1.0
    return theta


def invert_theta(theta):
    B = theta.shape[0]
    A = torch.cat([theta, torch.tensor([0., 0., 1.], device=theta.device).view(1, 1, 3).expand(B, 1, 3)], 1)
    return torch.linalg.inv(A)[:, :2, :]


def warp(t, theta, mode='bilinear', padding_mode='zeros'):
    grid = F.affine_grid(theta, list(t.shape), align_corners=False)
    return F.grid_sample(t, grid, mode=mode, padding_mode=padding_mode, align_corners=False)


def inverse_warp(t_aug, theta, mode='bilinear'):
    return warp(t_aug, invert_theta(theta), mode=mode)


def valid_pixel_mask(theta, size_hw, device=None, thr=0.999):
    ones = torch.ones(theta.shape[0], 1, size_hw[0], size_hw[1], device=theta.device)
    real_content = (warp(ones, theta) > thr).float()
    return (inverse_warp(real_content, theta) > thr).float()


def intensity_jitter(x, contrast=(0.9, 1.1), brightness=0.1, generator=None):
    B = x.shape[0]
    c = contrast[0] + (contrast[1] - contrast[0]) * torch.rand(B, 1, 1, 1, device=x.device, generator=generator)
    b = (torch.rand(B, 1, 1, 1, device=x.device, generator=generator) * 2 - 1) * brightness
    return x * c + b


def apply_label_preserving(x, theta, jitter=True):
    xa = warp(x, theta)
    return intensity_jitter(xa) if jitter else xa


# ============================ Explanation consistency ============================
def prediction_consistency(logits1, logits2):
    p1, p2 = F.softmax(logits1.float(), 1), F.softmax(logits2.float(), 1)
    l1, l2 = F.log_softmax(logits1.float(), 1), F.log_softmax(logits2.float(), 1)
    kl12 = (p1 * (l1 - l2)).sum(1)
    kl21 = (p2 * (l2 - l1)).sum(1)
    return (0.5 * (kl12 + kl21)).mean()


def evidence_alignment(E1, E2, theta, min_valid_frac=0.25):
    E1n, E2n = normalize_evidence(E1.float()), normalize_evidence(E2.float())
    back = inverse_warp(E2n, theta)
    V = valid_pixel_mask(theta, E1.shape[-2:])
    a, b = (E1n * V).flatten(1), (back * V).flatten(1)
    cos = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + EPS)
    ok = (V.mean((1, 2, 3)) >= min_valid_frac).float()
    return cos, ok, back, V


def evidence_consistency(E1, E2, theta):
    cos, ok, _, _ = evidence_alignment(E1, E2, theta)
    return ((1 - cos) * ok).sum() / ok.sum().clamp(min=1.0)


# ============================ Tổng loss ============================
class LossCfg:
    def __init__(self, lam_seg=1.0, lam_cls=1.0, lam_exp=0.5, expl_mode='none', ring_cells=2, delta=0.5, density_margin=1.0,
                 lam_cons_pred=0.0, lam_cons_ev=0.0, label_smoothing=0.0, lam_edge=0.0, lam_faith=0.0, faith_q=0.2, faith_margin=0.15):
        self.__dict__.update(locals()); self.__dict__.pop('self')


def total_loss(out, masks, labels, cfg, class_weights=None, out_aug=None, theta=None, logits_drop_hi=None, logits_drop_lo=None):
    parts = {}
    if out.get('seg') is not None and cfg.lam_seg > 0:
        parts['seg'] = cfg.lam_seg * seg_loss(out['seg'], masks)
    if out.get('seg') is not None and getattr(cfg, 'lam_edge', 0.0) > 0:
        parts['edge'] = cfg.lam_edge * boundary_loss(out['seg'], masks)
    if out.get('logits') is not None and cfg.lam_cls > 0:
        parts['cls'] = cfg.lam_cls * cls_loss(out['logits'], labels, class_weights, getattr(cfg, 'label_smoothing', 0.0))
    if cfg.expl_mode != 'none' and out.get('evidence') is not None and cfg.lam_exp > 0:
        parts['exp'] = cfg.lam_exp * explanation_loss(out['evidence'], masks, cfg.expl_mode, cfg.ring_cells, cfg.delta, cfg.density_margin)[0]
    if out_aug is not None and theta is not None:
        if cfg.lam_cons_pred > 0 and out.get('logits') is not None:
            parts['cons_pred'] = cfg.lam_cons_pred * prediction_consistency(out['logits'], out_aug['logits'])
        if cfg.lam_cons_ev > 0 and out.get('evidence') is not None and out_aug.get('evidence') is not None:
            parts['cons_ev'] = cfg.lam_cons_ev * evidence_consistency(out['evidence'], out_aug['evidence'], theta)
    if getattr(cfg, 'lam_faith', 0.0) > 0 and logits_drop_hi is not None and logits_drop_lo is not None and out.get('logits') is not None:
        parts['faith'] = cfg.lam_faith * counterfactual_faithfulness_loss(
            out['logits'], logits_drop_hi, logits_drop_lo, margin=getattr(cfg, 'faith_margin', 0.15)
        )
    total = sum(parts.values()) if parts else torch.zeros((), requires_grad=True)
    return total, {k: float(v.detach()) for k, v in parts.items()}

In [ ]:
%%writefile bsv4/src/v3/models_xai.py
# models_xai.py - BreastVision-XAI V3: SCSE encoder/decoder, BI-RADS posterior zone, evidence coupling toàn diện & phân rã logit
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from bus_core import build_effnet_b3, UNetFamily
from losses_xai import lesion_ring, posterior_zone, EPS


# ============================ Khối chú ý nhẹ & Encoder/Decoder ============================
class SCSEBlock(nn.Module):
    # Concurrent Spatial and Channel Squeeze & Excitation (nhẹ, tăng chất lượng biên tổn thương nhỏ)
    def __init__(self, ch, r=8):
        super().__init__()
        mid = max(4, ch // r)
        self.cse = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True),
                                 nn.Conv2d(mid, ch, 1), nn.Sigmoid())
        self.sse = nn.Sequential(nn.Conv2d(ch, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.cse(x) + x * self.sse(x)


class EffNetB3Encoder(nn.Module):
    TAPS = (1, 2, 3, 5, 8)
    CH = {1: 24, 2: 32, 3: 48, 5: 136, 8: 1536}

    def __init__(self, pretrained=True):
        super().__init__()
        net, self.pretrained_loaded = build_effnet_b3(2, pretrained)
        self.features = net.features

    def forward(self, x):
        out = {}
        for i, blk in enumerate(self.features):
            x = blk(x)
            if i in self.TAPS:
                out[i] = x
        return out


def conv_block(cin, cout, use_scse=False):
    layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
              nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
    if use_scse:
        layers.append(SCSEBlock(cout))
    return nn.Sequential(*layers)


def _up(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)


class SegDecoder(nn.Module):
    def __init__(self, ch=(256, 128, 64, 32), use_scse=False):
        super().__init__()
        C = EffNetB3Encoder.CH
        self.b4 = conv_block(C[8] + C[5], ch[0], use_scse=use_scse)
        self.b3 = conv_block(ch[0] + C[3], ch[1], use_scse=use_scse)
        self.b2 = conv_block(ch[1] + C[2], ch[2], use_scse=use_scse)
        self.b1 = conv_block(ch[2] + C[1], ch[3], use_scse=use_scse)
        self.out = nn.Conv2d(ch[3], 1, 1)

    def forward(self, f, out_size):
        d = self.b4(torch.cat([_up(f[8], f[5]), f[5]], 1))
        d = self.b3(torch.cat([_up(d, f[3]), f[3]], 1))
        d = self.b2(torch.cat([_up(d, f[2]), f[2]], 1))
        d = self.b1(torch.cat([_up(d, f[1]), f[1]], 1))
        return F.interpolate(self.out(d), size=out_size, mode='bilinear', align_corners=False)


class Neck(nn.Module):
    def __init__(self, dim, use_scse=False):
        super().__init__()
        C = EffNetB3Encoder.CH
        self.l3, self.l5, self.l8 = nn.Conv2d(C[3], dim, 1), nn.Conv2d(C[5], dim, 1), nn.Conv2d(C[8], dim, 1)
        fuse_layers = [nn.Conv2d(dim, dim, 3, padding=1, bias=False), nn.BatchNorm2d(dim), nn.ReLU(inplace=True)]
        if use_scse:
            fuse_layers.append(SCSEBlock(dim))
        self.fuse = nn.Sequential(*fuse_layers)

    def forward(self, f):
        z = self.l3(f[3]) + _up(self.l5(f[5]), f[3]) + _up(self.l8(f[8]), f[3])
        return self.fuse(z)


# ============================ Gộp đa vùng ============================
def masked_avg_pool(Z, m, eps=1e-3):
    return (Z * m).sum((2, 3)) / (m.sum((2, 3)) + eps)


def evidence_zone_weights(Eu, zone_mask, eps=1e-3):
    # V3: trọng số không gian trong vùng giải phẫu dựa trên evidence Eu (B,1,h,w).
    # Nếu zone_mask rỗng -> w = 0 (hữu hạn); nếu khác rỗng -> chuẩn hóa tổng theo không gian (có +eps ổn định).
    raw = Eu * zone_mask
    return raw / (raw.sum((2, 3), keepdim=True) + eps)


class RegionFusion(nn.Module):
    def __init__(self, mode, n_regions, dim):
        super().__init__()
        self.mode, self.n, self.dim = (mode if n_regions > 1 else 'concat'), n_regions, dim
        self.proj = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_regions)])
        if self.mode == 'weighted':
            self.a = nn.Parameter(torch.zeros(n_regions))
        if self.mode == 'gated':
            self.gate = nn.Linear(n_regions * dim, n_regions * dim)
        self.out_dim = n_regions * dim if self.mode == 'concat' else dim

    def forward(self, xs):
        v = [p(x) for p, x in zip(self.proj, xs)]
        if self.mode == 'concat':
            return torch.cat(v, 1)
        if self.mode == 'weighted':
            w = F.softmax(self.a, 0)
            return sum(wi * vi for wi, vi in zip(w, v))
        g = F.softmax(self.gate(torch.cat(xs, 1)).view(-1, self.n, self.dim), 1)
        return (g * torch.stack(v, 1)).sum(1)


# ============================ Mô hình chính ============================
@dataclass
class XAIConfig:
    num_classes: int = 2
    dim: int = 128
    fusion: str = 'concat'          # concat | weighted | gated
    use_lesion: bool = True
    use_peri: bool = True
    evidence_mode: str = 'ewp'      # none | ewp | egr | decoupled
    ring_cells: int = 2
    detach_mask: bool = True
    dropout: float = 0.3
    pretrained: bool = True
    evidence_tau: float = 1.0
    # --- Nâng cấp V3 (mặc định False để X0-X6 giữ nguyên 100%) ---
    use_scse: bool = False
    use_posterior: bool = False
    post_depth: int = 6
    evidence_coupling_all: bool = False


class BreastVisionXAI(nn.Module):
    # Suy luận CHỈ dùng ảnh; mọi vùng giải phẫu (lesion, peri, posterior) được suy ra từ mask DỰ ĐOÁN.
    def __init__(self, cfg: XAIConfig):
        super().__init__()
        assert cfg.evidence_mode in ('none', 'ewp', 'egr', 'decoupled')
        self.cfg = cfg
        self.enc = EffNetB3Encoder(cfg.pretrained)
        self.pretrained_loaded = self.enc.pretrained_loaded
        self.dec = SegDecoder(use_scse=cfg.use_scse)
        self.neck = Neck(cfg.dim, use_scse=cfg.use_scse)
        n = 1 + int(cfg.use_lesion) + int(cfg.use_peri) + int(cfg.use_posterior)
        self.fusion = RegionFusion(cfg.fusion, n, cfg.dim)
        self.head = nn.Sequential(nn.Dropout(cfg.dropout), nn.Linear(self.fusion.out_dim, cfg.num_classes))
        if cfg.evidence_mode != 'none':
            self.ev_head = nn.Sequential(nn.Conv2d(cfg.dim, cfg.dim // 2, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(cfg.dim // 2, 1, 1))

    def _compute_regions(self, Z, m, ring, post, E, ev_mask=None):
        cfg = self.cfg
        if cfg.evidence_mode == 'none' or cfg.evidence_mode == 'decoupled':
            g = Z.mean((2, 3))
            Eu = None
            wg = torch.ones_like(m) / (m.shape[-2] * m.shape[-1])
        else:
            Eu = E if ev_mask is None else E * ev_mask
            wg = Eu if cfg.evidence_mode == 'ewp' else Eu / (Z.shape[-2] * Z.shape[-1])
            g = (Z * Eu).sum((2, 3)) if cfg.evidence_mode == 'ewp' else (Z * Eu).mean((2, 3))
        regions = [g]
        weights = [wg]
        zone_masks = []
        if cfg.use_lesion:
            zone_masks.append(m)
        if cfg.use_peri:
            zone_masks.append(ring)
        if cfg.use_posterior:
            zone_masks.append(post)
        for zm in zone_masks:
            if cfg.evidence_coupling_all and Eu is not None:
                wz = evidence_zone_weights(Eu, zm)
                regions.append((Z * wz).sum((2, 3)))
                weights.append(wz)
            else:
                regions.append(masked_avg_pool(Z, zm))
                weights.append(zm / (zm.sum((2, 3), keepdim=True) + 1e-3))
        return regions, weights

    def forward_heads_from_Z(self, Z, m, ring, post, E, ev_mask=None):
        # Dùng trong counterfactual faithfulness loss: chạy lại phần pooling + fusion + classifier mà KHÔNG chạy lại encoder/decoder
        regions, weights = self._compute_regions(Z, m, ring, post, E, ev_mask=ev_mask)
        h_vec = self.fusion(regions)
        return self.head(h_vec), regions, weights

    def forward(self, x, ev_mask=None):
        cfg = self.cfg
        f = self.enc(x)
        seg = self.dec(f, x.shape[-2:])
        Z = self.neck(f)
        m = F.adaptive_avg_pool2d(torch.sigmoid(seg.float()), Z.shape[-2:])
        if cfg.detach_mask:
            m = m.detach()
        E = None
        if cfg.evidence_mode != 'none':
            s = self.ev_head(Z).float() / cfg.evidence_tau
            B, _, h, w = s.shape
            if cfg.evidence_mode in ('ewp', 'decoupled'):
                E = F.softmax(s.flatten(2), -1).view(B, 1, h, w)
            else:
                E = torch.sigmoid(s)
        ring = lesion_ring(m, cfg.ring_cells) if cfg.use_peri else None
        post = posterior_zone(m, cfg.post_depth) if cfg.use_posterior else None
        regions, weights = self._compute_regions(Z, m, ring, post, E, ev_mask=ev_mask)
        logits = self.head(self.fusion(regions))
        return dict(seg=seg, logits=logits, evidence=E, mask_feat=m, ring=ring, post=post, Z=Z, regions=regions, region_weights=weights)

    @torch.no_grad()
    def infer(self, x):
        self.eval()
        o = self.forward(x)
        return dict(prob=F.softmax(o['logits'].float(), 1), seg_prob=torch.sigmoid(o['seg'].float()), evidence=o['evidence'])

    @torch.no_grad()
    def decompose_logits(self, x):
        # Phân rã chính xác logit:
        # - Khi evidence_coupling_all=False (V2): C_k(x) = E(x)*(W_g(A_g Z(x) + b_g))_k, rest_k chứa các vùng còn lại + bias.
        # - Khi evidence_coupling_all=True  (V3): TẤT CẢ các vùng đều đi qua weights(x) dựa trên E(x), nên
        #   C_k(x) = sum_i w_i(x) * [W_i A_i Z(x)]_k và rest_k = bias_k + sum_i [W_i b_i]_k (hằng số thuần túy!).
        assert self.cfg.evidence_mode == 'ewp' and self.fusion.mode == 'concat', 'phân rã chính xác chỉ có cho ewp + concat'
        self.eval()
        o = self.forward(x)
        D = self.cfg.dim
        W, bias = self.head[1].weight, self.head[1].bias
        if not self.cfg.evidence_coupling_all:
            A, b = self.fusion.proj[0].weight, self.fusion.proj[0].bias
            Wg = W[:, :D]
            C = o['evidence'] * (torch.einsum('kd,bdhw->bkhw', Wg @ A, o['Z']) + (Wg @ b).view(1, -1, 1, 1))
            rest = bias.expand(x.shape[0], -1).clone()
            for i, r in enumerate(o['regions'][1:], start=1):
                v = self.fusion.proj[i](r)
                rest = rest + v @ W[:, i * D:(i + 1) * D].T
            return C, rest, o['logits']

        C = torch.zeros(x.shape[0], self.cfg.num_classes, o['Z'].shape[2], o['Z'].shape[3], device=x.device, dtype=o['Z'].dtype)
        rest = bias.expand(x.shape[0], -1).clone()
        for i, w_map in enumerate(o['region_weights']):
            Ai, bi = self.fusion.proj[i].weight, self.fusion.proj[i].bias
            Wi = W[:, i * D:(i + 1) * D]
            C = C + w_map * torch.einsum('kd,bdhw->bkhw', Wi @ Ai, o['Z'])
            rest = rest + (Wi @ bi).view(1, -1)
        return C, rest, o['logits']


# ============================ Bọc mô hình cũ (E0–E4) để dùng chung engine ============================
class LegacyWrapper(nn.Module):
    def __init__(self, model, task, pretrained_loaded=None):
        super().__init__()
        self.model, self.task = model, task
        self.pretrained_loaded = getattr(model, 'pretrained_loaded', False) if pretrained_loaded is None else bool(pretrained_loaded)

    def forward(self, x, ev_mask=None):
        o = self.model(x)
        if self.task == 'cls':
            return dict(seg=None, logits=o, evidence=None)
        if self.task == 'seg':
            return dict(seg=o, logits=None, evidence=None)
        return dict(seg=o[0], logits=o[1], evidence=None)


# ============================ Danh mục thực nghiệm ============================
EXPERIMENTS = {
    # --- baseline gốc ---
    'E0': dict(kind='legacy', arch='effnet_b3', task='cls', desc='EfficientNet-B3 phân loại'),
    'E1': dict(kind='legacy', arch='unet', task='seg', desc='U-Net'),
    'E2': dict(kind='legacy', arch='attunet', task='seg', desc='Attention U-Net'),
    'E3': dict(kind='legacy', arch='attunet_mt', task='mt', desc='Attention U-Net đa nhiệm'),
    'E4': dict(kind='legacy', arch='attunet', task='seg', augment=False, desc='Attention U-Net không augmentation (ablation)'),
    # --- chuỗi X (V2) ---
    'X0': dict(kind='xai', model=dict(use_lesion=False, use_peri=False, evidence_mode='none'), loss={}, desc='Encoder chung đa nhiệm (baseline)'),
    'X1': dict(kind='xai', model=dict(use_lesion=True, use_peri=False, evidence_mode='none'), loss={}, desc='X0 + gộp global/lesion'),
    'X2': dict(kind='xai', model=dict(use_lesion=True, use_peri=True, evidence_mode='none'), loss={}, desc='X1 + vùng quanh tổn thương'),
    'X3': dict(kind='xai', model=dict(evidence_mode='ewp'), loss={}, desc='X2 + evidence-guided classification (EWP)'),
    'X4': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='lesion_context'), desc='X3 + explanation constraint'),
    'X5': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(lam_cons_pred=0.5, lam_cons_ev=0.5), desc='X3 + consistency'),
    'X6': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5), desc='X3 + constraint + consistency'),
    # --- chuỗi V3 (nâng cấp tính mới & hiệu năng) ---
    'V3_base': dict(kind='xai', model=dict(evidence_mode='ewp', use_scse=True),
                    loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5, label_smoothing=0.05, lam_edge=0.5),
                    ema=True, tta=True, desc='V3 Base: X6 + SCSE + boundary loss + label smoothing + EMA'),
    'V3_birads': dict(kind='xai', model=dict(evidence_mode='ewp', use_scse=True, use_posterior=True, evidence_coupling_all=True),
                      loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5, label_smoothing=0.05, lam_edge=0.5),
                      ema=True, tta=True, desc='V3 BI-RADS: + vùng hậu âm siêu âm + evidence coupling toàn vùng'),
    'V3_full': dict(kind='xai', model=dict(evidence_mode='ewp', use_scse=True, use_posterior=True, evidence_coupling_all=True),
                    loss=dict(expl_mode='lesion_context', lam_cons_pred=0.5, lam_cons_ev=0.5, label_smoothing=0.05, lam_edge=0.5, lam_faith=0.3),
                    ema=True, tta=True, desc='V3 Full: + Counterfactual Faithfulness Loss (đề xuất chính V3)'),
    # --- ablation bổ sung ---
    'A_fusion_weighted': dict(kind='xai', model=dict(evidence_mode='none', fusion='weighted'), loss={}, desc='X2 với weighted fusion'),
    'A_fusion_gated': dict(kind='xai', model=dict(evidence_mode='none', fusion='gated'), loss={}, desc='X2 với gated fusion'),
    'A_evidence_egr': dict(kind='xai', model=dict(evidence_mode='egr'), loss={}, desc='X3 formulation đối chứng: evidence-gated features (EGR)'),
    'A_evidence_decoupled': dict(kind='xai', model=dict(evidence_mode='decoupled'), loss=dict(expl_mode='lesion_context'), desc='Evidence KHÔNG nối vào logit (kiểm tra vai trò coupling)'),
    'A_exp_mask_only': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='mask_only'), desc='X3 + ràng buộc mask-only'),
    'A_exp_density': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(expl_mode='density'), desc='X3 + ràng buộc density-based'),
    'A_cons_pred_only': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(lam_cons_pred=0.5), desc='X3 + chỉ prediction consistency'),
    'A_cons_ev_only': dict(kind='xai', model=dict(evidence_mode='ewp'), loss=dict(lam_cons_ev=0.5), desc='X3 + chỉ evidence consistency'),
}
MAIN_CONFIGS = ['X0', 'X3', 'X4', 'X6', 'V3_full']      # cấu hình chính: chạy nhiều seed


def build_experiment(name, num_classes=2, pretrained=True, dim=128, base=32):
    spec = EXPERIMENTS[name]
    if spec['kind'] == 'xai':
        cfg = XAIConfig(num_classes=num_classes, pretrained=pretrained, dim=dim, **spec['model'])
        return BreastVisionXAI(cfg), asdict(cfg)
    from bus_core import build_model
    model, loaded = build_model(spec['arch'], num_classes, base=base, pretrained=pretrained)
    return LegacyWrapper(model, spec['task'], pretrained_loaded=loaded), dict(arch=spec['arch'], task=spec['task'], base=base)

In [ ]:
%%writefile bsv4/configs/bus_verify.json
{
  "_comment": "Cấu hình BUS-VERIFY V4. Các mục trong 'preregistered' được CỐ ĐỊNH TRƯỚC khi chạy thực nghiệm; đổi sau khi thấy kết quả là vi phạm giao thức và phải được ghi vào báo cáo.",
  "version": "V4-BUS-VERIFY-0.1",
  "preprocessing": {"img_size": 256, "seg_threshold": 0.5, "min_area_frac": 0.0015, "keep_components": "largest", "fill_holes": true},
  "regions": {"ring_frac_of_radius": 0.5, "ring_px_min": 6, "ring_px_max": 20, "bg_margin_px": 4},
  "counterfactual": {
    "kinds": ["blur", "replace", "permute"],
    "strengths": {"blur": [3.0, 7.0], "replace": [0.6, 1.0], "permute": [0.5, 1.0]},
    "strength_valid_range": {"blur": [0.5, 12.0], "replace": [0.1, 1.0], "permute": [0.1, 1.0]},
    "repeats_stochastic": 2,
    "n_matched_translations": 2,
    "evidence_top_q": 0.2,
    "seed": 20260101
  },
  "stability": {"tta": ["hflip", "affine_small", "brightness"], "affine_max_rot_deg": 6.0, "affine_scale": [0.95, 1.05], "affine_max_trans_frac": 0.03},
  "risk_model": {"type": "logistic_l2", "C_grid": [0.01, 0.03, 0.1, 0.3, 1.0], "oof_folds": 5, "compare_gradient_boosting": true, "calibrate": "platt"},
  "preregistered": {
    "base_model": "X6",
    "base_model_rationale": "Pipeline V3 cho thấy V3_full không vượt X6/X0 có ý nghĩa thống kê (artifacts/v3_verified/paired_comparisons.csv) và có AURC nội bộ kém hơn; X6 đơn giản hơn nhưng vẫn có đầu ra phân vùng + evidence. Lựa chọn này dựa một phần trên kết quả test V3 đã được khảo sát => phải khai báo, không coi là độc lập.",
    "seeds": [42, 43, 44],
    "oof_folds": 5,
    "primary_endpoints": ["error_auprc", "fn_alert_recall_at_alert_rate"],
    "alert_rates": [0.10, 0.20, 0.30],
    "primary_alert_rate": 0.20,
    "high_risk_alert_rate": 0.10,
    "n_bootstrap": 2000,
    "alpha": 0.05,
    "multiple_comparison": "holm",
    "primary_comparisons": [["P1", "B1"], ["P1", "B3"], ["P1", "B4"], ["P1", "B6"]],
    "test_labels_used_for_training_or_thresholds": false
  },
  "oos_gate": {"min_mask_area_quantile": 0.005, "max_mask_area_quantile": 0.995, "require_nonempty_mask": true},
  "ui": {"max_upload_mb": 15, "store_uploads": false, "send_images_to_external_services": false}
}

In [ ]:
%%writefile bsv4/artifacts/busbra_split_manifest.sha256.json
{"manifest": "busbra_split_manifest.csv", "sha256": "a5b0f502f3bf067b50264aa7bed4499d97e27b4f9d8f47620de91b9d8335037a", "n": 1875}

## 2. Cấu hình (cố định trước)

In [ ]:
import os, sys, json, time, copy, gc, hashlib, importlib
import numpy as np, pandas as pd
RUN_MODE = "full_experiment"          # "smoke_test" (dữ liệu tổng hợp, ~vài phút) | "full_experiment"
BUSBRA_ROOT, BUSI_ROOT = None, None
BUSBRA_COL_OVERRIDES = {}
OFFICIAL_TEST_FOLD = None
SPLIT_SEED = 42
CFG = json.load(open('bsv4/configs/bus_verify.json', encoding='utf-8'))
PRE = CFG['preregistered']
BASE = PRE['base_model']                          # 'X6' — khai báo trước
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./working')
SMOKE = RUN_MODE == 'smoke_test'
assert RUN_MODE in ('smoke_test', 'full_experiment')
WORK_DIR = os.path.join(WORK_DIR, RUN_MODE)
if SMOKE:
    IMG_SIZE, EPOCHS, PATIENCE, BATCH, SEEDS, K, B, ROB_N, NUM_WORKERS, PRETRAINED = 64, 2, 3, 4, [42, 43], 3, 50, 4, 0, False
else:
    IMG_SIZE, EPOCHS, PATIENCE, BATCH, SEEDS, K, B, ROB_N, NUM_WORKERS, PRETRAINED = CFG['preprocessing']['img_size'], 40, 10, 16, PRE['seeds'], PRE['oof_folds'], PRE['n_bootstrap'], 30, 2, True
SEG_THR, MIN_AREA_FRAC = CFG['preprocessing']['seg_threshold'], CFG['preprocessing']['min_area_frac']
RATES, PRIMARY_RATE, HIGH_RATE = PRE['alert_rates'], PRE['primary_alert_rate'], PRE['high_risk_alert_rate']
CKPT_DIR, OUT_DIR, FEAT_DIR, DEPLOY_DIR = (os.path.join(WORK_DIR, d) for d in ('checkpoints', 'outputs', 'features', 'deploy'))
for d in (WORK_DIR, CKPT_DIR, OUT_DIR, FEAT_DIR, DEPLOY_DIR):
    os.makedirs(d, exist_ok=True)
print(f'RUN_MODE={RUN_MODE} IMG={IMG_SIZE} EPOCHS={EPOCHS} SEEDS={SEEDS} K={K} B={B}')
if SMOKE:
    print('!' * 70 + '\n SMOKE TEST: dữ liệu TỔNG HỢP. MỌI SỐ LIỆU KHÔNG CÓ GIÁ TRỊ KHOA HỌC.\n' + '!' * 70)

In [ ]:
sys.path.insert(0, os.path.abspath('bsv4'))
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('src', 'bus_core', 'bus_trust', 'losses_xai', 'models_xai', 'data_audit', 'bus_core_v2', 'evaluation_xai')]:
    sys.modules.pop(m, None)
importlib.invalidate_caches()
import torch
from src import data_audit as DA, pipeline as PL, benchmark as BM, failure_risk as FR, inference as INF, preprocessing as PP, segmentation as SG
from src import selective_prediction as SP, evaluation as EV, robustness as RB, counterfactual_verifier as CF, uncertainty as UN
import bus_core as core, bus_core_v2 as C2, models_xai as MX, bus_trust as TRUST
DEVICE = C2.get_device()
TRAIN_CFG = C2.TrainCfg(epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH, amp=DEVICE.type == 'cuda', num_workers=NUM_WORKERS)
C2.seed_everything(SPLIT_SEED)
print('device', DEVICE, '| torch', torch.__version__)

## 3. Dữ liệu: tìm, kiểm toán, chia theo bệnh nhân, khớp manifest V3

In [ ]:
if SMOKE:
    BUSBRA_ROOT = DA.make_synthetic_busbra(os.path.join(WORK_DIR, 'synthetic_busbra'), n_patients=40, seed=0)
    BUSI_ROOT = DA.make_synthetic_busi(os.path.join(WORK_DIR, 'synthetic_busi'), n=24, seed=5)
else:
    def one(c, what):
        if len(c) != 1:
            raise FileNotFoundError(f'{what}: tìm thấy {len(c)} ứng viên {c}. Đặt đường dẫn thủ công.')
        return c[0]
    BUSBRA_ROOT = BUSBRA_ROOT or one(DA.find_busbra_candidates(INPUT_DIR), 'BUS-BRA')
    BUSI_ROOT = BUSI_ROOT or one(DA.find_busi_candidates(INPUT_DIR), 'BUSI')
df_raw, info = DA.index_busbra(BUSBRA_ROOT, col_overrides=BUSBRA_COL_OVERRIDES)
df_a, rep = DA.audit_dataframe(df_raw)
df_clean, drop = DA.drop_invalid(df_a)
busi_raw, busi_info = DA.index_busi_binary(BUSI_ROOT)
busi_a, busi_rep = DA.audit_dataframe(busi_raw, mask_col='mask_paths')
busi_clean, _ = DA.drop_invalid(busi_a)
dup = DA.cross_dataset_duplicate_rows(df_clean, busi_clean)
busi_clean = busi_clean[~dup].reset_index(drop=True)
assert DA.cross_dataset_duplicates(df_clean, busi_clean)['ok']
df = DA.grouped_split(df_clean, seed=SPLIT_SEED, official_test_fold=OFFICIAL_TEST_FOLD)
leak = DA.leakage_report(df); assert leak['ok'], leak
DA.check_label_mapping(df)
df = df.reset_index(drop=True)
print('pairing:', DA.check_pairing(df))
mp = DA.write_manifest(df, os.path.join(OUT_DIR, 'busbra_split_manifest.csv'))
rec = json.load(open('bsv4/artifacts/busbra_split_manifest.sha256.json'))
MANIFEST_MATCH_V3 = (mp == rec['sha256'])
print('manifest sha256 khớp V3:', MANIFEST_MATCH_V3, '(nếu False: chia khác V3 -> phải khai báo trong báo cáo; tập test cũ KHÔNG còn là test V3)')
X, M, y = C2.load_arrays(df, IMG_SIZE)
Xb, Mb, yb = C2.load_arrays(busi_clean, IMG_SIZE, mask_col='mask_paths')
split = df['split'].values
dev_idx = np.where(split != 'test')[0]; test_idx = np.where(split == 'test')[0]
fold = DA.dev_folds(df, k=K, seed=SPLIT_SEED)
orep = DA.oof_leakage_report(df, fold); assert orep['ok'], orep
groups_all = df['group'].values
groups_busi = np.arange(len(busi_clean))              # BUSI không có patient ID: mỗi ảnh một nhóm -> CI mức ảnh (lạc quan)
print('dev', len(dev_idx), 'test', len(test_idx), 'busi', len(busi_clean), '| fold malignant rate', DA.fold_malignant_rates(df, fold))
json.dump(dict(busbra=rep, busi=busi_rep, oof_leakage=orep, manifest_sha=mp, manifest_match_v3=MANIFEST_MATCH_V3), open(os.path.join(OUT_DIR, 'data_audit_v4.json'), 'w'), indent=1, default=str)

## 4. Huấn luyện X6: mô hình OOF (mỗi fold x seed) và mô hình cuối (resume được)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
SPEC = MX.EXPERIMENTS[BASE]
def inner_val(pool):
    sgk = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=SPLIT_SEED)
    tr, va = next(sgk.split(pool, y[pool], groups_all[pool]))
    return pool[tr], pool[va]
def train_one(tag, tr_idx, va_idx, seed):
    data = dict(X=X, M=M, y=y, train=tr_idx, val=va_idx)
    model, cfgd = MX.build_experiment(BASE, pretrained=PRETRAINED)
    meta = dict(experiment=BASE, num_classes=2, dim=128, base=32, img_size=IMG_SIZE, seed=seed, model_cfg=cfgd, run_mode=RUN_MODE, tag=tag,
                pretrained_loaded=bool(getattr(model, 'pretrained_loaded', False)))
    if PRETRAINED and not meta['pretrained_loaded']:
        print(f'CẢNH BÁO [{tag}]: không nạp được pretrained -> backbone ngẫu nhiên.')
    cfg_run = copy.copy(TRAIN_CFG); cfg_run.use_ema = bool(SPEC.get('ema', False))
    r = C2.train_with_oom_fallback(tag, model, C2.make_loss_cfg(SPEC), data, cfg=cfg_run, out_dir=CKPT_DIR, seed=seed, device=DEVICE,
                                   augment=SPEC.get('augment', True), meta=meta)
    del model; gc.collect()
    return r['best_path']
PATHS = {}
t0 = time.time()
for f in range(K):                                     # OOF: huấn luyện trên dev trừ fold f
    pool = np.where((fold >= 0) & (fold != f))[0]
    tr, va = inner_val(pool)
    assert not set(np.where(fold == f)[0]) & (set(tr) | set(va))
    for s in SEEDS:
        PATHS[('oof', f, s)] = train_one(f'{BASE}_oof{f}', tr, va, s)
for s in SEEDS:                                        # mô hình cuối: train/val theo manifest (giống V3)
    PATHS[('final', 0, s)] = train_one(f'{BASE}_final', np.where(split == 'train')[0], np.where(split == 'val')[0], s)
print('huấn luyện xong trong', round((time.time() - t0) / 60, 1), 'phút')

## 5. Dự đoán OOF -> T-scaling và ngưỡng Youden CHỈ từ dữ liệu phát triển

In [ ]:
from sklearn.metrics import roc_curve
MAIN = SEEDS[0]
def predictor(kind, f, s):
    return INF.load_predictor(PATHS[(kind, f, s)], DEVICE)
logp_dev = np.zeros((len(df), 2))
for f in range(K):
    idx = np.where(fold == f)[0]
    pr = predictor('oof', f, MAIN)
    logp_dev[idx] = pr(X[idx])['logits']
TEMP = TRUST.fit_temperature(logp_dev[dev_idx], y[dev_idx])
p_dev = UN.softmax(logp_dev[dev_idx], TEMP)[:, 1]
fpr, tpr, th = roc_curve(y[dev_idx], p_dev)
THR = float(th[int(np.argmax(tpr - fpr))])
THR = float(np.clip(THR, 1e-3, 1 - 1e-3))
print(f'T={TEMP:.3f}  ngưỡng Youden (dev OOF)={THR:.3f}  [không dùng nhãn test]')

## 6. Trích đặc trưng (có cache; ensemble = cùng fold khác seed, chỉ cho baseline B3)

In [ ]:
def extract_cached(name, imgs, ids, main, members):
    path = os.path.join(FEAT_DIR, f'{name}.csv')
    if os.path.exists(path):
        return pd.read_csv(path)
    d = PL.extract_dataframe(imgs, ids, main, base_seed=SPLIT_SEED, T=TEMP, thr=THR, seg_thr=SEG_THR, min_area_frac=MIN_AREA_FRAC, members=members, with_cf=True,
                             progress=lambda i: print(name, i, flush=True))
    d.to_csv(path, index=False); return d
parts = []
for f in range(K):
    idx = np.where(fold == f)[0]
    main = predictor('oof', f, MAIN); mem = [predictor('oof', f, s) for s in SEEDS[1:]]
    parts.append(extract_cached(f'dev_fold{f}', X[idx], idx, main, mem))
df_dev = pd.concat(parts).sort_values('id').reset_index(drop=True)
assert (df_dev['id'].values == dev_idx).all()
fin_main = predictor('final', 0, MAIN); fin_mem = [predictor('final', 0, s) for s in SEEDS[1:]]
df_test = extract_cached('busbra_test', X[test_idx], test_idx, fin_main, fin_mem)
df_busi = extract_cached('busi', Xb, np.arange(len(Xb)), fin_main, fin_mem)
y_dev, y_test, y_busi = y[dev_idx], y[test_idx], yb
tg = {n: PL.make_error_targets(yy, d['p_mal'].values, THR) for n, yy, d in (('dev', y_dev, df_dev), ('test', y_test, df_test), ('busi', y_busi, df_busi))}
for n, t in tg.items():
    print(n, 'n=', len(t['wrong']), 'sai=', int(t['wrong'].sum()), 'FN=', int(t['is_fn'].sum()), 'FP=', int(t['is_fp'].sum()))

## 7. Giao thức benchmark (B0–B6, P1, ablation; Holm; bootstrap theo bệnh nhân)

In [ ]:
n_err = int(tg['dev']['wrong'].sum())
print('số lỗi trên dev OOF:', n_err)
RISK_FITTED = n_err >= 10
if not RISK_FITTED:
    assert SMOKE, f'Quá ít ca lỗi ({n_err}) trên dev: không thể huấn luyện bộ phát hiện — dừng (không chỉnh giao thức).'
    print('SMOKE: quá ít lỗi trên dữ liệu tổng hợp, bỏ qua fit risk (chỉ ở smoke).')
if RISK_FITTED:
    split_dev = split[dev_idx]; FR.assert_no_test(split_dev)
    RES = BM.run_protocol(df_dev, tg['dev']['wrong'], tg['dev']['is_fn'], groups_all[dev_idx], split_dev,
                          {'test': df_test, 'busi': df_busi}, {'test': tg['test']['wrong'], 'busi': tg['busi']['wrong']},
                          {'test': tg['test']['is_fn'], 'busi': tg['busi']['is_fn']}, {'test': groups_all[test_idx], 'busi': groups_busi},
                          seed=SPLIT_SEED, k=K, B=B, rates=RATES)
    display(RES['dev_table'].round(3)); display(RES['dev_pairs'].round(4))

## 8. Khóa risk model, chính sách cảnh báo, cổng phạm vi, cấu hình triển khai

In [ ]:
def img_stats(imgs):
    return np.stack([FR.image_stats(a, PP.valid_content_mask(a)) for a in imgs])
if RISK_FITTED:
    model = RES['model']; policy = RES['policy']
    model.save(os.path.join(DEPLOY_DIR, 'risk'))
    gate = FR.ImageStatGate(q=0.995).fit(img_stats(X[dev_idx]))
    content = dict(t_review=policy['t_review'], t_high=policy['t_high'], gate=gate.to_dict(), source=policy['source'], n_dev=policy['n_dev'])
    pol_path = os.path.join(DEPLOY_DIR, 'alert_policy.locked.json')
    if os.path.exists(pol_path): os.remove(pol_path)    # chỉ cho phép khi chạy lại TOÀN BỘ từ đầu
    FR.lock_policy(pol_path, content)
    import shutil
    shutil.copy(PATHS[('final', 0, MAIN)], os.path.join(DEPLOY_DIR, 'model.pt'))
    dcfg = dict(img_size=IMG_SIZE, temperature=TEMP, threshold=THR, seg_threshold=SEG_THR, min_area_frac=MIN_AREA_FRAC, seed=CFG['counterfactual']['seed'],
                model_name=BASE, version=CFG['version'], checkpoint='model.pt', run_mode=RUN_MODE)
    json.dump(dcfg, open(os.path.join(DEPLOY_DIR, 'deploy_config.json'), 'w'), indent=1)
    print('đã khóa:', {k: v for k, v in content.items() if k != 'gate'})

## 9. Ghi artifact kết quả cho ứng dụng web

In [ ]:
sys.path.insert(0, os.path.abspath('bsv4'))
V4 = dict(dev_table='v4_dev_nested_cv_table.csv', dev_pairs='v4_dev_paired_comparisons.csv', test_table='v4_busbra_test_table.csv', test_pairs='v4_busbra_test_paired_comparisons.csv',
          busi_table='v4_busi_external_table.csv', busi_pairs='v4_busi_external_paired_comparisons.csv', ablation='v4_ablation_dev.csv', robustness='v4_robustness_summary.csv',
          oracle='v4_oracle_experiment.csv', pred_dev='v4_predictions_dev_oof.csv', pred_test='v4_predictions_busbra_test.csv', pred_busi='v4_predictions_busi.csv',
          risk_coverage='v4_risk_coverage_curves.csv', classification='v4_classification_metrics.csv', status_table='v4_alert_status_counts.csv')
def W(key, d):
    d.to_csv(os.path.join(OUT_DIR, V4[key]), index=False)
def seg_dice(pr, Xs, Ms):
    ds = []
    for i in range(0, len(Xs), 32):
        o = pr(Xs[i:i + 32])
        for sp, m in zip(o['seg_prob'], Ms[i:i + 32]):
            ds.append(SG.dice(SG.postprocess_mask(sp, SEG_THR, MIN_AREA_FRAC)[0], m > 0))
    return float(np.mean(ds))
if RISK_FITTED:
    W('dev_table', RES['dev_table']); W('dev_pairs', RES['dev_pairs'])
    W('ablation', RES['dev_table'][RES['dev_table']['method'].str.startswith(('P1', 'B4', 'B5', 'B6'))])
    for key, nm in (('test', 'test'), ('busi', 'busi')):
        W(f'{key}_table', RES['held_out'][nm]['table']); W(f'{key}_pairs', RES['held_out'][nm]['pairs'])
    bundle = PL.Bundle(DEPLOY_DIR); assert bundle.risk_enabled
    pol = bundle.policy
    sets = dict(dev=(df_dev, RES['dev_scores'], tg['dev'], y_dev, X[dev_idx]), test=(df_test, RES['held_out']['test']['scores'], tg['test'], y_test, X[test_idx]),
                busi=(df_busi, RES['held_out']['busi']['scores'], tg['busi'], y_busi, Xb))
    rc_rows, cls_rows, st_rows = [], [], []
    for nm, (d, sc, t, yy, imgs) in sets.items():
        risk = sc['P1']
        ins = (d['mask_empty'].values == 0) & np.array([bundle.gate.in_scope(s) for s in img_stats(imgs)])
        cnt, st = BM.status_counts(risk, pol, ins)
        for k_, v in cnt.items(): st_rows.append(dict(dataset=nm, status=k_, n=int(v)))
        W(f'pred_{nm}', pd.DataFrame(dict(id=d['id'], y=yy, p_mal=d['p_mal'], pred=t['pred'], wrong=t['wrong'], is_fn=t['is_fn'], risk=risk, status=st)))
        for meth, s in sc.items():
            cov, sr, _ = SP.risk_coverage_curve(s, t['wrong']); sel = np.unique(np.linspace(0, len(cov) - 1, min(200, len(cov))).astype(int))
            rc_rows += [dict(dataset=nm, method=meth, coverage=float(cov[i]), selective_risk=float(sr[i])) for i in sel]
        cm = EV.classification_metrics(yy, d['p_mal'].values, THR)
        cm.update(dataset=nm, ece=EV.ece_binary(d['p_mal'].values, yy), brier=EV.brier_binary(d['p_mal'].values, yy), nll=EV.nll_binary(d['p_mal'].values, yy))
        if nm == 'test': cm['dice_pred_mask'] = seg_dice(fin_main, X[test_idx], M[test_idx])
        if nm == 'busi': cm['dice_pred_mask'] = seg_dice(fin_main, Xb, Mb)
        cls_rows.append(cm)
    W('risk_coverage', pd.DataFrame(rc_rows)); W('classification', pd.DataFrame(cls_rows)); W('status_table', pd.DataFrame(st_rows))
    json.dump({k: RES['held_out'][k]['risk_calibration'] for k in RES['held_out']}, open(os.path.join(OUT_DIR, 'risk_calibration.json'), 'w'), indent=1)
    display(pd.DataFrame(cls_rows).round(3)); display(pd.DataFrame(st_rows))

## 10. Robustness Lab (label-preserving vs counterfactual tách riêng)

In [ ]:
if RISK_FITTED:
    rng = np.random.default_rng(SPLIT_SEED); sel = rng.choice(len(test_idx), size=min(ROB_N, len(test_idx)), replace=False)
    imgs = [X[test_idx][i] for i in sel]
    def risk_fn(img):
        row, _ = PL.extract_features(img, fin_main, T=TEMP, thr=THR, seg_thr=SEG_THR, min_area_frac=MIN_AREA_FRAC, seed=0, with_cf=True)
        d = PL.add_derived(pd.DataFrame([row]))
        for c in bundle.risk.feature_names:
            if c not in d: d[c] = np.nan
        return float(bundle.risk.predict_risk(d)[0])
    rob = RB.run_robustness_lab(imgs, fin_main, None, (1, 3, 5), seed=SPLIT_SEED, T=TEMP, thr=THR, seg_thr=SEG_THR, min_area_frac=MIN_AREA_FRAC, risk_fn=risk_fn)
    rob.to_csv(os.path.join(OUT_DIR, 'v4_robustness_raw.csv'), index=False)
    rs = RB.summarize_robustness(rob); W('robustness', rs); display(rs.round(3))

## 11. Thí nghiệm ORACLE (dùng GT mask — CHỈ phân tích, tách khỏi suy luận thật)

In [ ]:
if RISK_FITTED:
    rows = []
    for i in sel:
        gi = test_idx[i]; img = X[gi]; gt = (M[gi] > 0).astype(np.uint8)
        if gt.sum() == 0: continue
        o = fin_main(img[None]); valid = PP.valid_content_mask(img)
        pm = SG.postprocess_mask(o['seg_prob'][0], SEG_THR, MIN_AREA_FRAC)[0]
        for src, mk in (('predicted', pm), ('oracle_gt', gt)):
            if mk is None or mk.sum() == 0: continue
            r = CF.verify_image(img, mk, fin_main, T=TEMP, thr=THR, cfg=CF.CFConfig(seed=0), valid=valid, evidence=o['evidence'][0], rng=np.random.default_rng([0, 17]))
            rows.append(dict(id=int(gi), y=int(y[gi]), mask_source=src, dice_pred_gt=SG.dice(pm, gt), status=r['status'], **{k: v for k, v in r['features'].items() if isinstance(v, (int, float, np.floating))}))
    orc = pd.DataFrame(rows); W('oracle', orc)
    display(orc.groupby('mask_source')[[c for c in ('lesion_sens', 'bg_sens', 'cf_specificity', 'lesion_bg_contrast', 'ev_faithfulness') if c in orc]].mean().round(3))

## 12. RUN_MANIFEST và gói tải xuống cho web app

In [ ]:
def sha(p): return FR.sha256_file(p)
files = {f: sha(os.path.join(OUT_DIR, f)) for f in sorted(os.listdir(OUT_DIR)) if f.endswith(('.csv', '.json'))}
man = dict(version=CFG['version'], run_mode=RUN_MODE, scientific_validity='NONE (synthetic smoke test)' if SMOKE else 'real run — xem giới hạn trong báo cáo',
           created=time.strftime('%Y-%m-%d %H:%M:%S'), torch=torch.__version__, device=str(DEVICE), base_model=BASE, seeds=SEEDS, oof_folds=K, n_bootstrap=B,
           img_size=IMG_SIZE, temperature=TEMP, threshold=THR, alert_rates=RATES, primary_rate=PRIMARY_RATE, risk_fitted=bool(RISK_FITTED),
           manifest_match_v3=MANIFEST_MATCH_V3, config_sha256=hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest(),
           test_labels_used_for_training_or_thresholds=False,
           notes=['BUS-BRA test và BUSI đã được khảo sát ở V3: không phải tập chưa từng quan sát',
                  'risk học từ dự đoán OOF của mô hình fold, áp dụng cho mô hình cuối (cross-fit, khai báo)',
                  'ensemble chỉ dùng cho baseline B3; risk triển khai dùng TTA vì web chỉ có một checkpoint',
                  'BUSI không có patient ID: CI mức ảnh lạc quan'], artifacts=files)
if RISK_FITTED:
    man['deploy'] = {f: sha(os.path.join(DEPLOY_DIR, f)) for f in ('model.pt', 'deploy_config.json', 'alert_policy.locked.json')}
json.dump(man, open(os.path.join(OUT_DIR, 'RUN_MANIFEST.json'), 'w'), indent=1, ensure_ascii=False)
import shutil
zp = shutil.make_archive(os.path.join(WORK_DIR, 'BreastVision_V4_run_artifacts'), 'zip', WORK_DIR, 'outputs')
if RISK_FITTED: shutil.make_archive(os.path.join(WORK_DIR, 'BreastVision_V4_deploy'), 'zip', WORK_DIR, 'deploy')
print('Chép outputs/*.csv|json vào artifacts/v4/ và deploy/ vào thư mục deploy/ của app. Gói:', zp)